<!-- Reference: https://github.com/raphaelmansuy/edgequake -->


<!-- Reference: https://www.youtube.com/watch?v=Cyn_Dm05_eU -->


<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

# 📑 Table of Contents

---

### ⚙️ Phase 0: Infrastructure & Environment Setup
Establishing the project root, medallion directory structure, and audit trail.

### 📥 Phase 1: Ingestion, Validation & Silver Layer

* **1a** — Raw Ingestion & Structural Integrity Anchor
* **1b** — CDC ICD-10 Validation & Noisy 111 Audit (FY2026)
* **1c** — Token Volume & Truncation Audit
* **1d** — Raw Discovery Auditor *(human-in-the-loop)*
* **1e** — Pydantic Firewall & Silver Layer Registration
* **1f** — Surgical Signal Auditor *(human-in-the-loop)*
* **1g** — DuckDB Silver Vault Persistence

### 🔬 Phase 2: Gold Layer Construction

* **2a** — Biological Audit & Decimal Restoration
* **2b** — Gold Layer Discovery Auditor *(human-in-the-loop)*
* **2c** — ICD-10 Status Annotation

### 🔄 Phase 3: Signal Optimisation & Leakage Neutralisation

* **3a** — APSO-Flip (Diagnostic Signal Prioritisation)
* **3b** — ICD-10 Leakage Detection
* **3b.1** — Leakage Forensic Auditor *(human-in-the-loop)*
* **3b.2** — ICD-10 String Format Corpus Audit
* **3c** — ICD-10 Code Redaction
* **3c.1** — Post-Redaction Integrity Auditor *(human-in-the-loop)*

### 💾 Phase 4: Gold Layer Parquet Export

---

### 📚 Dataset Reference
MedSynth (Rezaie Mianroodi et al., 2025) — arXiv:2508.01401

---

### 🎯 Project Objective

This notebook performs a rigorous forensic ingestion and preparation pipeline
for the MedSynth clinical dataset — 10,240 synthetic doctor-patient dialogue
and SOAP note pairs covering 2,037 distinct ICD-10 codes.

The pipeline moves from raw synthetic data through structural validation,
CDC reference verification, SOAP extraction, canonical code formatting,
ICD-10 status annotation, APSO-recomposition, and leakage redaction — producing
a fully annotated Gold layer ready for downstream ICD-10 classification or
clinical NLP tasks.

**Key findings:**
- 94.3% of records carry billable CDC FY2026 leaf codes
- 0.6% carry valid parent-level codes (Noisy 111) reflecting real clinical billing practice
- 64.7% of notes exceed ClinicalBERT's 512-token limit — addressed by APSO-Flip
- 28.5% of records contained explicit ICD-10 code strings — fully redacted
- All 10,240 records are retained; no records were removed

**Output:** `medsynth_gold_apso_20260319_111429.parquet` — 58.9 MB,
10,240 records, 13 columns, 0 ICD-10 leaks remaining.

</div>

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🕵️ Project Narrative

This notebook transforms the raw MedSynth synthetic clinical dataset into a
fully validated, annotated, and redacted Gold layer suitable for downstream
ICD-10 classification and clinical NLP research.

The pipeline is structured as a progressive refinement across four stages:

**Ingestion & Validation (Phases 1a–1g)** — Raw data is loaded, structurally
validated, verified against the CDC FY2026 ICD-10 reference, and promoted
through Bronze → Silver layers with full audit traceability.

**Gold Layer Construction (Phases 2a–2c)** — ICD-10 codes are canonicalised
to decimal format, each record is annotated with a CDC-grounded status
classification, and the dataset is persisted as a queryable DuckDB vault.

**Signal Optimisation & Leakage Neutralisation (Phases 3a–3c)** — Clinical
notes are recomposed into APSO format to protect diagnostic signal from
truncation, explicit ICD-10 code strings are detected and redacted to prevent
label leakage, and both transformations are verified through human-in-the-loop
auditors and automated checks.

**Export (Phase 4)** — The Gold layer is persisted to a versioned,
schema-consistent Parquet artifact ready for downstream use.

Throughout, a Zero-Trust philosophy is applied: every assumption is validated
against evidence, every transformation is logged to an audit trail, and no
data is irreversibly discarded without documented justification.

</div>

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 📚 Dataset Reference: MedSynth (Rezaie Mianroodi et al., 2025)

> Rezaie Mianroodi, A., Rezaie, A., Todorov, N.G., Rakovski, C., & Rudzicz, F. (2025).
> *MedSynth: Realistic, Synthetic Medical Dialogue-Note Pairs.*
> arXiv:2508.01401v1 [cs.CL]
> Dataset: https://huggingface.co/datasets/Ahmad0067/MedSynth

MedSynth is a fully synthetic clinical dataset containing doctor-patient dialogue and
paired SOAP clinical notes, designed to advance automated medical documentation
research without HIPAA constraints.

---

### Dataset Composition

MedSynth contains **10,035 dialogue-note pairs** covering **2,001 unique ICD-10 codes** at publication (the full HuggingFace release used in this pipeline contains **2,037 unique codes** across **10,240 records**)
at publication. Our loaded dataset of 10,240 records reflects the full HuggingFace
release. Key structural statistics from the paper:

| Field    | Avg Tokens | Avg Sentences |
|----------|------------|---------------|
| Dialogue | 932        | 55            |
| Note     | 621        | 23            |

These figures directly validate our Phase 1c token pressure findings: both fields
exceed the ClinicalBERT 512-token limit on average, confirming that truncation is the
structural default for this dataset, not an edge case.

---

### ICD-10 Code Distribution Strategy

The authors used **uniform sampling** — generating exactly 5 dialogue-note pairs per
code across the top 2,000 most frequent ICD-10 codes from the IQVIA PharMetrics Plus
database (800 million insurance claims) — despite real-world disease distributions
being highly skewed. This was intentional:

> *"We use this uniform sampling, despite the data skew, to prevent MedSynth from
> being dominated by common diseases."*

This directly explains our Phase 1g finding of **2,037 distinct ICD-10 codes with a
minimum frequency of 5**, and why Phase 2a found **0 rare labels (freq < 2)** —
equal representation is a design property, not a data quality outcome.

---

### ICD-10 Hierarchical Structure

ICD-10 codes follow a strict hierarchy that determines the granularity and billability
of each code — a key factor in the Noisy 111 analysis performed in Phase 1b:

- **Level 1 (Chapter):** `M` → Musculoskeletal diseases
- **Level 2 (Category):** `M25` → Other joint disorders *(non-billable parent)*
- **Level 3 (Subcategory):** `M25.5` → Pain in joint *(non-billable parent)*
- **Level 4 (Specificity):** `M25.56` → Pain in knee *(non-billable parent)*
- **Level 5 (Laterality):** `M25.562` → Pain in left knee ✅ *(billable leaf code)*

Only leaf-level codes are billable. Parent codes at levels 2–4 are valid ICD-10
identifiers but cannot be submitted for reimbursement — these form the **Noisy 111**
category identified in Phase 1b (0.6% of the dataset, 60 records). The presence of
parent codes in MedSynth reflects real clinical coding practice: the authors selected
from the top 2,001 most frequent codes in actual insurance claims, some of which are
inherently category-level.

---

### SOAP Structure

Notes are generated in strict SOAP format using a four-agent GPT-4o pipeline:
Scenario Provider → Scenario Judge → Note Writer → Note Polisher. This enforced
structure accounts for the **100% SOAP extraction success** achieved in Phase 1e.
The Subjective section optionally includes sub-sections (CC, HPI, ROS), which
explains the regex variation observed in the Phase 1f Surgical Signal Auditor.

---

### Intended Use & Limitations

MedSynth is designed for **model training and development**, not as a source of
clinical truth:

> *"MedSynth should not be considered a source of reliable medical information but
> rather a tool for model development."*

The dataset targets the **Dial-2-Note** (dialogue → clinical note) and **Note-2-Dial**
(clinical note → dialogue) tasks. Our pipeline's ICD-10 validation against the CDC
FY2026 reference is complementary to this intent — establishing label quality before
any downstream classification or fine-tuning work proceeds.

</div>

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 📋 Phase 1 Overview: The Anatomy of a MedSynth Record

Before ingestion begins, we establish a forensic baseline by mapping the
expected internal architecture of a MedSynth record. This map defines where
the diagnostic signal should be located and identifies where the pipeline is
most vulnerable to truncation and leakage.

### The Dual-Source Container

A single MedSynth record is a dual-source container, joining a structured
clinical summary with a raw doctor-patient transcript. Phase 1c quantifies
whether this combined text volume exceeds ClinicalBERT's 512-token context
window.
```mermaid
graph TD
    subgraph Record["MedSynth Dataset Record"]
        direction TB
        Note["📄 Clinical Note (Structured SOAP summary)"]
        Dialogue["💬 Doctor-Patient Dialogue (Raw transcript)"]
    end
```

### The SOAP Linear Progression

The Clinical Note follows the SOAP structure. In standard format, the
Assessment section — which contains the primary diagnostic signal — appears
third, after the Subjective and Objective sections. For long notes this means
the diagnostic signal is frequently truncated by a fixed context window.
The APSO-Flip in Phase 3a addresses this by reordering to place Assessment
first.
```mermaid
graph TD
    subgraph NoteStructure["Standard SOAP Ordering"]
        S["Subjective<br/>(History / Complaints)"]
        O["Objective<br/>(Exam / Vitals)"]
        A["Assessment<br/>(DIAGNOSTIC SIGNAL)"]
        P["Plan<br/>(Treatment / Next Steps)"]
        S --> O --> A --> P
    end
    style A fill:#f96,stroke:#333,stroke-width:4px
    NoteStructure -.-> Target((Truncation Risk))
```

</div>

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🛠️ Phase 0: Infrastructure & Environment Setup

Before ingestion begins, we must ensure the **project environment** is structurally
consistent and audit-ready. This cell establishes the foundational anchor that all
subsequent pipeline phases depend on [cite: 2026-02-10].

### 🛡️ Zero-Trust Infrastructure Protocol

1. **Project Root Alignment:** Absolute pathing is enforced by walking up the directory
   tree to locate `artifacts.yaml`. This ensures that all artifacts — logs, datasets,
   and outputs — are written to deterministic locations regardless of where the notebook
   is launched from [cite: 2026-02-10].

2. **Config & Registry Ingestion:** The `src.config` singleton is loaded and validated,
   confirming that the notebook's discovered root matches the config's declared root.
   This is the Single Source of Truth for all path resolution throughout the pipeline
   [cite: 2026-02-10].

3. **Medallion Directory Health Check:** All five layers of the medallion architecture
   are verified for existence and writability — `data/raw`, `data/silver`, `data/gold`,
   `db/`, and `outputs/evaluations`. Failures here surface I/O issues before any data
   is loaded [cite: 2026-01-29].

> **🛡️ Engineering Note:** Phase 0 is a prerequisite gate, not a transformation step.
> No data is loaded or modified here. Its sole purpose is to guarantee that every
> subsequent phase — ingestion, validation, EDA, and modelling — operates from a
> verified, portable, and fully traceable environment.

</div>

In [1]:
# ==============================================================================
# PHASE 0: INFRASTRUCTURE & PATH ALIGNMENT (AUDIT READY)
# ==============================================================================

# Auto-reload src modules during development (IPython only)
%load_ext autoreload
%autoreload 2

import os
import sys
import json
from pathlib import Path
from datetime import datetime

# ==============================================================================
# 1. DISCOVER PROJECT ROOT (To enable 'import src')
# ==============================================================================
# find_project_root() walks upward to artifacts.yaml and adds root to sys.path.
# Defined in notebooks/utils/nb_setup.py — single source of truth for all notebooks.
try:
    # Minimal bootstrap: nb_setup may not be importable yet, so we add its
    # directory first if needed.
    import importlib
    _nb_utils = Path.cwd()
    while _nb_utils != _nb_utils.parent:
        if (_nb_utils / "notebooks" / "utils" / "nb_setup.py").exists():
            if str(_nb_utils) not in sys.path:
                sys.path.insert(0, str(_nb_utils))
            break
        _nb_utils = _nb_utils.parent
    from notebooks.utils.nb_setup import find_project_root
    PROJECT_ROOT = find_project_root()
    print(f"📍 Project Root: .../{PROJECT_ROOT.name}")
except Exception as e:
    print(f"❌ CRITICAL: Could not bootstrap project root: {e}")
    raise

# ==============================================================================
# 3. CONFIG & REGISTRY INGESTION (Single Source of Truth)
# ==============================================================================
try:
    from src.config import config
    
    # Production-safe validation (replaces unsafe 'assert' statements)
    required_attrs = ['project_root', 'log_path', 'log_event']
    missing = [attr for attr in required_attrs if not hasattr(config, attr)]
    if missing:
        raise RuntimeError(f"Config missing required attributes: {missing}. Verify src/config.py.")
    
    # Verify consistency between notebook discovery and config singleton
    if Path(config.project_root) != PROJECT_ROOT:
        raise RuntimeError(
            f"Config project_root mismatch!\n"
            f"  Notebook found: {PROJECT_ROOT}\n"
            f"  Config reports: {config.project_root}\n"
            f"  Action: Ensure only one artifacts.yaml exists in your hierarchy."
        )
    
    print(f"✅ Config loaded from: {config.config_path}")
    
except ImportError as e:
    print(f"❌ CRITICAL: Config import failed.")
    print(f"   Action: Verify src/config.py exists and dependencies are installed.")
    raise e
except RuntimeError as e:
    print(f"❌ CRITICAL: Config validation failed.")
    print(f"   Error: {e}")
    raise e

# ==============================================================================
# 4. SURGICAL ERA IMPORTS (With integrity verification)
# ==============================================================================
try:
    import polars as pl
    import duckdb
    
    # Defensive imports for Phase 1a (optional - will fail gracefully if missing)
    try:
        from src.data_loader import load_medsynth_raw
        print(f"   ✅ Data loader imported: load_medsynth_raw")
    except ImportError as e:
        print(f"   ⚠️  Data loader not available (will fail in Phase 1a): {e}")
        load_medsynth_raw = None
    
    print(f"✅ Core imports successful")
    print(f"   📊 Polars: {pl.__version__}")
    print(f"   🦆 DuckDB: {duckdb.__version__}")
    
except ImportError as e:
    print(f"❌ CRITICAL: Core library import failed.")
    print(f"   Action: Run 'uv pip install polars duckdb'")
    raise e

# ==============================================================================
# 5. MEDALLION DIRECTORY HEALTH CHECK (Using Config Methods)
# ==============================================================================
# Map logical names to config.resolve_path() keys for consistency
LAYER_PATH_KEYS = {
    "data/raw": ("data", "raw"),
    "data/medsynth": ("data", "medsynth"),  # <-- canonical DVC input
    "data/silver": ("data", "silver"),
    "data/gold": ("data", "gold"),
    "db/": ("data", "db"),
    "outputs/evaluations": ("outputs", "evaluations"),
}

print("\n📁 Verifying Medallion Directory Structure...")

for layer_name, path_keys in LAYER_PATH_KEYS.items():
    try:
        # Use config resolver to ensure alignment with artifacts.yaml
        dir_path = config.resolve_path(*path_keys)
        dir_path.mkdir(parents=True, exist_ok=True)
        
        # Writability check
        test_file = dir_path / ".write_test"
        test_file.touch()
        test_file.unlink()
        print(f"   ✅ {layer_name}: writable")
        
    except KeyError:
        print(f"   ❌ {layer_name}: path keys not found in config")
        raise
    except (OSError, PermissionError) as e:
        print(f"   ❌ {layer_name}: not writable - {e}")
        raise

print(f"✅ Medallion layers verified (all writable)")

# ==============================================================================
# 6. AUDIT TRAIL INITIALIZATION (With Integrity Check)
# ==============================================================================
event_details = {
    "status": "Declarative configuration respected (no os.chdir)", 
    "root": str(PROJECT_ROOT),
    "config_path": str(config.config_path),
    "markers_found": [m for m in ["artifacts.yaml", "src"] 
                     if (PROJECT_ROOT / m).exists()],
    "cwd": str(Path.cwd()),
    # Note: timestamp added automatically by config.log_event()
}

config.log_event(
    phase="Phase 0: Environment Setup",
    action="initialize_session",
    details=event_details
)

# Verify log integrity: read last line to confirm event was written
if config.log_path.exists():
    try:
        with open(config.log_path, "r", encoding="utf-8") as f:
            lines = f.readlines()
            last_entry = json.loads(lines[-1]) if lines else None
            
        if last_entry and last_entry.get("action") == "initialize_session":
            print(f"📝 Audit Trail Verified: {config.log_path}")
        else:
            print(f"⚠️  WARNING: Audit log exists but last entry mismatch")
    except (json.JSONDecodeError, IndexError) as e:
        print(f"⚠️  WARNING: Could not verify audit log content: {e}")
else:
    print(f"❌ ERROR: Audit log was not created at {config.log_path}")

# ==============================================================================
# 7. ZERO-TRUST VERDICT & BUSINESS SUMMARY
# ==============================================================================
print(f"\n✅ PHASE 0 COMPLETE: Environment secured for forensic audit (Zero-Trust)")

print(f"\n🛡️  ENVIRONMENT READINESS SUMMARY")
print(f"   " + "─" * 50)
print(f"   ✅ Project Root: {PROJECT_ROOT.name}")
print(f"   ✅ Data Layers: {len(LAYER_PATH_KEYS)} verified & writable")
print(f"   ✅ Audit Logging: Active")
print(f"   ✅ Reproducibility: Versions logged (Polars {pl.__version__}, DuckDB {duckdb.__version__})")
print(f"   " + "─" * 50)
print(f"   🎯 Business Impact: All data transformations will be")
print(f"      traceable, reproducible, and compliant with audit requirements.")

if Path.cwd() != PROJECT_ROOT:
    print(f"\n   ✅ Working directory preserved (Notebook portable)")
else:
    print(f"\n   ⚠️  Working directory matches project root (Acceptable)")

   📦 Project root already in sys.path: .../Notes_to_ICD10_prj
📍 Project Root: .../Notes_to_ICD10_prj
✅ Config loaded from: /Users/jroche/Workspace/Python/Notes_to_ICD10_prj/artifacts.yaml
   ✅ Data loader imported: load_medsynth_raw
✅ Core imports successful
   📊 Polars: 1.39.2
   🦆 DuckDB: 1.5.0

📁 Verifying Medallion Directory Structure...
   ✅ data/raw: writable
   ✅ data/medsynth: writable
   ✅ data/silver: writable
   ✅ data/gold: writable
   ✅ db/: writable
   ✅ outputs/evaluations: writable
✅ Medallion layers verified (all writable)
📝 Audit Trail Verified: /Users/jroche/Workspace/Python/Notes_to_ICD10_prj/outputs/evaluations/processing_log.jsonl

✅ PHASE 0 COMPLETE: Environment secured for forensic audit (Zero-Trust)

🛡️  ENVIRONMENT READINESS SUMMARY
   ──────────────────────────────────────────────────
   ✅ Project Root: Notes_to_ICD10_prj
   ✅ Data Layers: 6 verified & writable
   ✅ Audit Logging: Active
   ✅ Reproducibility: Versions logged (Polars 1.39.2, DuckDB 1.5.0)
   ─

## 🔒 Q8 — Diagnosis-Description Leakage (assessment-only scope)

**Problem.** The certified redaction in `src/preprocessing.py` strips ICD-10
*codes* but leaves the human-readable *description* ("Pain in left knee" for
M25.562) in the note. The model can read the diagnosis off the text instead of
inferring it. This biases the canonical 0.849 upward (**D010**, provisional).

**This section** quantifies the leak, decides redaction scope, builds a
deterministic description-redactor, and validates it with a local-LLM audit.
Decisions recorded as **D010** (canonical number) and **D011** (assessment-only
scope) in `docs/decisions.md`. All cells read gold and write only to
`outputs/audits/` — **gold is never modified here**. The notebook is the lab;
the production fix later moves into `preprocessing.py` + `prepare_data.py`.

**Architecture decision:** the deterministic rule does the redaction; the local
LLM only *audits* the result (advisory, never writes gold) — this keeps the
redaction reproducible. See `serving_local_models.md` for the LLM stack.

> Cells below the "—— SUPERSEDED ——" markers are kept as a **breadcrumb trail**
> of what we tried and why it was replaced. They are rendered as markdown (not
> runnable) so "Run All" executes only the current pipeline.

### 1 · Leakage inventory  *(builds `g`)*

Loads gold, filters to billable, re-applies the certified **code**-redaction
(`redact_icd10_sections`) so we measure the *true model-input state*, then joins
each record's CDC reference description and measures content-token overlap
between that description and the assessment.

**Result:** of 9,660 billable records, **76.2%** have the full description present
in the assessment (overlap = 1.0). This is the leak Q8 targets. *Builds `g`,
reused by later cells.*

In [2]:
# === Q8 leakage inventory: does the CDC description survive in the (redacted) assessment? ===
import polars as pl, re
from src.preprocessing import redact_icd10_sections

GOLD = PROJECT_ROOT / "data/gold/medsynth_gold_apso.parquet"
CDC  = PROJECT_ROOT / "data/ontology/icd10cm_2026.parquet"   # 2nd CDC file at data/gold/cdc_fy2026_icd10.parquet — compare later

g   = pl.read_parquet(GOLD)
cdc = pl.read_parquet(CDC)   # columns: code_no_decimal, description

# Restrict to the regime that actually trains (billable) for a fair count
g = g.filter(pl.col("code_status") == "billable")

# Re-apply the CERTIFIED existing redaction so we measure the true model-input state
g = redact_icd10_sections(g)

# Join CDC description onto each record via normalized (un-dotted) code
g = g.with_columns(pl.col("standard_icd10").str.replace_all(r"\.", "").alias("_code_nodot"))
g = g.join(cdc, left_on="_code_nodot", right_on="code_no_decimal", how="left")

def toks(s: str) -> set:
    stop = {"of","the","and","in","with","to","a","an","or","unspecified","other","site"}
    return {w for w in re.findall(r"[a-z0-9]+", (s or "").lower()) if w not in stop and len(w) > 2}

rows = g.select(["id","standard_icd10","description","assessment"]).to_dicts()
missing_desc = sum(1 for r in rows if not r["description"])
scored = []
for r in rows:
    if not r["description"]:
        continue
    d = toks(r["description"]); a = toks(r["assessment"])
    if not d:
        continue
    overlap = len(d & a) / len(d)
    scored.append((overlap, r))

n = len(scored)
def pct(thr): return sum(1 for o,_ in scored if o >= thr)
print(f"billable records:            {len(rows)}")
print(f"  missing CDC description:   {missing_desc}  (no join match — investigate if >0)")
print(f"  scored:                    {n}")
print(f"leakage by description-token overlap with assessment:")
for thr in (1.0, 0.8, 0.6, 0.5, 0.3):
    print(f"  >= {thr:>3}: {pct(thr):5d}  ({100*pct(thr)/n:4.1f}%)")

print("\n--- highest-overlap examples (description  ||  assessment) ---")
for o,r in sorted(scored, key=lambda x:-x[0])[:8]:
    print(f"[{o:.2f}] {r['standard_icd10']}: {r['description']!r}")
    print(f"        assess: {r['assessment'][:160].replace(chr(10),' ')!r}")

billable records:            9660
  missing CDC description:   0  (no join match — investigate if >0)
  scored:                    9660
leakage by description-token overlap with assessment:
  >= 1.0:  7360  (76.2%)
  >= 0.8:  7562  (78.3%)
  >= 0.6:  8092  (83.8%)
  >= 0.5:  8333  (86.3%)
  >= 0.3:  8596  (89.0%)

--- highest-overlap examples (description  ||  assessment) ---
[1.00] M25.562: 'Pain in left knee'
        assess: '**    - Pain in the left knee.    - Possible exacerbation of pre-existing mild osteoarthritis.  **'
[1.00] M25.562: 'Pain in left knee'
        assess: '**     - **Diagnosis:**      - Pain in left knee      - Contributing factors: Obesity and sedentary lifestyle.       **'
[1.00] M25.562: 'Pain in left knee'
        assess: '**  - **Primary Diagnosis:**     - Left knee pain, most likely exacerbated by underlying osteoarthritis.  - **Differential Diagnosis:**   - Meniscal tear   - Li'
[1.00] N39.0: 'Urinary tract infection, site not specified'
        assess: "**

### 2 · All-4-SOAP-sections audit

The inventory above is assessment-only. This checks whether the description also
leaks into Plan / Subjective / Objective, per-section and note-level.

**Result:** note-level exposure **77.9%** (vs 76.2% assessment-only). The other
three sections add only **1.7pp** (169 "blind-spot" records). P/S/O overlap is
mostly *scattered legitimate clinical vocabulary* (e.g. "physical therapy for the
left knee"), not contiguous label restatement — confirmed by eyeball in the
dashboard (tab 3). **This is the evidence for the D011 assessment-only scope
decision**, and for consciously accepting the 169 records as residual leak.

In [3]:
# === Q8 leakage audit — ALL 4 SOAP sections (per-section + note-level) ===
import polars as pl, re

SECTIONS = ["assessment", "plan", "subjective", "objective"]

# g already: billable, code-redacted, joined to CDC `description`. Pull all sections.
rows = g.select(["id", "standard_icd10", "description", *SECTIONS]).to_dicts()

def toks(s):
    stop = {"of","the","and","in","with","to","a","an","or","unspecified","other","site"}
    return {w for w in re.findall(r"[a-z0-9]+", (s or "").lower()) if w not in stop and len(w) > 2}

def overlap(desc_t, sec):
    st = toks(sec)
    return len(desc_t & st)/len(desc_t) if desc_t else 0.0

per_section_hits = {s: 0 for s in SECTIONS}          # count at overlap >= 1.0 per section
note_level_hits = 0                                   # description fully present in ANY section
asm_clean_but_leaks_elsewhere = 0                     # the blind-spot number
n = 0
for r in rows:
    dt = toks(r["description"])
    if not dt:
        continue
    n += 1
    sec_full = {s: overlap(dt, r[s]) >= 1.0 for s in SECTIONS}
    for s in SECTIONS:
        if sec_full[s]:
            per_section_hits[s] += 1
    if any(sec_full.values()):
        note_level_hits += 1
    if not sec_full["assessment"] and any(sec_full[s] for s in ("plan","subjective","objective")):
        asm_clean_but_leaks_elsewhere += 1

print(f"scored billable records: {n}\n")
print("FULL description present (overlap=1.0), per section:")
for s in SECTIONS:
    print(f"  {s:11s}: {per_section_hits[s]:5d}  ({100*per_section_hits[s]/n:4.1f}%)")
print(f"\nNOTE-LEVEL (present in ANY section): {note_level_hits:5d}  ({100*note_level_hits/n:4.1f}%)")
print(f"  vs assessment-only baseline:       {per_section_hits['assessment']:5d}  ({100*per_section_hits['assessment']/n:4.1f}%)")
print(f"\nBLIND SPOT — assessment clean but leaks in P/S/O: {asm_clean_but_leaks_elsewhere:5d}  ({100*asm_clean_but_leaks_elsewhere/n:4.1f}%)")
print("\n⚠ note: high `subjective` overlap may be LEGIT symptom signal, not a redactable label — needs eyeball, not just the number.")

scored billable records: 9660

FULL description present (overlap=1.0), per section:
  assessment :  7360  (76.2%)
  plan       :  1442  (14.9%)
  subjective :  1521  (15.7%)
  objective  :   605  ( 6.3%)

NOTE-LEVEL (present in ANY section):  7529  (77.9%)
  vs assessment-only baseline:        7360  (76.2%)

BLIND SPOT — assessment clean but leaks in P/S/O:   169  ( 1.7%)

⚠ note: high `subjective` overlap may be LEGIT symptom signal, not a redactable label — needs eyeball, not just the number.


### 3 · Raw-note probe

Checks whether the `note` column is raw/unredacted and what text sits adjacent to
the ICD-10 code, to source the phrasing dictionary.

**Result:** the code appears in `note` for ~26% of records, in the clean pattern
`<description> (ICD-10[ code]: CODE)`. So we can harvest MedSynth's *actual*
diagnosis phrasing from the code-tagged minority and apply it to all records.
Lead-in to the dictionary build (cells below).

In [4]:
import polars as pl, re

g = pl.read_parquet(PROJECT_ROOT / "data/gold/medsynth_gold_apso.parquet")
g = g.filter(pl.col("code_status") == "billable")

# For a few records, find the code (dotted or undotted) inside `note` and show surrounding text
def show_code_context(note, code):
    if not note: return "(empty note)"
    nod = code.replace(".", "")
    hits = []
    for pat in (re.escape(code), re.escape(nod)):
        for m in re.finditer(pat, note):
            s = max(0, m.start()-120); e = min(len(note), m.end()+40)
            hits.append(f"...{note[s:e]}...")
    return hits if hits else "(code string not found in note)"

for r in g.select(["id","standard_icd10","note"]).head(6).to_dicts():
    print(f"\n{'='*70}\n{r['standard_icd10']}  (id={r['id']})")
    ctx = show_code_context(r["note"], r["standard_icd10"])
    if isinstance(ctx, list):
        for c in ctx: print("  ", repr(c))
    else:
        print("  ", ctx)

# Also: is `note` raw? check whether ANY code still appears in note across billable set
sample = g.select(["standard_icd10","note"]).head(500).to_dicts()
present = sum(1 for r in sample if r["note"] and (r["standard_icd10"] in r["note"] or r["standard_icd10"].replace(".","") in r["note"]))
print(f"\n\ncode string present in `note` for {present}/500 sampled billable records")
print("(high => `note` is raw/unredacted and usable as the dictionary source)")


M25.562  (id=0)
   '...ests; slight discomfort with varus and valgus stress tests.\n\n**3. Assessment:**\n   - Pain in the left knee (ICD-10 code M25.562).\n   - Possible exacerbation of pre-exi...'

M25.562  (id=1)
   '...s:**\n     - Not applicable (Telemedicine)\n   \n**3. Assessment:**\n\n   - **Diagnosis:**\n     - Pain in left knee (ICD-10: M25.562)\n     - Contributing factors: Obesity a...'

M25.562  (id=2)
   (code string not found in note)

M25.562  (id=3)
   (code string not found in note)

M25.562  (id=4)
   (code string not found in note)

N39.0  (id=5)
   (code string not found in note)


code string present in `note` for 129/500 sampled billable records
(high => `note` is raw/unredacted and usable as the dictionary source)


### 4 · Review dashboard (Panel, 3 tabs)

Visual review surface — one server, three tabs: **(1)** leakage inventory,
**(2)** redaction before/after, **(3)** P/S/O leak examples (used to make the
assessment-only scope call). Compute cells build the tab HTML (`full`, `tab2`,
`tab3`); the final cell serves all three. Reuses `g`. Click *Close Dashboard*
or re-run to relaunch on a fresh port.

In [5]:
# Tab 1 compute (inventory HTML → `full`)
# ==============================================================================
# Q8 REVIEW DASHBOARD — Tab 1: Leakage Inventory
# ==============================================================================
import panel as pn
import polars as pl, re
from src.preprocessing import redact_icd10_sections

pn.extension(design='material')

GOLD = PROJECT_ROOT / "data/gold/medsynth_gold_apso.parquet"
CDC  = PROJECT_ROOT / "data/ontology/icd10cm_2026.parquet"

# ---- compute the inventory (identical logic to the audit cell) ----------------
g   = pl.read_parquet(GOLD)
cdc = pl.read_parquet(CDC)
g = g.filter(pl.col("code_status") == "billable")
g = redact_icd10_sections(g)
g = g.with_columns(pl.col("standard_icd10").str.replace_all(r"\.", "").alias("_code_nodot"))
g = g.join(cdc, left_on="_code_nodot", right_on="code_no_decimal", how="left")

def toks(s: str) -> set:
    stop = {"of","the","and","in","with","to","a","an","or","unspecified","other","site"}
    return {w for w in re.findall(r"[a-z0-9]+", (s or "").lower()) if w not in stop and len(w) > 2}

rows = g.select(["id","standard_icd10","description","assessment"]).to_dicts()
missing_desc = sum(1 for r in rows if not r["description"])
scored = []
for r in rows:
    if not r["description"]:
        continue
    d = toks(r["description"]); a = toks(r["assessment"])
    if not d:
        continue
    scored.append((len(d & a) / len(d), r))
n = len(scored)
def pct(thr): return sum(1 for o, _ in scored if o >= thr)

# ---- CSS lifted from the 1b.1 dashboard idiom --------------------------------
CSS = """
<style>
  .q8-wrap { font-family:'Segoe UI',Tahoma,sans-serif; max-width:1100px; margin:0 auto; padding:10px; }
  .q8-cards { display:flex; justify-content:center; flex-wrap:wrap; gap:20px; margin:20px 0; }
  .q8-card { background:linear-gradient(135deg,#667eea,#764ba2); border-radius:16px; padding:26px 34px;
             color:white; text-align:center; box-shadow:0 10px 40px rgba(102,126,234,.3); min-width:200px; }
  .q8-card.red    { background:linear-gradient(135deg,#f093fb,#f5576c); box-shadow:0 10px 40px rgba(245,87,108,.3); }
  .q8-card.green  { background:linear-gradient(135deg,#11998e,#38ef7d); box-shadow:0 10px 40px rgba(17,153,142,.3); }
  .q8-val   { font-size:3rem; font-weight:800; margin:8px 0; text-shadow:2px 2px 4px rgba(0,0,0,.2); }
  .q8-lab   { font-size:.85rem; text-transform:uppercase; letter-spacing:2px; opacity:.9; }
  .q8-head  { background:linear-gradient(90deg,#1a1a2e,#16213e); color:white; padding:16px 26px;
              border-radius:12px; margin:24px 0 16px; font-size:1.3rem; font-weight:600; }
  .q8-box   { background:#f8f9fa; border-left:5px solid #667eea; padding:20px; border-radius:0 12px 12px 0;
              margin:16px 0; line-height:1.7; }
  .q8-tbl   { width:100%; border-collapse:separate; border-spacing:0; margin:16px 0; border-radius:12px;
              overflow:hidden; box-shadow:0 4px 20px rgba(0,0,0,.1); }
  .q8-tbl th{ background:#1a1a2e; color:white; padding:14px 20px; text-align:left; font-size:.85rem;
              text-transform:uppercase; letter-spacing:1px; }
  .q8-tbl td{ padding:14px 20px; border-bottom:1px solid #e0e0e0; }
  .q8-tbl tr:nth-child(even){ background:#f8f9fa; }
  .q8-bar   { height:22px; background:#e0e0e0; border-radius:6px; overflow:hidden; }
  .q8-fill  { height:100%; background:linear-gradient(90deg,#f5576c,#f093fb); }
</style>
"""

thr_rows = ""
for t in (1.0, 0.8, 0.6, 0.5, 0.3):
    c = pct(t); p = 100*c/n
    thr_rows += (f"<tr><td><strong>≥ {t}</strong></td><td>{c:,}</td><td>{p:.1f}%</td>"
                 f"<td><div class='q8-bar'><div class='q8-fill' style='width:{p:.1f}%'></div></div></td></tr>")

full = f"""
{CSS}
<div class="q8-wrap">
  <div class="q8-cards">
    <div class="q8-card">       <div class="q8-lab">Billable Records</div><div class="q8-val">{len(rows):,}</div><div class="q8-lab">Scored: {n:,}</div></div>
    <div class="q8-card red">   <div class="q8-lab">Full Label Present</div><div class="q8-val">{100*pct(1.0)/n:.1f}%</div><div class="q8-lab">{pct(1.0):,} records @ overlap 1.0</div></div>
    <div class="q8-card green"> <div class="q8-lab">Missing CDC Desc</div><div class="q8-val">{missing_desc}</div><div class="q8-lab">join gaps</div></div>
  </div>

  <div class="q8-head">📊 Description-Token Overlap with Assessment</div>
  <div class="q8-box">
    For each billable record we re-apply the certified code-redaction, then measure what fraction of the
    CDC reference description's content-tokens still appear in the assessment. <strong>Overlap ≥ 1.0 means the
    full diagnosis label is present in the model input</strong> — the leak Q8 targets.
  </div>
  <table class="q8-tbl">
    <tr><th>Overlap</th><th># Records</th><th>% of Scored</th><th></th></tr>
    {thr_rows}
  </table>
  <div class="q8-box">
    <strong>Reading note.</strong> This measures description <em>presence</em>, not causal accuracy inflation —
    the magnitude only comes from the Q8 rerun on redacted gold. Token-overlap can hit 1.0 by chance on very
    short descriptions, but the 1.0 cohort is dominated by multi-token exact restatements.
  </div>
</div>
"""

In [8]:
# Tab 2 compute (matcher + before/after HTML → `tab2`; writes sample)
# ==============================================================================
# Q8 REVIEW DASHBOARD — Tab 2: Redaction before/after  (match-driven, drop-entirely)
# Relaunches the app with BOTH tabs. Writes a sample artifact; gold untouched.
# ==============================================================================
import panel as pn, polars as pl, re, json, html
from datetime import datetime

MATCH_BAR = 0.8   # the knob your eyeball pass tunes

def content_tokens(s):
    stop = {"of","the","and","in","with","to","a","an","or","unspecified","other","site"}
    return [w for w in re.findall(r"[a-z0-9]+", (s or "").lower()) if w not in stop and len(w) > 2]

def find_span(assessment, description):
    """Best fuzzy window-match of description inside assessment → (start,end) char span or None."""
    if not assessment or not description: return None
    dset = set(content_tokens(description))
    if not dset: return None
    wi = list(re.finditer(r"\S+", assessment)); words = [m.group(0) for m in wi]
    nd = len(re.findall(r"\S+", description))
    best = (0.0, None)
    for win in range(max(1, nd-1), nd+3):
        for i in range(0, len(words)-win+1):
            ctoks = set(t for w in words[i:i+win] for t in content_tokens(w))
            if not ctoks: continue
            sc = len(dset & ctoks)/len(dset)
            if sc > best[0]:
                best = (sc, (wi[i].start(), wi[i+win-1].end()))
    return best[1] if best[0] >= MATCH_BAR else None

def cleanup(text):
    text = re.sub(r'(?im)^[ \t]*[-*•]\s*$', '', text)
    text = re.sub(r'(?im)^[ \t]*\*{0,2}[A-Za-z ]{0,30}:\*{0,2}\s*$', '', text)
    return re.sub(r'\n{3,}', '\n\n', text).strip()

work = g.select(["id","standard_icd10","description","assessment"]).to_dicts()
samples, n_red, n_no = [], 0, 0
for r in work:
    span = find_span(r["assessment"], r["description"])
    if span is None:
        n_no += 1; continue
    s, e = span; before = r["assessment"]
    after = cleanup(before[:s] + before[e:]); n_red += 1
    samples.append({"id": r["id"], "code": r["standard_icd10"], "description": r["description"],
                    "removed": before[s:e], "before": before, "after": after,
                    "b_pre": before[:s], "b_hit": before[s:e], "b_post": before[e:]})

# persist sample artifact (gold NOT modified)
out = PROJECT_ROOT / "outputs" / "audits" / "q8_redaction_sample.json"
sample_out = samples[::max(1, len(samples)//40)][:40]
out.write_text(json.dumps({"generated": datetime.now().isoformat(timespec="seconds"),
                           "match_bar": MATCH_BAR, "replacement": "DROP",
                           "n_redacted": n_red, "n_nomatch": n_no, "samples": sample_out}, indent=2))

# ---- render cards ----
def esc(x): return html.escape(x).replace("\n","<br>")
cards = ""
for s in sample_out:
    cards += f"""
    <div style="background:white;border:1px solid #e0e0e0;border-radius:12px;padding:18px;margin:14px 0;box-shadow:0 2px 10px rgba(0,0,0,.06)">
      <div style="font-weight:700;color:#1565c0;margin-bottom:4px">{esc(s['code'])}
        <span style="color:#888;font-weight:400">— CDC: “{esc(s['description'])}”</span></div>
      <div style="font-size:.8rem;color:#b71c1c;margin-bottom:10px">removed span: “{esc(s['removed'][:120])}”</div>
      <div style="display:flex;gap:16px;flex-wrap:wrap">
        <div style="flex:1;min-width:300px">
          <div style="font-size:.75rem;text-transform:uppercase;letter-spacing:1px;color:#888;margin-bottom:6px">Before</div>
          <div style="background:#fafafa;border-radius:8px;padding:12px;font-size:.9rem;line-height:1.5">
            {esc(s['b_pre'])}<span style="background:#ffcdd2;color:#b71c1c;text-decoration:line-through">{esc(s['b_hit'])}</span>{esc(s['b_post'])}</div>
        </div>
        <div style="flex:1;min-width:300px">
          <div style="font-size:.75rem;text-transform:uppercase;letter-spacing:1px;color:#888;margin-bottom:6px">After</div>
          <div style="background:#e8f5e9;border-radius:8px;padding:12px;font-size:.9rem;line-height:1.5">{esc(s['after'])}</div>
        </div>
      </div>
    </div>"""

tab2 = f"""
<div style="max-width:1100px;margin:0 auto;padding:10px">
  <div style="background:linear-gradient(90deg,#1a1a2e,#16213e);color:white;padding:16px 26px;border-radius:12px;margin:16px 0;font-size:1.3rem;font-weight:600">
    ✂️ Redaction Before / After — match-driven, drop-entirely (bar={MATCH_BAR})</div>
  <div style="background:#f8f9fa;border-left:5px solid #667eea;padding:18px;border-radius:0 12px 12px 0;margin:14px 0;line-height:1.7">
    <strong>{n_red:,}</strong> rows had the description span found & removed;
    <strong>{n_no:,}</strong> cleared no span at bar {MATCH_BAR} (left untouched).
    Confirm two things per card: <strong>(1)</strong> the red strike-through is the diagnosis label and
    <strong>(2)</strong> the green “after” still keeps the real clinical content (reasoning, differentials, plan).
  </div>
  {cards}
</div>"""

In [6]:
# --- Tab 3 compute: P/S/O leak examples (reconstructed; reuses g) ---
# NOTE: this cell's source was reconstructed during the 2026-06-01 cleanup —
# the original Tab 3 ran in-kernel but was never saved to the notebook file.
import re as _re, html as _html

_SECTIONS_PSO = ["plan", "subjective", "objective"]

def _toks_pso(s):
    stop = {"of","the","and","in","with","to","a","an","or","unspecified","other","site"}
    return {w for w in _re.findall(r"[a-z0-9]+", (s or "").lower()) if w not in stop and len(w) > 2}

def _highlight(text, desc_tokens):
    def repl(m):
        w = m.group(0)
        return (f"<span style='background:#fff3cd;color:#7c4d00;font-weight:600'>{_html.escape(w)}</span>"
                if w.lower() in desc_tokens else _html.escape(w))
    return _re.sub(r"\\S+", repl, text or "").replace("\\n", "<br>")

_rows_pso = g.select(["id","standard_icd10","description","plan","subjective","objective"]).to_dicts()

def _section_examples(sec, k=12):
    hits = [r for r in _rows_pso
            if (_dt := _toks_pso(r["description"])) and len(_dt & _toks_pso(r[sec]))/len(_dt) >= 1.0]
    return len(hits), hits[::max(1, len(hits)//k)][:k]

def _build_section_html(sec):
    total, sample = _section_examples(sec)
    if sec == "subjective":
        note = ("<strong>Watch:</strong> Subjective legitimately reports symptoms — overlap here is often "
                "the clinical signal the model SHOULD use, not a label leak. Redact only true restatements.")
    elif sec == "plan":
        note = ("<strong>Watch:</strong> Plan may name the dx as part of management (\u201ccontinue management of X\u201d) "
                "— restatement = leak; treatment instructions = legit.")
    else:
        note = "<strong>Watch:</strong> Objective is exam/vitals — overlap is most likely incidental token-sharing."
    cards = ""
    for r in sample:
        dt = _toks_pso(r["description"])
        cards += f"""
        <div style="background:white;border:1px solid #e0e0e0;border-radius:12px;padding:16px;margin:12px 0;box-shadow:0 2px 10px rgba(0,0,0,.06)">
          <div style="font-weight:700;color:#1565c0">{_html.escape(r['standard_icd10'])}
            <span style="color:#888;font-weight:400">— CDC: \u201c{_html.escape(r['description'])}\u201d</span></div>
          <div style="margin-top:8px;background:#fafafa;border-radius:8px;padding:12px;font-size:.9rem;line-height:1.55">
            {_highlight(r[sec], dt)}</div>
        </div>"""
    return f"""
    <div style="max-width:1100px;margin:0 auto;padding:10px">
      <div style="background:linear-gradient(90deg,#1a1a2e,#16213e);color:white;padding:14px 24px;border-radius:12px;margin:14px 0;font-size:1.2rem;font-weight:600">
        \U0001f50e {sec.title()} — {total:,} records with full description present (overlap 1.0)</div>
      <div style="background:#fff8e1;border-left:5px solid #ffb300;padding:14px 18px;border-radius:0 12px 12px 0;margin:12px 0;line-height:1.6">{note}</div>
      {cards}
    </div>"""

tab3 = (_build_section_html("plan") + _build_section_html("subjective") + _build_section_html("objective"))
print("tab3 computed (plan + subjective + objective)")

tab3 computed (plan + subjective + objective)


In [9]:
# --- Single dashboard serve: all 3 tabs (consolidated 2026-06-01) ---
import panel as pn
pn.extension(design='material')

stop_button = pn.widgets.Button(name='🛑 Close Dashboard', button_type='danger')
header = pn.pane.Markdown("# 🔬 Q8 Leakage Review")
tabs = pn.Tabs(
    ("1 · Leakage Inventory",      pn.pane.HTML(full, sizing_mode='stretch_width')),
    ("2 · Redaction Before/After", pn.pane.HTML(tab2, sizing_mode='stretch_width')),
    ("3 · P/S/O Leak Examples",    pn.pane.HTML(tab3, sizing_mode='stretch_width')),
)
app = pn.Column(header, stop_button, tabs, sizing_mode='stretch_width')
server_thread = pn.serve(app, show=True, threaded=True, port=0, title="Q8 Leakage Review")

def stop_server(event):
    try: server_thread.stop(); print("✅ Dashboard stopped.")
    except Exception as e: print(f"⚠️  {e}")
stop_button.on_click(stop_server)
print("✅ Q8 dashboard launched — 3 tabs.")

✅ Q8 dashboard launched — 3 tabs.


Launching server at http://localhost:56521


### 5 · Persist leakage audit

Writes the per-record overlap scores + summary to
`outputs/audits/q8_leakage_scores.parquet` and `q8_leakage_summary.json` — the
reusable inventory artifact behind the 76.2% / 77.9% figures.

In [10]:
# === Persist Q8 leakage audit (per-record scores + summary) ===
import json, polars as pl
from datetime import datetime

OUT_DIR = PROJECT_ROOT / "outputs" / "audits"
OUT_DIR.mkdir(parents=True, exist_ok=True)

audit_df = pl.DataFrame([
    {"id": r["id"], "code": r["standard_icd10"], "description": r["description"],
     "overlap": round(o, 4), "assessment": r["assessment"]}
    for o, r in scored
]).sort("overlap", descending=True)

scores_path = OUT_DIR / "q8_leakage_scores.parquet"
audit_df.write_parquet(scores_path)

summary = {
    "generated": datetime.now().isoformat(timespec="seconds"),
    "gold_file": str(GOLD),
    "cdc_file": str(CDC),
    "metric": "fraction of CDC-description content-tokens present in assessment (stopwords/<=2char dropped)",
    "billable_records": len(rows),
    "missing_cdc_description": missing_desc,
    "scored": n,
    "thresholds": {str(t): pct(t) for t in (1.0, 0.8, 0.6, 0.5, 0.3)},
    "thresholds_pct": {str(t): round(100*pct(t)/n, 1) for t in (1.0, 0.8, 0.6, 0.5, 0.3)},
    "caveats": [
        "Measures description PRESENCE, not causal accuracy inflation — magnitude requires the Q8 rerun.",
        "Token-overlap; very short descriptions can hit 1.0 by chance (not the driver of the 76.2%).",
        "Redaction must anchor on CDC description, not the Diagnosis: header (header not universal)."
    ],
}
summary_path = OUT_DIR / "q8_leakage_summary.json"
summary_path.write_text(json.dumps(summary, indent=2))

print(f"✅ wrote {scores_path}  ({len(audit_df)} rows)")
print(f"✅ wrote {summary_path}")
print(f"   overlap=1.0: {summary['thresholds']['1.0']} ({summary['thresholds_pct']['1.0']}%)")

✅ wrote /Users/jroche/Workspace/Python/Notes_to_ICD10_prj/outputs/audits/q8_leakage_scores.parquet  (9660 rows)
✅ wrote /Users/jroche/Workspace/Python/Notes_to_ICD10_prj/outputs/audits/q8_leakage_summary.json
   overlap=1.0: 7360 (76.2%)


### 6 · Phrasing dictionary — harvest & inspect {Note this cee run too slowly !!!}

Harvests MedSynth's actual diagnosis phrasing (the text before each
`(ICD-10: CODE)` tag) from the ~26% code-tagged raw notes, grouping by code into
`code → {phrasings, freq}`. This is a stronger redaction anchor than the CDC
string because it captures MedSynth's *own* surface forms (e.g. title-case "Lyme
Disease" vs CDC "Lyme disease"). **Inspect before trusting** — confirms the
extraction boundary is clean. Coverage: ~1,108 of 1,926 billable codes get a
harvested phrasing; the rest fall back to CDC.

In [11]:
# === Phrasing dictionary — HARVEST & INSPECT (read-before-build) ===
# Harvest MedSynth's actual diagnosis phrasing that precedes the (ICD-10: CODE) tag,
# from the ~26% of raw notes that carry a code tag. Inspect before trusting.
import polars as pl, re
from collections import defaultdict, Counter

g   = pl.read_parquet(PROJECT_ROOT / "data/gold/medsynth_gold_apso.parquet").filter(pl.col("code_status")=="billable")
cdc = pl.read_parquet(PROJECT_ROOT / "data/ontology/icd10cm_2026.parquet")
cdc_map = dict(zip(cdc["code_no_decimal"].to_list(), cdc["description"].to_list()))

# Capture the phrase immediately before a (ICD-10[ code][:] CODE) parenthetical.
# Phrase = run of text back to the previous newline / bullet / colon (the natural left boundary).
TAG = re.compile(
    r'(?P<phrase>[^\n:•]*?)\s*'                       # phrase: no newline/colon/bullet
    r'\(\s*ICD-?10(?:[ -]?(?:CM|code))?\s*:?\s*'      # opener: (ICD-10 / (ICD-10 code: / (ICD-10:
    r'(?P<code>[A-Z][0-9]{2}\.?[0-9A-Z]{0,4})\s*\)',  # the code
    re.IGNORECASE)

def clean_phrase(p):
    # strip leading bullets/markdown/list numbers and trailing punctuation/markdown
    p = re.sub(r'^[\s\-\*•\d\.\)]+', '', p)
    p = re.sub(r'[\*\s\.]+$', '', p)
    return p.strip()

phrasings = defaultdict(Counter)
for r in g.select(["standard_icd10","note"]).to_dicts():
    note = r["note"] or ""
    for m in TAG.finditer(note):
        ph = clean_phrase(m.group("phrase"))
        if 3 <= len(ph) <= 90:                       # sanity bounds
            phrasings[r["standard_icd10"]][ph] += 1

# ---- coverage ----
all_billable = set(g["standard_icd10"].unique().to_list())
covered = set(phrasings.keys())
print(f"billable codes total:        {len(all_billable)}")
print(f"codes with harvested phrase: {len(covered)}  ({100*len(covered)/len(all_billable):.1f}%)")
print(f"codes needing CDC fallback:  {len(all_billable - covered)}")
total_phrases = sum(sum(c.values()) for c in phrasings.values())
print(f"total phrase instances harvested: {total_phrases}\n")

# ---- inspect 30: code | harvested phrasing(s) | CDC description ----
print(f"{'CODE':10} {'HARVESTED (freq)':52} {'CDC DESCRIPTION'}")
print("-"*115)
for code in list(sorted(covered))[:30]:
    variants = phrasings[code].most_common()
    shown = "; ".join(f"{p!r}×{c}" for p,c in variants[:2])
    cdc_d = cdc_map.get(code.replace(".",""), "(none)")
    print(f"{code:10} {shown[:50]:52} {cdc_d[:45]}")

# ---- how often does harvested phrasing DIFFER from CDC? (the value-add) ----
diff = same = 0
for code, ctr in phrasings.items():
    top = ctr.most_common(1)[0][0].lower()
    cdc_d = (cdc_map.get(code.replace(".",""),"") or "").lower()
    if top == cdc_d: same += 1
    else: diff += 1
print(f"\ntop harvested phrasing vs CDC:  identical={same}  differ={diff}")
print("(differ = MedSynth uses its own surface form — exactly why the dictionary beats CDC-fuzzy)")

billable codes total:        1926
codes with harvested phrase: 1108  (57.5%)
codes needing CDC fallback:  818
total phrase instances harvested: 1950

CODE       HARVESTED (freq)                                     CDC DESCRIPTION
-------------------------------------------------------------------------------------------------------------------
A04.72     'Enterocolitis due to Clostridium difficile, not s   Enterocolitis due to Clostridium difficile, n
A09        'Infectious gastroenteritis and colitis, unspecifi   Infectious gastroenteritis and colitis, unspe
A41.51     'Sepsis due to Escherichia coli [E. coli]'×1         Sepsis due to Escherichia coli [E. coli]
A41.9      'Sepsis, unspecified organism'×1                     Sepsis, unspecified organism
A63.0      'Anogenital (Venereal) Warts'×1                      Anogenital (venereal) warts
A64        'Unspecified sexually transmitted disease'×2; 'Uns   Unspecified sexually transmitted disease
A69.20     'Lyme Disease, unspecified'×

### 7 · Dictionary anomaly flag

Flags harvested phrasings that disagree sharply with CDC (content-token overlap
< 0.3). **Finding:** 197 suspects — mostly *comorbidity intrusions* (e.g. a note
labeled F12.20 cannabis-dependence whose harvested phrase is "Hypertension",
because the `(ICD-10:)` tag we harvested from was attached to a different
diagnosis in the same note). Legitimate case/article variants score high overlap
and are NOT flagged — so this cleanly separates intrusions from real variants.

In [12]:
# === Dictionary anomaly flag: harvested phrasings that DISAGREE with CDC ===
import re
def toks(s):
    stop = {"of","the","and","in","with","to","a","an","or","unspecified","other","site","due","as","the"}
    return {w for w in re.findall(r"[a-z0-9]+", (s or "").lower()) if w not in stop and len(w) > 2}

flagged = []
for code, ctr in phrasings.items():
    cdc_d = cdc_map.get(code.replace(".",""), "")
    ct = toks(cdc_d)
    for phrase, freq in ctr.items():
        pt = toks(phrase)
        if not pt or not ct:
            ov = 0.0
        else:
            ov = len(pt & ct) / len(pt)          # fraction of phrase tokens found in CDC desc
        if ov < 0.3:                              # suspect threshold
            flagged.append((ov, code, phrase, freq, cdc_d))

flagged.sort()
print(f"phrasings total:            {sum(len(c) for c in phrasings.values())}")
print(f"SUSPECT (cdc_overlap<0.3):  {len(flagged)}\n")
print(f"{'OV':4} {'CODE':9} {'FREQ':5} {'HARVESTED':40} {'CDC'}")
print("-"*110)
for ov, code, phrase, freq, cdc_d in flagged[:40]:
    print(f"{ov:.2f} {code:9} x{freq:<4} {phrase[:38]:40} {cdc_d[:35]}")

phrasings total:            1539
SUSPECT (cdc_overlap<0.3):  197

OV   CODE      FREQ  HARVESTED                                CDC
--------------------------------------------------------------------------------------------------------------
0.00 B07.0     x1    Allergic Rhinitis                        Plantar wart
0.00 C43.59    x1    Atopic Dermatitis                        Malignant melanoma of other part of
0.00 C44.41    x1    Atopic Dermatitis (Eczema)               Basal cell carcinoma of skin of sca
0.00 C76.0     x1    Non-small cell lung cancer               Malignant neoplasm of head, face an
0.00 D46.9     x1    Essential Thrombocythemia                Myelodysplastic syndrome, unspecifi
0.00 E11.51    x1    Polycystic Ovary Syndrome (PCOS)         Type 2 diabetes mellitus with diabe
0.00 E21.0     x1    Adrenal Insufficiency                    Primary hyperparathyroidism
0.00 E21.0     x1    Hypothyroidism                           Primary hyperparathyroidism
0.00 E27.40 

### 8 · Build the phrasing dictionary

Keeps phrasings with CDC-overlap ≥ 0.3, **quarantines** the < 0.3 suspects to a
side file (not used in redaction; candidate input for a future LLM-audit of the
dictionary). Writes `q8_phrasing_dictionary.json` (1,056 codes, 1,342 phrasings)
and `q8_phrasing_quarantine.json` (197). This dictionary is the redactor's
primary anchor, with CDC description as fallback for the ~870 uncovered codes.

In [13]:
# === Build phrasing dictionary: keep cdc_overlap≥0.3, quarantine <0.3 ===
import re, json
from datetime import datetime

def toks(s):
    stop = {"of","the","and","in","with","to","a","an","or","unspecified","other","site","due","as"}
    return {w for w in re.findall(r"[a-z0-9]+", (s or "").lower()) if w not in stop and len(w) > 2}

KEEP_THRESHOLD = 0.3
dictionary = {}      # code -> [{phrase, freq, cdc_overlap}]  (kept)
quarantine = []      # [{code, phrase, freq, cdc_overlap, cdc_description}]  (dropped, for audit)

for code, ctr in phrasings.items():
    cdc_d = cdc_map.get(code.replace(".",""), "")
    ct = toks(cdc_d)
    kept = []
    for phrase, freq in ctr.items():
        pt = toks(phrase)
        ov = (len(pt & ct)/len(pt)) if (pt and ct) else 0.0
        entry = {"phrase": phrase, "freq": freq, "cdc_overlap": round(ov, 3)}
        if ov >= KEEP_THRESHOLD:
            kept.append(entry)
        else:
            quarantine.append({**entry, "code": code, "cdc_description": cdc_d})
    if kept:
        dictionary[code] = sorted(kept, key=lambda e: -e["freq"])

n_codes_kept   = len(dictionary)
n_phrases_kept = sum(len(v) for v in dictionary.values())
print(f"codes with kept phrasing(s): {n_codes_kept}")
print(f"phrasings kept:              {n_phrases_kept}")
print(f"phrasings quarantined:       {len(quarantine)}")
print(f"codes now needing CDC fallback: {1926 - n_codes_kept}")

OUT = PROJECT_ROOT / "outputs" / "audits"
dict_path = OUT / "q8_phrasing_dictionary.json"
dict_path.write_text(json.dumps({
    "generated": datetime.now().isoformat(timespec="seconds"),
    "source": "harvested from raw note (ICD-10:CODE) tags, billable records",
    "keep_threshold_cdc_overlap": KEEP_THRESHOLD,
    "n_codes": n_codes_kept,
    "n_phrasings": n_phrases_kept,
    "dictionary": dictionary,
}, indent=2))

quar_path = OUT / "q8_phrasing_quarantine.json"
quar_path.write_text(json.dumps({
    "generated": datetime.now().isoformat(timespec="seconds"),
    "note": "harvested phrasings with cdc_overlap<0.3 - likely comorbidity intrusions or degenerate harvests. NOT used in redaction. Candidate input for LLM-audit-of-dictionary to rescue any genuine ones.",
    "count": len(quarantine),
    "quarantined": sorted(quarantine, key=lambda e: e["cdc_overlap"]),
}, indent=2))

print(f"\n✅ wrote {dict_path.name}  ({n_codes_kept} codes)")
print(f"✅ wrote {quar_path.name}  ({len(quarantine)} quarantined)")

multi = {c: v for c, v in dictionary.items() if len(v) > 1}
print(f"\ncodes with >1 kept phrasing: {len(multi)} - sample:")
for c, v in list(multi.items())[:5]:
    parts = []
    for e in v:
        parts.append(repr(e["phrase"]) + " x" + str(e["freq"]))
    print(f"  {c}: " + " | ".join(parts))

codes with kept phrasing(s): 1056
phrasings kept:              1342
phrasings quarantined:       197
codes now needing CDC fallback: 870

✅ wrote q8_phrasing_dictionary.json  (1056 codes)
✅ wrote q8_phrasing_quarantine.json  (197 quarantined)

codes with >1 kept phrasing: 259 - sample:
  M25.562: 'Pain in the left knee' x1 | 'Pain in left knee' x1 | 'Primary Osteoarthritis of the left knee' x1
  N39.0: 'Urinary Tract Infection, site not specified' x1 | 'Urinary Tract Infection (UTI), Site Not Specified' x1
  G89.29: 'Other Chronic Pain' x3 | 'Chronic Pain Syndrome' x1
  F11.20: 'Opioid Dependence, Uncomplicated' x1 | 'Opioid Dependence' x1
  E78.00: 'Pure Hypercholesterolemia, unspecified' x1 | 'Pure Hypercholesterolemia' x1


### 11 · Dictionary redactor (current — v3)

The deterministic description-redactor. For each record: looks up its code's
harvested phrasings (CDC fallback), matches the **full phrase** in the
assessment, and removes **all** occurrences — `[DIAGNOSIS]` placeholder when the
phrase is mid-sentence (preserves grammar), drop-the-line when it's a standalone
bullet/header. 

**Guards:** skips phrases with < 2 content tokens (the "Weakness" trap — a lone
common word would over-redact legitimate findings); those become consciously
accepted residual leak. Writes `q8_llm_audit_input_dict.json` for the audit
above. Produces: ~5,610 fired / 3,270 no-match / 780 guard-skip.

*Run order: this cell (11) produces the input that cell 9 audits and cell 10
reads. Run 11 → 9 → 10.*

In [14]:
# === Dictionary redactor v3 — remove ALL occurrences (fixes repeat-restatement leaks) ===
import polars as pl, re, json
from datetime import datetime
from collections import Counter
from src.preprocessing import redact_icd10_sections

SAMPLE_N    = 50
PLACEHOLDER = "[DIAGNOSIS]"
MIN_CONTENT_TOKENS = 2

STOP = {"of","the","and","in","with","to","a","an","or","unspecified","other","site","due","as","not",
        "elsewhere","classified","specified"}

def content_tokens(s):
    return [w for w in re.findall(r"[a-z0-9]+", (s or "").lower()) if w not in STOP and len(w) > 2]

dict_blob = json.loads((PROJECT_ROOT / "outputs/audits/q8_phrasing_dictionary.json").read_text())
DICT = dict_blob["dictionary"]
cdc  = pl.read_parquet(PROJECT_ROOT / "data/ontology/icd10cm_2026.parquet")
cdc_map = dict(zip(cdc["code_no_decimal"].to_list(), cdc["description"].to_list()))

def candidate_phrases(code):
    out = [e["phrase"] for e in DICT.get(code, [])]
    cdc_d = cdc_map.get(code.replace(".",""), "")
    if cdc_d:
        out.append(cdc_d)
    seen, uniq = set(), []
    for p in out:
        k = p.lower()
        if k in seen or len(p) < 3:
            continue
        if len(content_tokens(p)) < MIN_CONTENT_TOKENS:
            continue
        seen.add(k); uniq.append(p)
    return sorted(uniq, key=len, reverse=True)

def norm(s):
    return re.sub(r"\s+", " ", re.sub(r"[*_`]", "", (s or ""))).strip().lower()

def redact_one(text, phrase):
    """Replace every occurrence of `phrase` in text. Returns (text, n_removed)."""
    tokens = [re.escape(t) for t in phrase.split()]
    if not tokens:
        return text, 0
    rx = re.compile(r"\s+".join(tokens), re.IGNORECASE)
    n = 0
    while True:
        m = rx.search(text)
        if not m:
            break
        s, e = m.span()
        line_start = text.rfind("\n", 0, s) + 1
        line_end   = text.find("\n", e)
        if line_end == -1:
            line_end = len(text)
        line_core = re.sub(r"^[\s\-\*•\d\.\)]+", "", text[line_start:line_end])
        line_core = re.sub(r"[\s\.\*:]+$", "", line_core)
        if norm(line_core) == norm(m.group(0)):
            text = text[:line_start] + text[line_end:]
        else:
            text = text[:s] + PLACEHOLDER + text[e:]
        n += 1
        if n > 10:
            break
    return text, n

def redact_assessment(assessment, code):
    if not assessment:
        return assessment, [], "empty"
    cands = candidate_phrases(code)
    if not cands:
        return assessment, [], "guard_skip"
    text = assessment
    removed = []
    for phrase in cands:
        text, n = redact_one(text, phrase)
        if n:
            removed.append(phrase)
    text = re.sub(r"(?im)^[ \t]*[-*•]\s*$", "", text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return text, removed, ("fired" if removed else "nomatch")

g = pl.read_parquet(PROJECT_ROOT / "data/gold/medsynth_gold_apso.parquet")
g = g.filter(pl.col("code_status") == "billable")
g = redact_icd10_sections(g)
g = g.with_columns(pl.col("standard_icd10").str.replace_all(r"\.", "").alias("_nod"))
g = g.join(cdc, left_on="_nod", right_on="code_no_decimal", how="left")

records, status_ct = [], Counter()
for r in g.select(["id","standard_icd10","description","assessment"]).to_dicts():
    processed, removed, status = redact_assessment(r["assessment"], r["standard_icd10"])
    status_ct[status] += 1
    if status != "fired":
        continue
    records.append({
        "record_id": r["id"], "code": r["standard_icd10"],
        "cdc_description": r["description"], "section": "assessment",
        "removed_phrases": removed, "original": r["assessment"], "processed": processed,
        "llm_verdict": "", "llm_reasoning": "", "llm_notes": "",
    })

print("status tally:", dict(status_ct))

sample = records[::max(1, len(records)//SAMPLE_N)][:SAMPLE_N]
out = PROJECT_ROOT / "outputs" / "audits" / "q8_llm_audit_input_dict.json"
out.write_text(json.dumps({
    "generated": datetime.now().isoformat(timespec="seconds"),
    "task": "Judge whether redaction removed the diagnosis label (cdc_description). [DIAGNOSIS] placeholder = success. "
            "Fill llm_reasoning, llm_verdict, llm_notes only.",
    "verdict_values": ["clean","leak_remains","over_redacted","both"],
    "redaction_method": "dictionary+CDC, min2-token guard, remove-ALL-occurrences, [DIAGNOSIS] placeholder/drop-line",
    "n_total_fired": status_ct['fired'], "n_guard_skip": status_ct['guard_skip'],
    "n_nomatch": status_ct['nomatch'], "n_in_file": len(sample), "records": sample,
}, indent=2))
print(f"exported {len(sample)} to {out.name}")
for rec in sample:
    if len(rec["removed_phrases"]) and "rectum" in rec["cdc_description"].lower():
        print("\nC20 check:", repr(rec["processed"][:180]))

status tally: {'fired': 5610, 'nomatch': 3270, 'guard_skip': 780}
exported 50 to q8_llm_audit_input_dict.json

C20 check: '**\n- **[DIAGNOSIS] (Adenocarcinoma):** Based on colonoscopy findings and biopsy results, the patient is diagnosed with malignant neoplasm of the rectum, specifically adenocarcinoma'


In [18]:
# === A. Deterministic residual leak check: re-run leak test on redactor OUTPUT (all records) ===
# Answers: after applying the dictionary redactor to ALL billable records, how many
# still contain the full description? No LLM, no sample. The honest denominator.
import polars as pl, re, json
from collections import Counter

def toks(s):
    stop = {"of","the","and","in","with","to","a","an","or","unspecified","other","site","due","as","not",
            "elsewhere","classified","specified"}
    return {w for w in re.findall(r"[a-z0-9]+", (s or "").lower()) if w not in stop and len(w) > 2}

g_all = pl.read_parquet(PROJECT_ROOT / "data/gold/medsynth_gold_apso.parquet")
g_all = g_all.filter(pl.col("code_status") == "billable")
from src.preprocessing import redact_icd10_sections
g_all = redact_icd10_sections(g_all)
cdc = pl.read_parquet(PROJECT_ROOT / "data/ontology/icd10cm_2026.parquet")
g_all = g_all.with_columns(pl.col("standard_icd10").str.replace_all(r"\.", "").alias("_nod"))
g_all = g_all.join(cdc, left_on="_nod", right_on="code_no_decimal", how="left")

before_leak = after_leak = scored = 0
status_ct = Counter()
strata = {"fired": [], "nomatch": []}
for r in g_all.select(["id","standard_icd10","description","assessment"]).to_dicts():
    desc = r["description"]
    if not desc:
        continue
    scored += 1
    dt = toks(desc)
    if not dt:
        continue
    pre = len(dt & toks(r["assessment"]))/len(dt) >= 1.0
    if pre: before_leak += 1
    processed, removed, status = redact_assessment(r["assessment"], r["standard_icd10"])
    status_ct[status] += 1
    post = len(dt & toks(processed))/len(dt) >= 1.0
    if post: after_leak += 1
    rec = {"record_id": r["id"], "code": r["standard_icd10"], "cdc_description": desc,
           "original": r["assessment"], "processed": processed, "status": status,
           "llm_verdict":"", "llm_reasoning":"", "llm_notes":""}
    if status == "fired":
        strata["fired"].append(rec)
    elif status in ("nomatch","guard_skip"):
        strata["nomatch"].append(rec)

print("=== DETERMINISTIC RESIDUAL (all billable, token-overlap leak test) ===")
print(f"scored:                 {scored}")
print(f"status: {dict(status_ct)}")
print(f"leak BEFORE redaction:  {before_leak}  ({100*before_leak/scored:.1f}%)")
print(f"leak AFTER  redaction:  {after_leak}  ({100*after_leak/scored:.1f}%)")
print(f"leaks removed:          {before_leak-after_leak}  ({100*(before_leak-after_leak)/before_leak:.1f}% of pre-leaks)")
print(f"residual leak:          {after_leak}  ({100*after_leak/scored:.1f}% of all billable)")

# === B. Stratified LLM-audit export: ~300 fired + ~200 nomatch ===
import random
random.seed(42)
fired = strata["fired"]; nomatch = strata["nomatch"]
samp_fired   = random.sample(fired,   min(300, len(fired)))
samp_nomatch = random.sample(nomatch, min(200, len(nomatch)))
for r in samp_fired:   r["stratum"] = "fired"
for r in samp_nomatch: r["stratum"] = "nomatch_or_guardskip"
sample = samp_fired + samp_nomatch

out = PROJECT_ROOT / "outputs/audits/q8_llm_audit_input_stratified.json"
out.write_text(json.dumps({
    "task": "Per record judge if the diagnosis label (cdc_description) is absent from PROCESSED. "
            "[DIAGNOSIS] placeholder = success. NOTE: 'nomatch_or_guardskip' stratum records were NOT redacted "
            "(processed==original by design) - for those, leak_remains simply measures residual leakage, "
            "it is NOT a redaction failure. Fill llm_reasoning, llm_verdict, llm_notes only.",
    "verdict_values": ["clean","leak_remains","over_redacted","both"],
    "strata": {"fired": len(samp_fired), "nomatch_or_guardskip": len(samp_nomatch)},
    "records": sample,
}, indent=2))
print(f"\n=== STRATIFIED AUDIT EXPORT ===")
print(f"fired sample:   {len(samp_fired)}")
print(f"nomatch sample: {len(samp_nomatch)}")
print(f"wrote {out.name}  ({len(sample)} records)")

=== DETERMINISTIC RESIDUAL (all billable, token-overlap leak test) ===
scored:                 9660
status: {'fired': 5610, 'nomatch': 3270, 'guard_skip': 780}
leak BEFORE redaction:  7372  (76.3%)
leak AFTER  redaction:  2270  (23.5%)
leaks removed:          5102  (69.2% of pre-leaks)
residual leak:          2270  (23.5% of all billable)

=== STRATIFIED AUDIT EXPORT ===
fired sample:   300
nomatch sample: 200
wrote q8_llm_audit_input_stratified.json  (500 records)


In [19]:
# === Stratified LLM audit (500: ~300 fired + ~200 nomatch), reported by stratum ===
import json, requests, time
from collections import Counter, defaultdict

MODEL = "Qwen3-Coder-30B-A3B-Instruct-MLX-6bit"   # your model from the live roster
ENDPOINT = "http://127.0.0.1:8000/v1/chat/completions"
HEADERS  = {"Authorization": "Bearer claude", "Content-Type": "application/json"}
SYSTEM = (PROJECT_ROOT / "prompts/q8_audit_prompt.md").read_text()

audit_path = PROJECT_ROOT / "outputs/audits/q8_llm_audit_input_stratified.json"
data = json.loads(audit_path.read_text())

def judge(rec):
    user = (f'ORIGINAL: "{rec["original"]}"\n'
            f'cdc_description: "{rec["cdc_description"]}"\n'
            f'PROCESSED: "{rec["processed"]}"')
    body = {"model": MODEL, "temperature": 0, "top_p": 1, "max_tokens": 250,
            "messages": [{"role":"system","content":SYSTEM},{"role":"user","content":user}]}
    r = requests.post(ENDPOINT, headers=HEADERS, json=body, timeout=300)
    r.raise_for_status()
    txt = r.json()["choices"][0]["message"]["content"].strip().replace("```json","").replace("```","").strip()
    try:
        v = json.loads(txt)
        return v.get("reasoning",""), v.get("verdict",""), v.get("notes","")
    except Exception:
        return "", "PARSE_ERROR", txt[:120]

t0 = time.time()
n = len(data["records"])
for i, rec in enumerate(data["records"]):
    reasoning, verdict, notes = judge(rec)
    rec["llm_reasoning"], rec["llm_verdict"], rec["llm_notes"] = reasoning, verdict, notes
    if (i+1) % 25 == 0 or i+1 == n:
        print(f"  {i+1:3}/{n}  ({time.time()-t0:.0f}s elapsed)")

data["audit_model"] = MODEL
data["audit_temperature"] = 0
out_path = PROJECT_ROOT / "outputs/audits/q8_llm_audit_output_stratified.json"
out_path.write_text(json.dumps(data, indent=2))

# ---- report split by stratum ----
by = defaultdict(Counter)
for rec in data["records"]:
    by[rec.get("stratum","?")][rec["llm_verdict"]] += 1

print(f"\n✅ wrote {out_path.name}  ({time.time()-t0:.0f}s total)\n")
for stratum, ctr in by.items():
    tot = sum(ctr.values())
    print(f"=== stratum: {stratum}  (n={tot}) ===")
    for verdict, c in ctr.most_common():
        print(f"    {verdict:14} {c:4}  ({100*c/tot:.1f}%)")
    print()
print("Reading guide:")
print("  fired stratum  -> clean% = redaction quality (want high)")
print("  nomatch stratum-> leak_remains% = TRUE residual leak in untouched records;")
print("                    clean% here = records that never really had a contiguous leak")

   25/500  (28s elapsed)
   50/500  (55s elapsed)
   75/500  (85s elapsed)
  100/500  (115s elapsed)
  125/500  (145s elapsed)
  150/500  (176s elapsed)
  175/500  (208s elapsed)
  200/500  (240s elapsed)
  225/500  (272s elapsed)
  250/500  (308s elapsed)
  275/500  (346s elapsed)
  300/500  (382s elapsed)
  325/500  (424s elapsed)
  350/500  (463s elapsed)
  375/500  (504s elapsed)
  400/500  (552s elapsed)
  425/500  (597s elapsed)
  450/500  (644s elapsed)
  475/500  (697s elapsed)
  500/500  (749s elapsed)

✅ wrote q8_llm_audit_output_stratified.json  (749s total)

=== stratum: fired  (n=300) ===
    clean           288  (96.0%)
    over_redacted     7  (2.3%)
    leak_remains      5  (1.7%)

=== stratum: nomatch_or_guardskip  (n=200) ===
    leak_remains    122  (61.0%)
    clean            78  (39.0%)

Reading guide:
  fired stratum  -> clean% = redaction quality (want high)
  nomatch stratum-> leak_remains% = TRUE residual leak in untouched records;
                    clean% h

In [20]:
# === Categorize WHY the redactor missed the 122 confirmed-leak nomatch records ===
import json, re
from collections import Counter

data = json.loads((PROJECT_ROOT / "outputs/audits/q8_llm_audit_output_stratified.json").read_text())
misses = [r for r in data["records"]
          if r.get("stratum")=="nomatch_or_guardskip" and r["llm_verdict"]=="leak_remains"]
print(f"confirmed-leak nomatch records to diagnose: {len(misses)}\n")

STOP = {"of","the","and","in","with","to","a","an","or","unspecified","other","site","due","as","not",
        "elsewhere","classified","specified"}
def ctoks(s):
    return [w for w in re.findall(r"[a-z0-9]+", (s or "").lower()) if w not in STOP and len(w)>2]

def phrase_in(text, phrase, allow_articles=False):
    toks = [re.escape(t) for t in phrase.split()]
    if not toks: return False
    join = r"\s+(?:the\s+|a\s+|an\s+)?" if allow_articles else r"\s+"
    return re.search(join.join(toks), text or "", re.IGNORECASE) is not None

def diagnose(rec):
    code = rec["code"]; desc = rec["cdc_description"]; asm = rec["original"]
    in_dict = code in DICT
    cand = [e["phrase"] for e in DICT.get(code, [])] + ([desc] if desc else [])
    if cand and all(len(ctoks(p)) < 2 for p in cand):
        return "guard_skipped"
    for p in cand:
        if len(ctoks(p)) >= 2 and not phrase_in(asm, p, False) and phrase_in(asm, p, True):
            return "article_insertion"
    if not in_dict:
        return "on_CDC_fallback"
    return "reworded"

cats = Counter()
examples = {}
for r in misses:
    c = diagnose(r)
    cats[c] += 1
    examples.setdefault(c, []).append(r)

print("=== miss-cause breakdown ===")
for c, n in cats.most_common():
    print(f"  {c:18} {n:4}  ({100*n/len(misses):.0f}%)")

print("\n=== one example per category ===")
for c, recs in examples.items():
    r = recs[0]
    print(f"\n--- {c} ---  [{r['code']}] cdc={r['cdc_description']!r}")
    dict_phrases = [e['phrase'] for e in DICT.get(r['code'], [])]
    print(f"    dict phrasings: {dict_phrases if dict_phrases else '(none - CDC fallback)'}")
    print(f"    assessment: {r['original'][:170]!r}")
    print(f"    llm notes:  {r['llm_notes']}")

confirmed-leak nomatch records to diagnose: 122

=== miss-cause breakdown ===
  on_CDC_fallback      42  (34%)
  guard_skipped        32  (26%)
  article_insertion    24  (20%)
  reworded             24  (20%)

=== one example per category ===

--- article_insertion ---  [K31.9] cdc='Disease of stomach and duodenum, unspecified'
    dict phrasings: ['Disease of stomach and duodenum, unspecified']
    assessment: '**\n\n   - Diagnosis: Disease of the stomach and duodenum, unspecified.\n   - Differential Diagnosis includes Gastritis, Peptic Ulcer Disease, and Functional Dyspepsia.\n\n**'
    llm notes:  diagnosis label unchanged; no redaction performed

--- guard_skipped ---  [R57.8] cdc='Other shock'
    dict phrasings: ['Other Shock']
    assessment: '**\n\n**Diagnosis:**\n- Other Shock\n- Suspected Septic Shock\n- COPD exacerbation\n\n**'
    llm notes:  actual diagnosis label still present; no redaction occurred

--- on_CDC_fallback ---  [C22.1] cdc='Intrahepatic bile duct carcinoma'

In [21]:
# === Dictionary redactor v4 — adds (1) optional-article matching, (2) standalone-short-phrase recovery ===
import polars as pl, re, json
from datetime import datetime
from collections import Counter
from src.preprocessing import redact_icd10_sections

PLACEHOLDER = "[DIAGNOSIS]"
MIN_CONTENT_TOKENS = 2
STOP = {"of","the","and","in","with","to","a","an","or","unspecified","other","site","due","as","not",
        "elsewhere","classified","specified"}

def content_tokens(s):
    return [w for w in re.findall(r"[a-z0-9]+", (s or "").lower()) if w not in STOP and len(w) > 2]

dict_blob = json.loads((PROJECT_ROOT / "outputs/audits/q8_phrasing_dictionary.json").read_text())
DICT = dict_blob["dictionary"]
cdc  = pl.read_parquet(PROJECT_ROOT / "data/ontology/icd10cm_2026.parquet")
cdc_map = dict(zip(cdc["code_no_decimal"].to_list(), cdc["description"].to_list()))

def all_candidates(code):
    out = [e["phrase"] for e in DICT.get(code, [])]
    cdc_d = cdc_map.get(code.replace(".",""), "")
    if cdc_d:
        out.append(cdc_d)
    seen, uniq = set(), []
    for p in out:
        k = p.lower()
        if k in seen or len(p) < 3:
            continue
        seen.add(k); uniq.append(p)
    return sorted(uniq, key=len, reverse=True)

def norm(s):
    return re.sub(r"\s+", " ", re.sub(r"[*_`]", "", (s or ""))).strip().lower()

def make_rx(phrase):
    toks = [re.escape(t) for t in phrase.split()]
    return re.compile(r"\s+(?:the\s+|a\s+|an\s+)?".join(toks), re.IGNORECASE) if toks else None

def redact_one(text, phrase):
    rx = make_rx(phrase)
    if rx is None:
        return text, 0
    short = len(content_tokens(phrase)) < MIN_CONTENT_TOKENS
    n = 0
    while True:
        m = rx.search(text)
        if not m:
            break
        s, e = m.span()
        line_start = text.rfind("\n", 0, s) + 1
        line_end   = text.find("\n", e)
        if line_end == -1:
            line_end = len(text)
        line_core = re.sub(r"^[\s\-\*•\d\.\)]+", "", text[line_start:line_end])
        line_core = re.sub(r"[\s\.\*:]+$", "", line_core)
        is_standalone = norm(line_core) == norm(m.group(0))
        if short and not is_standalone:
            text = text[:s] + "\x00"*(e-s) + text[e:]
            continue
        if is_standalone:
            text = text[:line_start] + text[line_end:]
        else:
            text = text[:s] + PLACEHOLDER + text[e:]
        n += 1
        if n > 10:
            break
    text = text.replace("\x00", "")
    return text, n

def redact_assessment(assessment, code):
    if not assessment:
        return assessment, [], "empty"
    cands = all_candidates(code)
    if not cands:
        return assessment, [], "no_candidates"
    text = assessment
    removed = []
    for phrase in cands:
        text, k = redact_one(text, phrase)
        if k:
            removed.append(phrase)
    text = re.sub(r"(?im)^[ \t]*[-*•]\s*$", "", text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return text, removed, ("fired" if removed else "nomatch")

def toks_overlap(s):
    return {w for w in re.findall(r"[a-z0-9]+", (s or "").lower()) if w not in STOP and len(w)>2}

g = pl.read_parquet(PROJECT_ROOT / "data/gold/medsynth_gold_apso.parquet")
g = g.filter(pl.col("code_status") == "billable")
g = redact_icd10_sections(g)
g = g.with_columns(pl.col("standard_icd10").str.replace_all(r"\.", "").alias("_nod"))
g = g.join(cdc, left_on="_nod", right_on="code_no_decimal", how="left")

before = after = scored = 0
st = Counter()
for r in g.select(["id","standard_icd10","description","assessment"]).to_dicts():
    desc = r["description"]
    if not desc: continue
    dt = toks_overlap(desc)
    if not dt: continue
    scored += 1
    if len(dt & toks_overlap(r["assessment"]))/len(dt) >= 1.0: before += 1
    proc, removed, status = redact_assessment(r["assessment"], r["standard_icd10"])
    st[status] += 1
    if len(dt & toks_overlap(proc))/len(dt) >= 1.0: after += 1

print("=== v4 RESIDUAL (all billable) ===")
print(f"status: {dict(st)}")
print(f"leak BEFORE: {before}  ({100*before/scored:.1f}%)")
print(f"leak AFTER:  {after}  ({100*after/scored:.1f}%)")
print(f"removed:     {before-after}  ({100*(before-after)/before:.1f}% of pre-leaks)")
print(f"\n--- v3 was: residual 2270 (23.5%), removed 5102 (69.2%) ---")

=== v4 RESIDUAL (all billable) ===
status: {'fired': 6425, 'nomatch': 3235}
leak BEFORE: 7372  (76.3%)
leak AFTER:  1287  (13.3%)
removed:     6085  (82.5% of pre-leaks)

--- v3 was: residual 2270 (23.5%), removed 5102 (69.2%) ---


In [22]:
# === Export v4 precision-check sample: oversample article-matched / newly-risky fired records ===
import polars as pl, re, json, random
random.seed(7)

def rigid_rx(phrase):
    toks=[re.escape(t) for t in phrase.split()]
    return re.compile(r"\s+".join(toks), re.IGNORECASE) if toks else None

def why_fired(assessment, code):
    cands = all_candidates(code)
    article_only = False
    short_standalone = False
    for p in cands:
        ax = make_rx(p); rg = rigid_rx(p)
        if ax is None: continue
        a_hit = ax.search(assessment) is not None
        r_hit = rg.search(assessment) is not None if rg else False
        if a_hit and not r_hit:
            article_only = True
        if a_hit and len(content_tokens(p)) < 2:
            short_standalone = True
    if short_standalone: return "short_standalone"
    if article_only:     return "article_only"
    return "plain"

g = pl.read_parquet(PROJECT_ROOT / "data/gold/medsynth_gold_apso.parquet")
g = g.filter(pl.col("code_status") == "billable")
g = redact_icd10_sections(g)
g = g.with_columns(pl.col("standard_icd10").str.replace_all(r"\.", "").alias("_nod"))
g = g.join(cdc, left_on="_nod", right_on="code_no_decimal", how="left")

buckets = {"article_only": [], "short_standalone": [], "plain": []}
for r in g.select(["id","standard_icd10","description","assessment"]).to_dicts():
    proc, removed, status = redact_assessment(r["assessment"], r["standard_icd10"])
    if status != "fired":
        continue
    cat = why_fired(r["assessment"], r["standard_icd10"])
    buckets[cat].append({
        "record_id": r["id"], "code": r["standard_icd10"], "cdc_description": r["description"],
        "fire_reason": cat, "original": r["assessment"], "processed": proc,
        "llm_verdict":"", "llm_reasoning":"", "llm_notes":"",
    })

print("v4 fired breakdown:", {k: len(v) for k,v in buckets.items()})

def take(bucket, n): return random.sample(bucket, min(n, len(bucket)))
sample = take(buckets["article_only"],80) + take(buckets["short_standalone"],60) + take(buckets["plain"],60)
random.shuffle(sample)

out = PROJECT_ROOT / "outputs/audits/q8_v4_precision_input.json"
out.write_text(json.dumps({
    "task": "Judge if the diagnosis label (cdc_description) is absent from PROCESSED. [DIAGNOSIS] placeholder = success. "
            "Focus especially on OVER-redaction: did anything that is NOT the diagnosis label get removed or a sentence broken? "
            "Fill llm_reasoning, llm_verdict, llm_notes only.",
    "verdict_values": ["clean","leak_remains","over_redacted","both"],
    "composition": {k: len(v) for k,v in buckets.items()},
    "sample_by_reason": {r:sum(1 for x in sample if x["fire_reason"]==r) for r in ("article_only","short_standalone","plain")},
    "records": sample,
}, indent=2))
print("sample by fire_reason:", {r:sum(1 for x in sample if x['fire_reason']==r) for r in ('article_only','short_standalone','plain')})
print(f"wrote {out.name}  ({len(sample)} records)")

v4 fired breakdown: {'article_only': 652, 'short_standalone': 340, 'plain': 5433}
sample by fire_reason: {'article_only': 80, 'short_standalone': 60, 'plain': 60}
wrote q8_v4_precision_input.json  (200 records)


In [23]:
# === LLM precision audit of v4 (200 records), reported by fire_reason ===
import json, requests, time
from collections import Counter, defaultdict

MODEL = "Qwen3-Coder-30B-A3B-Instruct-MLX-6bit"   # your model
ENDPOINT = "http://127.0.0.1:8000/v1/chat/completions"
HEADERS  = {"Authorization": "Bearer claude", "Content-Type": "application/json"}
SYSTEM = (PROJECT_ROOT / "prompts/q8_audit_prompt.md").read_text()

data = json.loads((PROJECT_ROOT / "outputs/audits/q8_v4_precision_input.json").read_text())

def judge(rec):
    user = (f'ORIGINAL: "{rec["original"]}"\n'
            f'cdc_description: "{rec["cdc_description"]}"\n'
            f'PROCESSED: "{rec["processed"]}"')
    body = {"model": MODEL, "temperature": 0, "top_p": 1, "max_tokens": 250,
            "messages": [{"role":"system","content":SYSTEM},{"role":"user","content":user}]}
    r = requests.post(ENDPOINT, headers=HEADERS, json=body, timeout=300)
    r.raise_for_status()
    txt = r.json()["choices"][0]["message"]["content"].strip().replace("```json","").replace("```","").strip()
    try:
        v = json.loads(txt)
        return v.get("reasoning",""), v.get("verdict",""), v.get("notes","")
    except Exception:
        return "", "PARSE_ERROR", txt[:120]

t0=time.time(); n=len(data["records"])
for i, rec in enumerate(data["records"]):
    rec["llm_reasoning"], rec["llm_verdict"], rec["llm_notes"] = judge(rec)
    if (i+1)%25==0 or i+1==n: print(f"  {i+1:3}/{n}  ({time.time()-t0:.0f}s)")

data["audit_model"]=MODEL; data["audit_temperature"]=0
outp = PROJECT_ROOT / "outputs/audits/q8_v4_precision_output.json"
outp.write_text(json.dumps(data, indent=2))

by = defaultdict(Counter)
for rec in data["records"]:
    by[rec["fire_reason"]][rec["llm_verdict"]] += 1
overall = Counter(rec["llm_verdict"] for rec in data["records"])

print(f"\n✅ wrote {outp.name}  ({time.time()-t0:.0f}s)\n")
print("OVERALL:", dict(overall), f"  clean={100*overall['clean']/n:.0f}%")
for reason in ("article_only","short_standalone","plain"):
    ctr = by[reason]; tot=sum(ctr.values())
    if not tot: continue
    over = ctr.get("over_redacted",0)+ctr.get("both",0)
    print(f"\n{reason} (n={tot}):  clean={ctr.get('clean',0)} "
          f"leak={ctr.get('leak_remains',0)} over={ctr.get('over_redacted',0)} both={ctr.get('both',0)}"
          f"  → OVER-REDACT {100*over/tot:.0f}%")
print("\nDECISION RULE: if article_only OVER-REDACT% is low (~baseline), v4 is safe to migrate.")

   25/200  (142s)
   50/200  (191s)
   75/200  (239s)
  100/200  (282s)
  125/200  (328s)
  150/200  (373s)
  175/200  (418s)
  200/200  (461s)

✅ wrote q8_v4_precision_output.json  (461s)

OVERALL: {'clean': 183, 'leak_remains': 5, 'over_redacted': 12}   clean=92%

article_only (n=80):  clean=73 leak=1 over=6 both=0  → OVER-REDACT 8%

short_standalone (n=60):  clean=52 leak=3 over=5 both=0  → OVER-REDACT 8%

plain (n=60):  clean=58 leak=1 over=1 both=0  → OVER-REDACT 2%

DECISION RULE: if article_only OVER-REDACT% is low (~baseline), v4 is safe to migrate.


In [24]:
import json
data = json.loads((PROJECT_ROOT / "outputs/audits/q8_v4_precision_output.json").read_text())
for reason in ("article_only","short_standalone"):
    flagged = [r for r in data["records"] if r["fire_reason"]==reason and r["llm_verdict"] in ("over_redacted","both")]
    print(f"\n########## {reason}: {len(flagged)} over_redacted ##########")
    for r in flagged:
        print(f"\n[{r['code']}] cdc={r['cdc_description']!r}")
        print(f"  ORIGINAL:  {r['original'][:200]!r}")
        print(f"  PROCESSED: {r['processed'][:200]!r}")
        print(f"  notes: {r['llm_notes']}")


########## article_only: 6 over_redacted ##########

[S63.501A] cdc='Unspecified sprain of right wrist, initial encounter'
  ORIGINAL:  '**\n- Unspecified sprain of the right wrist, initial encounter.\n\n**'
  PROCESSED: '**\n\n**'
  notes: entire section removed; clinical content lost

[M79.661] cdc='Pain in right lower leg'
  ORIGINAL:  '**\n- Pain in the right lower leg due to suspected deep vein thrombosis (DVT).\n\n**'
  PROCESSED: '**\n\n**'
  notes: label removed but clinical content over-redacted

[S93.401A] cdc='Sprain of unspecified ligament of right ankle, initial encounter'
  ORIGINAL:  '**\nSprain of unspecified ligament of the right ankle, initial encounter.\n\n**'
  PROCESSED: '**\n\n**'
  notes: entire section removed; legitimate content lost

[S61.012A] cdc='Laceration without foreign body of left thumb without damage to nail, initial encounter'
  ORIGINAL:  '- Laceration without foreign body of the left thumb without damage to the nail.\n\n####'
  PROCESSED: '####'
 

In [25]:
# === Dictionary redactor v5 = v3 + article-tolerance ONLY (short-standalone fix dropped) ===
import polars as pl, re, json
from datetime import datetime
from collections import Counter
from src.preprocessing import redact_icd10_sections

PLACEHOLDER = "[DIAGNOSIS]"
MIN_CONTENT_TOKENS = 2
STOP = {"of","the","and","in","with","to","a","an","or","unspecified","other","site","due","as","not",
        "elsewhere","classified","specified"}

def content_tokens(s):
    return [w for w in re.findall(r"[a-z0-9]+", (s or "").lower()) if w not in STOP and len(w) > 2]

dict_blob = json.loads((PROJECT_ROOT / "outputs/audits/q8_phrasing_dictionary.json").read_text())
DICT = dict_blob["dictionary"]
cdc  = pl.read_parquet(PROJECT_ROOT / "data/ontology/icd10cm_2026.parquet")
cdc_map = dict(zip(cdc["code_no_decimal"].to_list(), cdc["description"].to_list()))

def candidate_phrases(code):
    out = [e["phrase"] for e in DICT.get(code, [])]
    cdc_d = cdc_map.get(code.replace(".",""), "")
    if cdc_d:
        out.append(cdc_d)
    seen, uniq = set(), []
    for p in out:
        k = p.lower()
        if k in seen or len(p) < 3:
            continue
        if len(content_tokens(p)) < MIN_CONTENT_TOKENS:
            continue
        seen.add(k); uniq.append(p)
    return sorted(uniq, key=len, reverse=True)

def norm(s):
    return re.sub(r"\s+", " ", re.sub(r"[*_`]", "", (s or ""))).strip().lower()

def make_rx(phrase):
    toks = [re.escape(t) for t in phrase.split()]
    return re.compile(r"\s+(?:the\s+|a\s+|an\s+)?".join(toks), re.IGNORECASE) if toks else None

def redact_one(text, phrase):
    rx = make_rx(phrase)
    if rx is None:
        return text, 0
    n = 0
    while True:
        m = rx.search(text)
        if not m:
            break
        s, e = m.span()
        line_start = text.rfind("\n", 0, s) + 1
        line_end   = text.find("\n", e)
        if line_end == -1:
            line_end = len(text)
        line_core = re.sub(r"^[\s\-\*•\d\.\)]+", "", text[line_start:line_end])
        line_core = re.sub(r"[\s\.\*:]+$", "", line_core)
        if norm(line_core) == norm(m.group(0)):
            text = text[:line_start] + text[line_end:]
        else:
            text = text[:s] + PLACEHOLDER + text[e:]
        n += 1
        if n > 10:
            break
    return text, n

def redact_assessment(assessment, code):
    if not assessment:
        return assessment, [], "empty"
    cands = candidate_phrases(code)
    if not cands:
        return assessment, [], "guard_skip"
    text = assessment
    removed = []
    for phrase in cands:
        text, k = redact_one(text, phrase)
        if k:
            removed.append(phrase)
    text = re.sub(r"(?im)^[ \t]*[-*•]\s*$", "", text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return text, removed, ("fired" if removed else "nomatch")

def tov(s):
    return {w for w in re.findall(r"[a-z0-9]+", (s or "").lower()) if w not in STOP and len(w)>2}

g = pl.read_parquet(PROJECT_ROOT / "data/gold/medsynth_gold_apso.parquet")
g = g.filter(pl.col("code_status") == "billable")
g = redact_icd10_sections(g)
g = g.with_columns(pl.col("standard_icd10").str.replace_all(r"\.", "").alias("_nod"))
g = g.join(cdc, left_on="_nod", right_on="code_no_decimal", how="left")

before = after = scored = 0
st = Counter()
for r in g.select(["id","standard_icd10","description","assessment"]).to_dicts():
    desc = r["description"]
    if not desc: continue
    dt = tov(desc)
    if not dt: continue
    scored += 1
    if len(dt & tov(r["assessment"]))/len(dt) >= 1.0: before += 1
    proc, removed, status = redact_assessment(r["assessment"], r["standard_icd10"])
    st[status] += 1
    if len(dt & tov(proc))/len(dt) >= 1.0: after += 1

print("=== v5 RESIDUAL (v3 + article-tolerance only) ===")
print(f"status: {dict(st)}")
print(f"leak BEFORE: {before}  ({100*before/scored:.1f}%)")
print(f"leak AFTER:  {after}  ({100*after/scored:.1f}%)")
print(f"removed:     {before-after}  ({100*(before-after)/before:.1f}% of pre-leaks)")
print(f"\n  v3: residual 23.5%, fired 5610")
print(f"  v4: residual 13.3%, fired 6425 (but short-standalone over-redacted -> dropped)")

=== v5 RESIDUAL (v3 + article-tolerance only) ===
status: {'fired': 6091, 'nomatch': 2789, 'guard_skip': 780}
leak BEFORE: 7372  (76.3%)
leak AFTER:  1799  (18.6%)
removed:     5573  (75.6% of pre-leaks)

  v3: residual 23.5%, fired 5610
  v4: residual 13.3%, fired 6425 (but short-standalone over-redacted -> dropped)


### 9 · LLM audit of the dictionary redactor

Runs the **same** local-LLM audit (Path 1 direct API, temperature 0, prompt from
`prompts/q8_audit_prompt.md`) over 50 records redacted by the current redactor
(cell 11 below), writing verdicts to `q8_llm_audit_output_dict.json`. Compare its
tally against the CDC-fuzzy **50% baseline** (in the SUPERSEDED history) to see
whether the dictionary approach improved redaction quality.

*Note:* the audit prompt declares `[DIAGNOSIS]` as the success marker, so the LLM
does not mistake the placeholder for a surviving leak.

In [15]:
# === Run the SAME LLM audit on the DICTIONARY redactor's output (temp 0) ===
import json, requests
from collections import Counter

MODEL = "Qwen3-Coder-30B-A3B-Instruct-MLX-6bit"   # same model as the baseline run
ENDPOINT = "http://127.0.0.1:8000/v1/chat/completions"
HEADERS  = {"Authorization": "Bearer claude", "Content-Type": "application/json"}
SYSTEM = (PROJECT_ROOT / "prompts/q8_audit_prompt.md").read_text()

audit_path = PROJECT_ROOT / "outputs/audits/q8_llm_audit_input_dict.json"   # ← the dictionary output
data = json.loads(audit_path.read_text())

def judge(rec):
    user = (f'ORIGINAL: "{rec["original"]}"\n'
            f'cdc_description: "{rec["cdc_description"]}"\n'
            f'PROCESSED: "{rec["processed"]}"')
    body = {"model": MODEL, "temperature": 0, "top_p": 1, "max_tokens": 250,
            "messages": [{"role":"system","content":SYSTEM},{"role":"user","content":user}]}
    r = requests.post(ENDPOINT, headers=HEADERS, json=body, timeout=300)
    r.raise_for_status()
    txt = r.json()["choices"][0]["message"]["content"].strip().replace("```json","").replace("```","").strip()
    try:
        v = json.loads(txt)
        return v.get("reasoning",""), v.get("verdict",""), v.get("notes","")
    except Exception:
        return "", "PARSE_ERROR", txt[:120]

for i, rec in enumerate(data["records"]):
    reasoning, verdict, notes = judge(rec)
    rec["llm_reasoning"], rec["llm_verdict"], rec["llm_notes"] = reasoning, verdict, notes
    print(f"[{i+1:2}/{len(data['records'])}] {rec['code']:9} → {verdict}")

data["audit_model"] = MODEL
data["audit_temperature"] = 0
out_path = PROJECT_ROOT / "outputs/audits/q8_llm_audit_output_dict.json"
out_path.write_text(json.dumps(data, indent=2))

tally = Counter(r["llm_verdict"] for r in data["records"])
print(f"\n✅ wrote {out_path.name}")
print("DICTIONARY redactor tally:", dict(tally))
print("baseline (CDC-fuzzy) was:  {'clean': 25, 'over_redacted': 7, 'leak_remains': 15, 'both': 3}")

[ 1/50] M25.562   → clean
[ 2/50] R10.13    → clean
[ 3/50] I48.0     → clean
[ 4/50] D22.5     → clean
[ 5/50] N30.00    → clean
[ 6/50] I35.0     → clean
[ 7/50] C20       → leak_remains
[ 8/50] I25.2     → leak_remains
[ 9/50] M48.02    → clean
[10/50] M25.531   → clean
[11/50] C67.9     → clean
[12/50] I11.9     → clean
[13/50] M17.9     → clean
[14/50] N28.1     → clean
[15/50] O99.824   → clean
[16/50] R79.9     → clean
[17/50] C34.12    → clean
[18/50] G56.02    → clean
[19/50] I50.31    → clean
[20/50] Z99.81    → clean
[21/50] O99.214   → clean
[22/50] R62.50    → clean
[23/50] A04.72    → clean
[24/50] C43.9     → clean
[25/50] B96.89    → clean
[26/50] J02.8     → over_redacted
[27/50] L85.3     → clean
[28/50] K50.80    → clean
[29/50] H26.492   → clean
[30/50] R26.0     → clean
[31/50] Z3A.36    → clean
[32/50] F91.3     → clean
[33/50] L50.0     → clean
[34/50] C49.9     → over_redacted
[35/50] K42.0     → clean
[36/50] F42.2     → clean
[37/50] F17.220   → clean
[38/50] 

### 10 · Leak-case reader

Pulls the records the LLM flagged (leak_remains / over_redacted / both) **with
its reasoning**, so we can judge the judge — confirm the LLM's verdicts match the
text before trusting the tally. (This is how we caught the LLM mis-reading the
`[DIAGNOSIS]` placeholder as a leak, and the genuine repeat-restatement cases.)

In [16]:
import json
data = json.loads((PROJECT_ROOT / "outputs/audits/q8_llm_audit_output_dict.json").read_text())
leaks = [r for r in data["records"] if r["llm_verdict"] == "leak_remains"]
print(f"leak_remains: {len(leaks)}\n")
for r in leaks[:10]:
    print(f"[{r['code']}] cdc: {r['cdc_description']!r}")
    print(f"  removed: {r.get('removed_phrase','')!r}")
    print(f"  processed: {r['processed'][:160]!r}")
    print(f"  notes: {r['llm_notes']}\n")

leak_remains: 3

[C20] cdc: 'Malignant neoplasm of rectum'
  removed: ''
  processed: '**\n- **[DIAGNOSIS] (Adenocarcinoma):** Based on colonoscopy findings and biopsy results, the patient is diagnosed with malignant neoplasm of the rectum, specifi'
  notes: label restated elsewhere; second occurrence not removed

[I25.2] cdc: 'Old myocardial infarction'
  removed: ''
  processed: '**\n\n   - The patient continues to experience mild exertional chest discomfort and shortness of breath, likely secondary to his history of myocardial infarction.'
  notes: label partially redacted; core diagnosis wording remains

[K63.2] cdc: 'Fistula of intestine'
  removed: ''
  processed: '**\n\n   - Medical Reasoning: The patient presents with significant symptoms including abdominal pain, fever, and purulent discharge from a surgical site. CT find'
  notes: original label removed but diagnostic statement retained



In [17]:
import json
data = json.loads((PROJECT_ROOT / "outputs/audits/q8_llm_audit_output_dict.json").read_text())
for r in data["records"]:
    if r["llm_verdict"] == "over_redacted":
        print("="*70)
        print(f"[{r['code']}] cdc: {r['cdc_description']!r}")
        print(f"  removed_phrases: {r.get('removed_phrases','')}")
        print(f"  ORIGINAL:  {r['original']!r}")
        print(f"  PROCESSED: {r['processed']!r}")
        print(f"  reasoning: {r.get('llm_reasoning','')}")
        print(f"  notes:     {r['llm_notes']}")

[J02.8] cdc: 'Acute pharyngitis due to other specified organisms'
  removed_phrases: ['Acute pharyngitis due to other specified organisms']
  ORIGINAL:  '**\nAcute pharyngitis due to other specified organisms.\n\n**'
  PROCESSED: '**\n\n**'
  reasoning: The entire assessment section was removed from PROCESSED, leaving only blank lines. The actual diagnosis wording is gone, but legitimate clinical content (the assessment section itself) was over-redacted.
  notes:     entire section removed; over-redaction
[C49.9] cdc: 'Malignant neoplasm of connective and soft tissue, unspecified'
  removed_phrases: ['Malignant neoplasm of connective and soft tissue, unspecified']
  ORIGINAL:  '**\n\nMalignant neoplasm of connective and soft tissue, unspecified.\n\n**'
  PROCESSED: '**\n\n**'
  reasoning: The entire assessment section was removed from PROCESSED, leaving only blank lines. The actual diagnosis wording is gone, but legitimate clinical content (the full assessment) was over-redacted and re

---
### —— SUPERSEDED —— CDC-fuzzy audit export

*Kept for provenance — not runnable.* The first LLM-audit export, using the
**CDC-fuzzy windowed matcher** (the tab-2 prototype). Superseded by the
dictionary redactor (§11), which matches MedSynth's actual phrasings instead of
fuzzy-matching the CDC string.

```python
# ==============================================================================
# Q8 LLM-AUDIT FILE — export records for local-LLM evaluation of the redaction
# Reuses g, find_span, cleanup from the dashboard cells (already in kernel).
# Writes outputs/audits/q8_llm_audit_input.json (gold untouched).
# LLM judges original→processed; fills llm_verdict/llm_notes. Advisory only.
# ==============================================================================
import json
from datetime import datetime

SAMPLE_N = 50

# g already has: assessment (code-redacted), description (CDC), standard_icd10, id
records = []
for r in g.select(["id","standard_icd10","description","assessment"]).to_dicts():
    if not r["description"]:
        continue
    span = find_span(r["assessment"], r["description"])
    if span is None:
        continue                                   # rule didn't fire — out of scope for this audit
    s, e = span
    processed = cleanup(r["assessment"][:s] + r["assessment"][e:])
    records.append({
        "record_id": r["id"],
        "code": r["standard_icd10"],
        "cdc_description": r["description"],         # the label that SHOULD be gone
        "section": "assessment",                     # D011 scope
        "original": r["assessment"],
        "processed": processed,
        "llm_verdict": "",                           # LLM fills: clean|leak_remains|over_redacted|both
        "llm_notes": "",                             # LLM fills: one-line reason
    })

sample = records[::max(1, len(records)//SAMPLE_N)][:SAMPLE_N]   # spread across codes

out = PROJECT_ROOT / "outputs" / "audits" / "q8_llm_audit_input.json"
out.write_text(json.dumps({
    "generated": datetime.now().isoformat(timespec="seconds"),
    "task": "For each record, judge whether the deterministic redaction (original→processed) removed the "
            "diagnosis label named by cdc_description WITHOUT removing other clinical content. Fill "
            "llm_verdict and llm_notes only. Do not edit any other field.",
    "verdict_values": ["clean", "leak_remains", "over_redacted", "both"],
    "redaction_method": "tab2 CDC-fuzzy windowed matcher, bar=0.8 (dry-run; dictionary version to follow)",
    "n_total_fired": len(records),
    "n_in_file": len(sample),
    "records": sample,
}, indent=2))

print(f"✅ wrote {out}")
print(f"   rule fired on {len(records):,} records; exported {len(sample)} for LLM audit")
print(f"   verdicts the LLM may use: clean | leak_remains | over_redacted | both")
```

### —— SUPERSEDED —— CDC-fuzzy input reader
*Kept for provenance — not runnable.* Reader for the CDC-fuzzy export above.

```python
import json
data = json.loads((PROJECT_ROOT / "outputs/audits/q8_llm_audit_input.json").read_text())
for rec in data["records"][:3]:
    print("="*70)
    print("code:           ", rec["code"])
    print("cdc_description:", rec["cdc_description"])
    print("original:       ", repr(rec["original"]))
    print("processed:      ", repr(rec["processed"]))
```

### —— SUPERSEDED —— CDC-fuzzy audit run *(the 50% baseline)*

*Kept for provenance — not runnable.* This produced the **baseline result**:
clean 25/50, leak_remains 15, over_redacted 7, both 3 — i.e. the CDC-fuzzy
matcher was clean only **50%** of the time. Its two failure modes — label
fragments left behind ("Other"/"Unspecified" not removed) and broken sentences —
are exactly what motivated the dictionary redactor (§11). The dictionary run (§9)
is the direct comparison.

```python
# === Run the local-LLM audit over the 50 records (Path 1 direct API, temp 0) ===
# Prompt loaded from prompts/q8_audit_prompt.md. LLM now writes reasoning + verdict + notes.
import json, requests

MODEL = "Qwen3-Coder-30B-A3B-Instruct-MLX-6bit"   # ← your model from the live roster
ENDPOINT = "http://127.0.0.1:8000/v1/chat/completions"
HEADERS  = {"Authorization": "Bearer claude", "Content-Type": "application/json"}

SYSTEM = (PROJECT_ROOT / "prompts/q8_audit_prompt.md").read_text()

audit_path = PROJECT_ROOT / "outputs/audits/q8_llm_audit_input.json"
data = json.loads(audit_path.read_text())

def judge(rec):
    user = (f'ORIGINAL: "{rec["original"]}"\n'
            f'cdc_description: "{rec["cdc_description"]}"\n'
            f'PROCESSED: "{rec["processed"]}"')
    body = {"model": MODEL, "temperature": 0, "top_p": 1, "max_tokens": 250,
            "messages": [{"role": "system", "content": SYSTEM},
                         {"role": "user",   "content": user}]}
    r = requests.post(ENDPOINT, headers=HEADERS, json=body, timeout=300)
    r.raise_for_status()
    txt = r.json()["choices"][0]["message"]["content"].strip()
    txt = txt.replace("```json", "").replace("```", "").strip()
    try:
        v = json.loads(txt)
        return v.get("reasoning", ""), v.get("verdict", ""), v.get("notes", "")
    except Exception:
        return "", "PARSE_ERROR", txt[:120]

for i, rec in enumerate(data["records"]):
    reasoning, verdict, notes = judge(rec)
    rec["llm_reasoning"], rec["llm_verdict"], rec["llm_notes"] = reasoning, verdict, notes
    print(f"[{i+1:2}/{len(data['records'])}] {rec['code']:9} → {verdict}")

data["audit_model"] = MODEL
data["audit_temperature"] = 0
out_path = PROJECT_ROOT / "outputs/audits/q8_llm_audit_output.json"
out_path.write_text(json.dumps(data, indent=2))

from collections import Counter
tally = Counter(r["llm_verdict"] for r in data["records"])
print(f"\n✅ wrote {out_path.name}")
print("verdict tally:", dict(tally))
```

### —— SUPERSEDED —— CDC-fuzzy leak reader
*Kept for provenance — not runnable.* Reader for the CDC-fuzzy audit output.

```python
import json
data = json.loads((PROJECT_ROOT / "outputs/audits/q8_llm_audit_output.json").read_text())
for v in ("leak_remains", "over_redacted", "both"):
    flagged = [r for r in data["records"] if r["llm_verdict"] == v]
    print(f"\n########## {v.upper()} ({len(flagged)}) ##########")
    for r in flagged[:3]:
        print(f"\n[{r['code']}] cdc: {r['cdc_description']!r}")
        print(f"  original:  {r['original'][:150]!r}")
        print(f"  processed: {r['processed'][:150]!r}")
        print(f"  REASONING: {r['llm_reasoning']}")
        print(f"  verdict:   {r['llm_verdict']}  |  {r['llm_notes']}")
```

---
### —— SUPERSEDED —— Dictionary redactor v1

*Kept for provenance — not runnable.* First dictionary redactor. Worked on most
cases but **over-redacted short/common-word descriptions** — e.g. R53.1
"Weakness" matched "severe **weakness** in lower extremities" and mangled a
legitimate finding. Fixed in v2 by the min-content-token guard.

```python
# === Dictionary-based redactor + re-export the 50-record audit file ===
# Uses the harvested phrasing dictionary (CDC fallback), full-phrase matching,
# [DIAGNOSIS] placeholder mid-sentence, drop-the-line when the phrase is standalone.
# Self-contained except for PROJECT_ROOT. Writes a NEW audit-input file (gold untouched).
import polars as pl, re, json
from datetime import datetime
from src.preprocessing import redact_icd10_sections

SAMPLE_N    = 50
PLACEHOLDER = "[DIAGNOSIS]"

# ---- load dictionary + CDC fallback ----
dict_blob = json.loads((PROJECT_ROOT / "outputs/audits/q8_phrasing_dictionary.json").read_text())
DICT = dict_blob["dictionary"]                      # code -> [{phrase, freq, cdc_overlap}]
cdc  = pl.read_parquet(PROJECT_ROOT / "data/ontology/icd10cm_2026.parquet")
cdc_map = dict(zip(cdc["code_no_decimal"].to_list(), cdc["description"].to_list()))

def candidate_phrases(code):
    """Dictionary phrasings (freq-desc) + CDC description as fallback, longest first."""
    out = []
    for e in DICT.get(code, []):
        out.append(e["phrase"])
    cdc_d = cdc_map.get(code.replace(".",""), "")
    if cdc_d:
        out.append(cdc_d)
    seen, uniq = set(), []
    for p in out:
        k = p.lower()
        if k not in seen and len(p) >= 3:
            seen.add(k); uniq.append(p)
    return sorted(uniq, key=len, reverse=True)

def norm(s):
    return re.sub(r"\s+", " ", re.sub(r"[*_`]", "", (s or ""))).strip().lower()

def redact_assessment(assessment, code):
    """Return (processed, list_of_removed_phrases). Placeholder mid-sentence; drop standalone line."""
    if not assessment:
        return assessment, []
    text = assessment
    removed = []
    for phrase in candidate_phrases(code):
        tokens = [re.escape(t) for t in phrase.split()]
        if not tokens:
            continue
        pat = r"\s+".join(tokens)
        rx = re.compile(pat, re.IGNORECASE)
        m = rx.search(text)
        if not m:
            continue
        s, e = m.span()
        line_start = text.rfind("\n", 0, s) + 1
        line_end   = text.find("\n", e)
        if line_end == -1:
            line_end = len(text)
        line = text[line_start:line_end]
        line_core = re.sub(r"^[\s\-\*•\d\.\)]+", "", line)
        line_core = re.sub(r"[\s\.\*:]+$", "", line_core)
        if norm(line_core) == norm(m.group(0)):
            text = text[:line_start] + text[line_end:]
        else:
            text = text[:s] + PLACEHOLDER + text[e:]
        removed.append(phrase)
        break

    text = re.sub(r"(?im)^[ \t]*[-*•]\s*$", "", text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    return text, removed

# ---- rebuild affected records (same basis as before) ----
g = pl.read_parquet(PROJECT_ROOT / "data/gold/medsynth_gold_apso.parquet")
g = g.filter(pl.col("code_status") == "billable")
g = redact_icd10_sections(g)
g = g.with_columns(pl.col("standard_icd10").str.replace_all(r"\.", "").alias("_nod"))
g = g.join(cdc, left_on="_nod", right_on="code_no_decimal", how="left")

records, n_fired, n_nofire = [], 0, 0
for r in g.select(["id","standard_icd10","description","assessment"]).to_dicts():
    processed, removed = redact_assessment(r["assessment"], r["standard_icd10"])
    if not removed:
        n_nofire += 1
        continue
    n_fired += 1
    records.append({
        "record_id": r["id"],
        "code": r["standard_icd10"],
        "cdc_description": r["description"],
        "section": "assessment",
        "removed_phrase": removed[0],
        "original": r["assessment"],
        "processed": processed,
        "llm_verdict": "", "llm_reasoning": "", "llm_notes": "",
    })

sample = records[::max(1, len(records)//SAMPLE_N)][:SAMPLE_N]
out = PROJECT_ROOT / "outputs" / "audits" / "q8_llm_audit_input_dict.json"
out.write_text(json.dumps({
    "generated": datetime.now().isoformat(timespec="seconds"),
    "task": "For each record, judge whether the redaction (original->processed) removed the diagnosis label "
            "named by cdc_description WITHOUT removing other clinical content. Fill llm_reasoning, llm_verdict, "
            "llm_notes only.",
    "verdict_values": ["clean","leak_remains","over_redacted","both"],
    "redaction_method": "dictionary phrasings + CDC fallback; [DIAGNOSIS] placeholder mid-sentence, drop-line standalone",
    "n_total_fired": n_fired, "n_nofire": n_nofire, "n_in_file": len(sample),
    "records": sample,
}, indent=2))

print(f"dictionary redactor fired on {n_fired:,} records; did NOT fire on {n_nofire:,}")
print(f"exported {len(sample)} to {out.name}")
print("\n--- 4 spot examples (original -> processed) ---")
for rec in sample[:4]:
    print(f"\n[{rec['code']}] removed: {rec['removed_phrase']!r}")
    print(f"  before: {rec['original'][:140]!r}")
    print(f"  after:  {rec['processed'][:140]!r}")
```

### —— SUPERSEDED —— Dictionary redactor v2 (guard added)

*Kept for provenance — not runnable.* Added the min-2-content-token guard (fixed
the "Weakness" over-redaction). But still removed only the **first** occurrence
of a phrase, so repeat-restatements leaked (e.g. C20 "malignant neoplasm of the
rectum" restated later in the same sentence). Fixed in v3 (§11) by removing all
occurrences.

```python
# === Dictionary redactor v2 — min-specificity guard, then re-export audit file ===
# Guard: skip any candidate phrase with <2 content tokens (e.g. "Weakness") to avoid
# over-redacting common clinical vocabulary. Skipped => residual leak, consciously accepted.
import polars as pl, re, json
from datetime import datetime
from src.preprocessing import redact_icd10_sections

SAMPLE_N    = 50
PLACEHOLDER = "[DIAGNOSIS]"
MIN_CONTENT_TOKENS = 2

STOP = {"of","the","and","in","with","to","a","an","or","unspecified","other","site","due","as","not",
        "elsewhere","classified","specified","unspecified"}

def content_tokens(s):
    return [w for w in re.findall(r"[a-z0-9]+", (s or "").lower()) if w not in STOP and len(w) > 2]

dict_blob = json.loads((PROJECT_ROOT / "outputs/audits/q8_phrasing_dictionary.json").read_text())
DICT = dict_blob["dictionary"]
cdc  = pl.read_parquet(PROJECT_ROOT / "data/ontology/icd10cm_2026.parquet")
cdc_map = dict(zip(cdc["code_no_decimal"].to_list(), cdc["description"].to_list()))

def candidate_phrases(code):
    out = [e["phrase"] for e in DICT.get(code, [])]
    cdc_d = cdc_map.get(code.replace(".",""), "")
    if cdc_d:
        out.append(cdc_d)
    seen, uniq = set(), []
    for p in out:
        k = p.lower()
        if k in seen or len(p) < 3:
            continue
        if len(content_tokens(p)) < MIN_CONTENT_TOKENS:
            continue
        seen.add(k); uniq.append(p)
    return sorted(uniq, key=len, reverse=True)

def norm(s):
    return re.sub(r"\s+", " ", re.sub(r"[*_`]", "", (s or ""))).strip().lower()

def redact_assessment(assessment, code):
    if not assessment:
        return assessment, [], "empty"
    text = assessment
    cands = candidate_phrases(code)
    if not cands:
        return text, [], "guard_skip"
    removed = []
    for phrase in cands:
        tokens = [re.escape(t) for t in phrase.split()]
        if not tokens:
            continue
        rx = re.compile(r"\s+".join(tokens), re.IGNORECASE)
        m = rx.search(text)
        if not m:
            continue
        s, e = m.span()
        line_start = text.rfind("\n", 0, s) + 1
        line_end   = text.find("\n", e)
        if line_end == -1:
            line_end = len(text)
        line = text[line_start:line_end]
        line_core = re.sub(r"^[\s\-\*•\d\.\)]+", "", line)
        line_core = re.sub(r"[\s\.\*:]+$", "", line_core)
        if norm(line_core) == norm(m.group(0)):
            text = text[:line_start] + text[line_end:]
        else:
            text = text[:s] + PLACEHOLDER + text[e:]
        removed.append(phrase)
        break
    text = re.sub(r"(?im)^[ \t]*[-*•]\s*$", "", text)
    text = re.sub(r"\n{3,}", "\n\n", text).strip()
    status = "fired" if removed else "nomatch"
    return text, removed, status

g = pl.read_parquet(PROJECT_ROOT / "data/gold/medsynth_gold_apso.parquet")
g = g.filter(pl.col("code_status") == "billable")
g = redact_icd10_sections(g)
g = g.with_columns(pl.col("standard_icd10").str.replace_all(r"\.", "").alias("_nod"))
g = g.join(cdc, left_on="_nod", right_on="code_no_decimal", how="left")

records = []
from collections import Counter
status_ct = Counter()
for r in g.select(["id","standard_icd10","description","assessment"]).to_dicts():
    processed, removed, status = redact_assessment(r["assessment"], r["standard_icd10"])
    status_ct[status] += 1
    if status != "fired":
        continue
    records.append({
        "record_id": r["id"], "code": r["standard_icd10"],
        "cdc_description": r["description"], "section": "assessment",
        "removed_phrase": removed[0], "original": r["assessment"], "processed": processed,
        "llm_verdict": "", "llm_reasoning": "", "llm_notes": "",
    })

print("status tally:", dict(status_ct))
print(f"fired (redacted): {status_ct['fired']:,}")
print(f"guard_skip (all phrasings too generic): {status_ct['guard_skip']:,}")
print(f"nomatch (phrase not found in assessment): {status_ct['nomatch']:,}")

sample = records[::max(1, len(records)//SAMPLE_N)][:SAMPLE_N]
out = PROJECT_ROOT / "outputs" / "audits" / "q8_llm_audit_input_dict.json"
out.write_text(json.dumps({
    "generated": datetime.now().isoformat(timespec="seconds"),
    "task": "For each record, judge whether the redaction (original->processed) removed the diagnosis label "
            "named by cdc_description WITHOUT removing other clinical content. Fill llm_reasoning, llm_verdict, llm_notes only.",
    "verdict_values": ["clean","leak_remains","over_redacted","both"],
    "redaction_method": "dictionary+CDC, min2-content-token guard, [DIAGNOSIS] placeholder/drop-line",
    "min_content_tokens": MIN_CONTENT_TOKENS,
    "n_total_fired": status_ct['fired'], "n_guard_skip": status_ct['guard_skip'],
    "n_nomatch": status_ct['nomatch'], "n_in_file": len(sample),
    "records": sample,
}, indent=2))
print(f"\nexported {len(sample)} to {out.name}")
print("\n--- spot examples incl. a previously-short one ---")
for rec in sample[:4]:
    print(f"\n[{rec['code']}] removed: {rec['removed_phrase']!r}")
    print(f"  before: {rec['original'][:130]!r}")
    print(f"  after:  {rec['processed'][:130]!r}")
```

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## Phase 1a – Ingestion, Normalisation & Structural Integrity Anchor

This phase establishes a **reproducible raw anchor** for the MedSynth dataset, moving
the pipeline from an assumption-driven workflow to a **validated, typed environment**.

### The Five Pillars of Ingestion

1. **Polars-Native Ingestion**
   The dataset is loaded using `load_medsynth_raw()` into a strictly typed **Polars
   DataFrame**. If no primary key exists, a forensic `ID` column is generated to ensure
   every record can be uniquely traced throughout the pipeline. The dataset contains
   10,240 records — 205 more than the 10,035 reported at publication, consistent with
   post-publication additions to the HuggingFace release.

2. **ICD-10 Schema Normalisation**
   The source dataset stores ICD-10 codes as plain strings without decimal points.
   This convention reflects how codes are stored in the IQVIA PharMetrics Plus
   insurance claims database used by the MedSynth authors to define their code
   selection — the raw format is intentional, not a data quality issue. Codes are
   normalised here into a **`List[String]` structure** to support multi-label
   workflows and safe Polars list operations.

3. **ICD-10 Structural Normalisation (Raw Format Preserved)**
   Codes are validated for structural plausibility — correct length (3–8 characters),
   leading alpha character, and alphanumeric content. No decimal injection or
   canonicalisation is applied at this layer; the raw format is preserved exactly
   as ingested. The original string value is retained in `ICD10_raw_original` for
   full audit traceability. Billability validation against the FY2026 CDC reference
   is deferred to Phase 1b.

4. **DuckDB Raw Vault Registration**
   The validated frame is registered as the `raw_vault` table inside DuckDB,
   providing a high-performance SQL interface for auditing and downstream
   transformations without reloading from disk.

5. **Structural Integrity Audit (Zero-Trust Validation)**
   A SQL scan validates the dataset for missing notes, empty dialogues, and malformed
   label arrays. The audit flagged **2 empty dialogues** in this run — likely
   generation artifacts from the MedSynth pipeline rather than data corruption,
   given that the MedSynth Dialogue Polisher Agent was specifically designed to
   ensure all note content is reflected in the dialogue. These records are retained
   in the raw vault and given a formal disposition in Phase 1e via sentinel
   imputation (`[NO_TRANSCRIPT_AVAILABLE]`).

### Telemetry & Traceability

Audit metrics and ingestion metadata are logged to the central telemetry system,
capturing row counts, schema types, label coverage, and timestamped execution
details. This guarantees that every pipeline run produces a **traceable ingestion
record** for later auditing.

> **Result:** A typed Polars dataset, normalised ICD-10 labels in raw decimal-free
> format, a DuckDB SQL audit layer, and a persisted Parquet anchor — forming the
> **T-zero ingestion baseline** for all downstream pipeline stages. The 2 empty
> dialogue records are explicitly tracked forward to Phase 1e for documented
> disposition.

</div>

In [ ]:
# ==============================================================================
# PHASE 1a: INGESTION PREVIEW (READ-ONLY FROM CANONICAL SOURCE)
# ==============================================================================

import polars as pl
import re
from pathlib import Path
from IPython.display import display

print("📥 Loading canonical MedSynth source...")

# 1. Load from the SINGLE source of truth
canonical_path = config.resolve_path("data", "medsynth") / "icd10_notes.parquet"

if not canonical_path.exists():
    raise FileNotFoundError(
        f"❌ Canonical file not found: {canonical_path}\n"
        f"   Run prepare_data.py first (it will download and lock the file),\n"
        f"   or run: dvc pull"
    )

df_raw = pl.read_parquet(canonical_path)
print(f" ✅ Loaded {len(df_raw):,} records from {canonical_path.name}")
print(f" 📊 Schema: {df_raw.schema}")

# 2. Validation helper
def validate_icd10_format(code: str) -> bool:
    if not code or not isinstance(code, str):
        return False
    code = code.strip().upper()
    return bool(re.match(r'^[A-Z][0-9A-Z]{2,7}$', code))

# 3. Preview
print("\n" + "=" * 80)
print("📋 RAW DATA PREVIEW (First 10 Records - from canonical source)")
print("=" * 80)

preview_df = df_raw.head(10).select([
    pl.col("ID"),
    pl.col("Note").str.slice(0, 100).alias("Note_preview"),
    pl.col("Dialogue").str.slice(0, 100).alias("Dialogue_preview"),
    pl.col("ICD10").list.first().alias("ICD10_sample"),  # show first code from list
    pl.col("ICD10").list.len().alias("ICD10_count"),
])
display(preview_df)

# 4. Structural sanity check
all_codes = df_raw.select(pl.col("ICD10").list.explode().alias("code")).drop_nulls()
total = all_codes.height
valid = all_codes.filter(pl.col("code").str.contains(r'^[A-Z][0-9A-Z]{2,7}$')).height

print(f"\n📊 STRUCTURAL SANITY:")
print(f"   Total codes: {total:,}")
print(f"   ✅ Plausible: {valid:,} ({valid/total*100:.1f}%)")
print(f"   ⚠️  Suspect: {total-valid:,}")

# 5. Log read-only access
config.log_event(
    phase="Phase 1a: Ingestion Preview",
    action="canonical_read",
    details={
        "source_path": str(canonical_path),
        "row_count": len(df_raw),
        "read_only": True,
        "immutable_source": True
    }
)

print("\n✅ Phase 1a complete: Preview loaded from locked source (no modifications)")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🔍 Phase 1b: ICD-10 Clinical Validation & "Noisy 111" Audit
**Status:** Strategic Extraction & Validation (FY2026 CDC Standard)

This phase validates every ICD-10 code in the dataset against the official **FY2026
CDC Reference**, classifying each into one of three categories and establishing the
diagnostic signal profile before any downstream transformation is applied.

### 🛠️ Technical Implementation

The validation logic uses **Polars** for vectorised string processing and classifies
every code into one of three buckets:

1. **Billable (Leaf Codes):** Valid, specific codes present in the FY2026 CDC tabular
   reference. These represent the highest-quality diagnostic signals and are the
   primary targets for downstream classification.

2. **Non-Billable Parent (The "Noisy 111"):** Codes that exist in the ICD-10 hierarchy
   but are too broad for direct billing (e.g., `J18` rather than `J18.9`). These are
   not errors — the MedSynth authors selected codes from the top 2,001 most frequent
   ICD-10 codes in real IQVIA insurance claims data, and clinicians do legitimately
   submit category-level codes in practice. The 0.6% Noisy 111 rate reflects real
   coding behaviour in the source claims data, not a generation artifact.

3. **Absent from CDC Reference:** Codes not found in the FY2026 tabular descriptions
   file. The 25 codes in this category (e.g., `T781XXA`, `H4011X1`) use the ICD-10-CM
   placeholder `X` convention for injury/trauma codes — a structural feature of the
   coding system where character positions must be held for specificity extensions.
   Their absence from the CM descriptions file is a known gap in the tabular reference,
   not a dataset error. These codes are confirmed as valid ICD-10-CM codes by the
   MedSynth paper's methodology, which used IQVIA claims filtered to ICD-10 codes
   applied after 2015.

### 📊 Validation Objectives

* **Quantify Signal Profile:** Measure the proportion of billable, parent-level, and
  reference-absent codes to establish the diagnostic label quality baseline.
* **Audit Readiness:** Confirm every code against the CDC source-of-truth, moving
  from assumption to validated schema.
* **Surface Reference Gaps:** Identify codes absent from the tabular reference for
  documented disposition before any modelling phase.

### 📊 Observed Results

* **94.3% billable** — strong signal density consistent with the paper's design intent
* **0.6% Noisy 111** — 60 records carrying category-level parent codes; top codes
  at 10 occurrences each (`E784`, `R972`, `R312`, `Z9889`, `R938`)
* **0.24% absent from CDC reference** — 25 placeholder-X injury codes; retained
  in the raw vault with disposition documented in Phase 1e

> **Note on Noisy 111:** These records are not discarded at this layer. The presence
> of parent-level codes reflects real clinical coding practice in the IQVIA source
> data. Their narratives will be assessed in downstream phases to determine whether
> sufficient clinical specificity exists to support classification tasks.

<br/>
<br/>


"The MedSynth authors selected the top 2,000 most frequent ICD-10 codes from IQVIA insurance claims without filtering for billability.
Parent codes that appear in the Noisy 111 category made the top 2,000 because they are frequently submitted in real clinical billing — this is by design, not a data quality issue (Rezaie Mianroodi et al., 2025, Section 3.2)."

</div>

In [ ]:
# ==============================================================================
# Phase 1b: ICD-10 VALIDATION & NOISY 111 AUDIT (FY2026 CDC)
# Matches prepare_data.py logic exactly
# ==============================================================================
import polars as pl
import re
from pathlib import Path
from IPython.display import display

# 1. Load CDC reference from pipeline cache
cdc_cache = config.resolve_path("data", "gold") / "cdc_fy2026_icd10.parquet"
if not cdc_cache.exists():
    raise FileNotFoundError(
        f"❌ CDC cache not found: {cdc_cache}\n"
        f"   Run: uv run python scripts/prepare_data.py"
    )
cdc_df = pl.read_parquet(cdc_cache)
print(f"✅ CDC Reference Loaded: {cdc_df.height:,} billable codes (from cache)")

# 2. Prepare dataset — df_raw from Phase 1a
df_codes = df_raw.select([
    pl.col("ID"),
    pl.col("ICD10").list.explode().alias("code")
]).with_columns(
    pl.col("code").str.replace(".", "", literal=True).str.to_uppercase().str.strip_chars().alias("code_no_decimal")
)

# 3. Build lookup sets (identical to prepare_data.py)
billable_set = set(cdc_df["code_no_decimal"].to_list())
parent_set = {c[:3] for c in billable_set if len(c) > 3 and c[:3] not in billable_set}

# 4. Classification function — EXACT match to headless
def classify_code(code: str) -> str:
    if not code:
        return "invalid_or_malformed"
    c = code.strip().upper().replace(".", "")
    if c in billable_set:
        return "billable"
    if len(c) == 3 and any(x.startswith(c) for x in billable_set):
        return "non_billable_parent (Noisy 111)"
    if re.match(r"^[A-Z][0-9]{2}[0-9A-Z]*X[0-9A-Z]*$", c):
        return "placeholder_x"
    return "invalid_or_malformed"

# 5. Classify all codes
df_checked = df_codes.with_columns(
    pl.col("code_no_decimal").map_elements(classify_code, return_dtype=pl.Utf8).alias("status")
)

# Enrich df_checked with CDC descriptions — available to ALL downstream cells.
df_checked = df_checked.join(
    cdc_df.select(["code_no_decimal", "description"]),
    on="code_no_decimal",
    how="left"
)
print(f"   ✅ df_checked enriched with CDC descriptions ({len(df_checked.columns)} columns)")

# 6. Summary
summary = df_checked.group_by("status").agg(pl.len().alias("count")).with_columns(
    (pl.col("count") / pl.col("count").sum() * 100).round(2).alias("pct")
).sort("count", descending=True)
print("\n📊 FINAL VALIDATION SUMMARY:")
display(summary)

# 7. Noisy 111 report
noisy = df_checked.filter(pl.col("status") == "non_billable_parent (Noisy 111)")
if noisy.height > 0:
    print("\n🚨 NOISY 111 FREQUENCY (Top 10):")
    display(
        noisy.group_by("code")
        .agg(pl.len().alias("occurrences"))
        .sort("occurrences", descending=True)
        .head(10)
    )

# 8. Placeholder X report
placeholders = df_checked.filter(pl.col("status") == "placeholder_x")
if placeholders.height > 0:
    print(f"\n🔤 PLACEHOLDER X codes: {placeholders.height}")
    display(placeholders.select(["code"]).unique().head(10))

# 9. Invalid debug
invalids = df_checked.filter(pl.col("status") == "invalid_or_malformed")
if invalids.height > 0:
    print(f"\n🔍 DEBUG: {invalids.height} invalid codes:")
    display(invalids.select(["code", "code_no_decimal"]).unique().head(10))

# 10. Audit trail
config.log_event(
    phase="Phase 1b: ICD-10 Validation",
    action="cdc_validation_complete",
    details={
        "cdc_reference_codes": cdc_df.height,
        "total_codes_checked": df_checked.height,
        "validation_summary": summary.to_dicts(),
        "noisy_111_count": noisy.height,
        "placeholder_x_count": placeholders.height,
        "invalid_count": invalids.height,
        "read_only": True,
        "matches_pipeline": True
    }
)
print("\n✅ Phase 1b complete: validated against FY2026 CDC reference (pipeline-aligned)")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 📊 Phase 1b.1: Dataset Composition Summary

Stakeholder-facing summary of dataset composition. Confirms alignment
with the MedSynth paper: 5 records per billable ICD-10 code, uniform
sampling across 2,037 unique codes — 2,026 codes appear exactly 5 times and 11 codes appear 10 times, giving 9,660 billable records in total.

</div>

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 📊 Phase 1b.1: ICD-10 Resolution Summary

Detailed breakdown of dataset composition explaining why some ICD-10
codes are billable and others are not, with counts at each resolution
level (chapter, ICD-3, full ICD-10).

</div>

In [ ]:
# ==============================================================================
# PHASE 1b.1: DATASET COMPOSITION & ICD-10 RESOLUTION SUMMARY
# ==============================================================================
# This cell provides a clear, validated summary of dataset composition for
# non-technical reviewers, with special attention to explaining WHY some
# ICD-10 codes are billable and others are not.
# ==============================================================================

print("📊 DATASET COMPOSITION SUMMARY")
print("=" * 70)

# ------------------------------------------------------------------------------
# 1. TOTAL RECORDS & UNIQUE CODES
# ------------------------------------------------------------------------------
total_records = len(df_raw)

unique_codes_df = (
    df_raw
    .select(pl.col("ICD10").list.explode().alias("code"))
    .filter(pl.col("code").is_not_null())
    .select(pl.col("code").n_unique().alias("unique_count"))
)
unique_code_count = unique_codes_df.item()

print(f"\n📋 Total Records:       {total_records:,}")
print(f"🏷️  Unique ICD-10 Codes: {unique_code_count:,}")

# ------------------------------------------------------------------------------
# 2. CODE FREQUENCY DISTRIBUTION
# ------------------------------------------------------------------------------
code_frequencies = (
    df_raw
    .select(pl.col("ICD10").list.explode().alias("code"))
    .filter(pl.col("code").is_not_null())
    .group_by("code")
    .agg(pl.len().alias("occurrences"))
    .group_by("occurrences")
    .agg(pl.len().alias("num_codes"))
    .sort("occurrences")
)

print(f"\n📈 Code Frequency Distribution:")
print("-" * 50)
print(f"   {'Occurrences':<15} {'# of Codes':<15} {'% of Codes':<10}")
print("-" * 50)

for row in code_frequencies.iter_rows(named=True):
    occ = row["occurrences"]
    num = row["num_codes"]
    pct = (num / unique_code_count) * 100
    print(f"   {occ:<15} {num:<15,} {pct:.1f}%")

print("-" * 50)

# ------------------------------------------------------------------------------
# 3. ICD-10 BILLABILITY BREAKDOWN (From Phase 1b validation)
# ------------------------------------------------------------------------------
print("\n" + "=" * 70)
print("🏥 ICD-10 CODE RESOLUTION: BILLABLE vs NON-BILLABLE")
print("=" * 70)

# Get counts from df_checked (created in Phase 1b)
billable_records = df_checked.filter(pl.col("status") == "billable").height
noisy_111_records = df_checked.filter(pl.col("status") == "non_billable_parent (Noisy 111)").height
invalid_records = df_checked.filter(pl.col("status") == "invalid_or_malformed").height

billable_pct = (billable_records / total_records) * 100
noisy_111_pct = (noisy_111_records / total_records) * 100
invalid_pct = (invalid_records / total_records) * 100

print(f"""
┌─────────────────────────────────────────────────────────────────────┐
│  STATUS                        │  RECORDS   │  PERCENTAGE           │
├─────────────────────────────────────────────────────────────────────┤
│  ✅ Billable (Leaf Codes)      │  {billable_records:>7,}   │  {billable_pct:>5.1f}%              │
│  ⚠️  Non-Billable (Parent)      │  {noisy_111_records:>7,}   │  {noisy_111_pct:>5.1f}%               │
│  ❓ Absent from CDC Reference  │  {invalid_records:>7,}   │  {invalid_pct:>5.1f}%               │
├─────────────────────────────────────────────────────────────────────┤
│  TOTAL                         │  {total_records:>7,}   │  100.0%              │
└─────────────────────────────────────────────────────────────────────┘
""")

# ------------------------------------------------------------------------------
# 4. PLAIN-LANGUAGE EXPLANATION OF ICD-10 HIERARCHY
# ------------------------------------------------------------------------------
print("=" * 70)
print("📚 UNDERSTANDING ICD-10 CODE RESOLUTION (Plain Language)")
print("=" * 70)

print("""
ICD-10 codes follow a HIERARCHICAL structure, like a folder system:

   Level 1 (Chapter):     M        → "Musculoskeletal diseases" (broad category)
   Level 2 (Category):    M25      → "Other joint disorders"
   Level 3 (Subcategory): M25.5    → "Pain in joint"
   Level 4 (Specificity): M25.56   → "Pain in knee"
   Level 5 (Laterality):  M25.562  → "Pain in LEFT knee" ✅ BILLABLE

ONLY the most specific "leaf" codes can be submitted for insurance billing.
Parent codes (M25, M25.5, M25.56) are valid ICD-10 identifiers but are
NOT billable — they exist to organize the hierarchy, not for claims.

WHY DO PARENT CODES APPEAR IN THIS DATASET?

The MedSynth authors selected the "top 2,000 most frequent ICD-10 codes"
from real insurance claims (IQVIA PharMetrics Plus database). In practice,
clinicians sometimes submit category-level codes when:

   • The specific subtype wasn't clinically distinguished
   • Documentation didn't support a more specific code
   • The condition genuinely spans multiple subtypes

These parent codes made the top 2,000 because they're FREQUENTLY USED
in real billing — this is a feature of real-world coding practice, not
a dataset error.
""")

# ------------------------------------------------------------------------------
# 5. CONCRETE EXAMPLES FROM THE DATASET
# ------------------------------------------------------------------------------
print("=" * 70)
print("🔬 CONCRETE EXAMPLES FROM THIS DATASET")
print("=" * 70)

# Get sample billable codes with descriptions
billable_samples = (
    df_checked
    .filter(pl.col("status") == "billable")
    .select(["code", "description"])
    .unique()
    .head(5)
)

print("\n✅ BILLABLE (Leaf) Codes — Ready for classification tasks:\n")
for row in billable_samples.iter_rows(named=True):
    code = row["code"]
    desc = row["description"][:50] + "..." if len(row["description"]) > 50 else row["description"]
    print(f"   {code:<10} → {desc}")

# Get sample parent codes
noisy_samples = (
    df_checked
    .filter(pl.col("status") == "non_billable_parent (Noisy 111)")
    .select(["code"])
    .unique()
    .head(5)
)

print("\n⚠️  NON-BILLABLE (Parent) Codes — The 'Noisy 111':\n")

# For parent codes, we need to show what they WOULD resolve to
for row in noisy_samples.iter_rows(named=True):
    parent_code = row["code"]
    # Find a billable child code that starts with this parent
    child_example = (
        cdc_df
        .filter(pl.col("code_no_decimal").str.starts_with(parent_code))
        .head(1)
    )
    if child_example.height > 0:
        child = child_example.row(0, named=True)
        child_code = child["code_no_decimal"]
        child_desc = child["description"][:40] + "..." if len(child["description"]) > 40 else child["description"]
        print(f"   {parent_code:<10} → Parent code (not billable)")
        print(f"   └── {child_code:<10} → Example billable child: {child_desc}")
        print()

# ------------------------------------------------------------------------------
# 6. DISPOSITION DECISION
# ------------------------------------------------------------------------------
print("=" * 70)
print("⚖️  DISPOSITION DECISION: WHAT DO WE DO WITH THESE CODES?")
print("=" * 70)

print(f"""
┌─────────────────────────────────────────────────────────────────────┐
│  CODE TYPE                │  COUNT  │  DISPOSITION                 │
├─────────────────────────────────────────────────────────────────────┤
│  ✅ Billable (Leaf)       │  {billable_records:>5,}  │  RETAIN — primary targets    │
│                           │         │  for ICD-10 classification   │
├─────────────────────────────────────────────────────────────────────┤
│  ⚠️  Non-Billable (Parent) │  {noisy_111_records:>5,}  │  RETAIN — narratives still   │
│                           │         │  contain valid clinical      │
│                           │         │  signal; downstream models   │
│                           │         │  can learn category-level    │
│                           │         │  patterns                    │
├─────────────────────────────────────────────────────────────────────┤
│  ❓ Absent from CDC Ref   │  {invalid_records:>5,}  │  RETAIN — valid ICD-10-CM    │
│                           │         │  placeholder-X codes (e.g.,  │
│                           │         │  T814XXA); absence from      │
│                           │         │  tabular file is a known     │
│                           │         │  reference gap, not an error │
└─────────────────────────────────────────────────────────────────────┘

DECISION: ALL {total_records:,} RECORDS ARE RETAINED.

No records are removed at this stage. The 555 parent-code records and
25 placeholder-X records carry valid clinical narratives that can inform
downstream classification, even if their labels are less specific than
ideal. Removing them would discard usable signal.

For downstream modelling, you may choose to:
   • Train on all records (recommended for robustness)
   • Filter to billable-only for strict classification benchmarks
   • Use parent codes as a separate evaluation cohort
""")

# ------------------------------------------------------------------------------
# 7. VALIDATION AGAINST PAPER METHODOLOGY
# ------------------------------------------------------------------------------
print("\n" + "=" * 70)
print("📄 VALIDATION AGAINST MEDSYNTH PAPER METHODOLOGY")
print("=" * 70)

codes_with_5 = code_frequencies.filter(pl.col("occurrences") == 5)
codes_with_5_count = codes_with_5.select("num_codes").item() if codes_with_5.height > 0 else 0

codes_with_10 = code_frequencies.filter(pl.col("occurrences") == 10)
codes_with_10_count = codes_with_10.select("num_codes").item() if codes_with_10.height > 0 else 0

print(f"""
   Paper states: "5 dialogue-note pairs for each [ICD-10 code]"
   Paper states: "2,001 unique ICD-10 codes" at publication
   Paper states: "10,035 dialogue-note pairs" at publication

   Our dataset (HuggingFace release):
   ─────────────────────────────────────────────────────────
   Total records:           {total_records:,}  (+{total_records - 10035} vs paper)
   Unique codes:            {unique_code_count:,}  (+{unique_code_count - 2001} vs paper)
   Codes with 5 records:    {codes_with_5_count:,}
   Codes with 10 records:   {codes_with_10_count:,}
   ─────────────────────────────────────────────────────────
   
   The additional records reflect post-publication additions to the
   HuggingFace release. The 5/10 frequency pattern confirms the paper's
   uniform sampling methodology.
""")

# ------------------------------------------------------------------------------
# 8. AUDIT TRAIL
# ------------------------------------------------------------------------------
config.log_event(
    phase="Phase 1b.1: Composition & Resolution Summary",
    action="dataset_composition_validated",
    details={
        "total_records": total_records,
        "unique_icd10_codes": unique_code_count,
        "billable_records": billable_records,
        "noisy_111_records": noisy_111_records,
        "invalid_records": invalid_records,
        "disposition": "all_records_retained",
        "frequency_distribution": code_frequencies.to_dicts(),
        "codes_with_5_occurrences": codes_with_5_count,
        "codes_with_10_occurrences": codes_with_10_count,
    }
)

# ==============================================================================
# 9. ICD-10 CODE FREQUENCY HISTOGRAM (R-006)
# ==============================================================================
# Unambiguous visual showing EXACTLY how many records each code has.
# This is the key plot for understanding dataset balance and model limitations.
# All codes with < 3 records are explicitly flagged.
# ==============================================================================
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
from src.plot_utils import save_figure

# Build per-code frequency series
code_freq_series = (
    df_raw
    .select(pl.col("ICD10").list.explode().alias("code"))
    .filter(pl.col("code").is_not_null())
    .group_by("code")
    .agg(pl.len().alias("occurrences"))
    .sort("occurrences")
)

freq_values = code_freq_series["occurrences"].to_list()
unique_freqs, counts = np.unique(freq_values, return_counts=True)

# Descriptive statistics
freq_arr = np.array(freq_values)
print("\n📊 ICD-10 CODE FREQUENCY STATISTICS")
print("=" * 55)
print(f"   Total unique codes:  {len(freq_values):,}")
print(f"   Min records/code:    {freq_arr.min()}")
print(f"   Max records/code:    {freq_arr.max()}")
print(f"   Mean records/code:   {freq_arr.mean():.2f}")
print(f"   Median records/code: {np.median(freq_arr):.1f}")
print(f"   Std dev:             {freq_arr.std():.2f}")
print()

# Explicit count for each frequency bucket
print(f"   {'Records/code':<15} {'# Codes':<12} {'% of codes':<12} {'Learnable?'}")
print("   " + "-" * 55)
for freq, count in zip(unique_freqs, counts):
    pct = count / len(freq_values) * 100
    learnable = "✅ Yes" if freq >= 3 else "⚠️  Low confidence"
    print(f"   {freq:<15} {count:<12,} {pct:<12.1f} {learnable}")

codes_below_3 = int((freq_arr < 3).sum())
print()
if codes_below_3 > 0:
    print(f"   ⚠️  CODES WITH < 3 RECORDS: {codes_below_3}")
    low_codes = code_freq_series.filter(pl.col("occurrences") < 3)["code"].to_list()
    print(f"   Codes: {', '.join(low_codes[:20])}" + (" ..." if len(low_codes) > 20 else ""))
else:
    print(f"   ✅ All codes have ≥ 3 records — minimum training signal met")
print("=" * 55)

# Plot
fig, ax = plt.subplots(figsize=(10, 5))

bars = ax.bar(
    unique_freqs.astype(str),
    counts,
    color=["#e74c3c" if f < 3 else "#2980b9" for f in unique_freqs],
    edgecolor="white",
    linewidth=0.8,
)

# Annotate each bar with exact count
for bar, count in zip(bars, counts):
    ax.text(
        bar.get_x() + bar.get_width() / 2,
        bar.get_height() + max(counts) * 0.01,
        f"{count:,}",
        ha="center", va="bottom", fontsize=11, fontweight="bold"
    )

ax.set_xlabel("Records per ICD-10 Code", fontsize=12)
ax.set_ylabel("Number of ICD-10 Codes", fontsize=12)
ax.set_title(
    f"ICD-10 Code Frequency Distribution\n"
    f"{len(freq_values):,} unique codes | {total_records:,} total records",
    fontsize=13, fontweight="bold"
)
ax.yaxis.set_major_formatter(ticker.FuncFormatter(lambda x, _: f"{int(x):,}"))

# Legend
from matplotlib.patches import Patch
legend_elements = [
    Patch(facecolor="#2980b9", label="Adequate training signal (≥ 3 records)"),
    Patch(facecolor="#e74c3c", label="Low confidence (< 3 records)"),
]
ax.legend(handles=legend_elements, loc="upper right")

plt.tight_layout()
save_figure(fig, notebook="01-EDA_SOAP", description="icd10_frequency_histogram")
plt.show()
plt.close()

print("=" * 70)
print("📝 Audit trail updated")
print("✅ Phase 1b.1 complete: Composition and resolution summary validated")
print("=" * 70)

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 📊 Phase 1b.1: Interactive Dashboard

Interactive web-based summary for non-technical stakeholders. Launches
in browser with dataset composition, code status breakdown, and
ICD-10 resolution statistics.

</div>

In [ ]:
# ==============================================================================
# PHASE 1b.1: DATASET COMPOSITION & ICD-10 RESOLUTION DASHBOARD
# ==============================================================================
# Interactive web-based summary for non-technical stakeholders.
# Launches in browser with the same quality as the Raw Discovery Auditor.
# ==============================================================================

import panel as pn
import polars as pl

pn.extension(design='material')


def launch_composition_dashboard(df_raw, df_checked, cdc_df):
    """
    Launches an interactive dashboard summarizing dataset composition
    and ICD-10 code resolution for non-technical stakeholders.
    """
    
    # ==========================================================================
    # DATA PREPARATION
    # ==========================================================================
    total_records = len(df_raw)
    
    # Unique codes
    unique_codes_df = (
        df_raw
        .select(pl.col("ICD10").list.explode().alias("code"))
        .filter(pl.col("code").is_not_null())
        .select(pl.col("code").n_unique().alias("unique_count"))
    )
    unique_code_count = unique_codes_df.item()
    
    # Code frequency distribution
    code_frequencies = (
        df_raw
        .select(pl.col("ICD10").list.explode().alias("code"))
        .filter(pl.col("code").is_not_null())
        .group_by("code")
        .agg(pl.len().alias("occurrences"))
        .group_by("occurrences")
        .agg(pl.len().alias("num_codes"))
        .sort("occurrences")
    )
    
    # Billability breakdown
    billable_records = df_checked.filter(pl.col("status") == "billable").height
    noisy_111_records = df_checked.filter(pl.col("status") == "non_billable_parent (Noisy 111)").height
    invalid_records = df_checked.filter(pl.col("status") == "invalid_or_malformed").height
    
    billable_pct = (billable_records / total_records) * 100
    noisy_111_pct = (noisy_111_records / total_records) * 100
    invalid_pct = (invalid_records / total_records) * 100
    
    # Frequency validation
    codes_with_5 = code_frequencies.filter(pl.col("occurrences") == 5)
    codes_with_5_count = codes_with_5.select("num_codes").item() if codes_with_5.height > 0 else 0
    
    codes_with_10 = code_frequencies.filter(pl.col("occurrences") == 10)
    codes_with_10_count = codes_with_10.select("num_codes").item() if codes_with_10.height > 0 else 0
    
    # Sample codes for examples
    billable_samples = (
        df_checked
        .filter(pl.col("status") == "billable")
        .select(["code", "description"])
        .unique()
        .head(5)
        .to_dicts()
    )
    
    noisy_samples = (
        df_checked
        .filter(pl.col("status") == "non_billable_parent (Noisy 111)")
        .select(["code"])
        .unique()
        .head(5)
        .to_dicts()
    )
    
    # Build parent code examples with children
    noisy_with_children = []
    for row in noisy_samples:
        parent_code = row["code"]
        child_example = (
            cdc_df
            .filter(pl.col("code_no_decimal").str.starts_with(parent_code))
            .head(1)
        )
        if child_example.height > 0:
            child = child_example.row(0, named=True)
            noisy_with_children.append({
                "parent": parent_code,
                "child_code": child["code_no_decimal"],
                "child_desc": child["description"][:50] + "..." if len(child["description"]) > 50 else child["description"]
            })
    
    # ==========================================================================
    # CSS STYLES
    # ==========================================================================
    dashboard_css = """
    <style>
        .dashboard-container {
            font-family: 'Segoe UI', Tahoma, Geneva, Verdana, sans-serif;
            max-width: 1200px;
            margin: 0 auto;
            padding: 20px;
        }
        .metric-card {
            background: linear-gradient(135deg, #667eea 0%, #764ba2 100%);
            border-radius: 16px;
            padding: 30px;
            color: white;
            text-align: center;
            box-shadow: 0 10px 40px rgba(102, 126, 234, 0.3);
            margin: 10px;
            min-width: 200px;
        }
        .metric-card.green {
            background: linear-gradient(135deg, #11998e 0%, #38ef7d 100%);
            box-shadow: 0 10px 40px rgba(17, 153, 142, 0.3);
        }
        .metric-card.orange {
            background: linear-gradient(135deg, #f093fb 0%, #f5576c 100%);
            box-shadow: 0 10px 40px rgba(245, 87, 108, 0.3);
        }
        .metric-value {
            font-size: 3rem;
            font-weight: 800;
            margin: 10px 0;
            text-shadow: 2px 2px 4px rgba(0,0,0,0.2);
        }
        .metric-label {
            font-size: 1rem;
            text-transform: uppercase;
            letter-spacing: 2px;
            opacity: 0.9;
        }
        .section-header {
            background: linear-gradient(90deg, #1a1a2e 0%, #16213e 100%);
            color: white;
            padding: 20px 30px;
            border-radius: 12px;
            margin: 30px 0 20px 0;
            font-size: 1.4rem;
            font-weight: 600;
            display: flex;
            align-items: center;
            gap: 15px;
        }
        .explanation-box {
            background: #f8f9fa;
            border-left: 5px solid #667eea;
            padding: 25px;
            border-radius: 0 12px 12px 0;
            margin: 20px 0;
            line-height: 1.8;
        }
        .hierarchy-visual {
            background: white;
            border: 2px solid #e0e0e0;
            border-radius: 12px;
            padding: 30px;
            font-family: 'Courier New', monospace;
            font-size: 1.1rem;
            line-height: 2;
            margin: 20px 0;
        }
        .hierarchy-level {
            display: flex;
            align-items: center;
            padding: 8px 0;
        }
        .hierarchy-code {
            background: #e3f2fd;
            padding: 5px 15px;
            border-radius: 6px;
            font-weight: bold;
            color: #1565c0;
            min-width: 100px;
            text-align: center;
        }
        .hierarchy-code.billable {
            background: #c8e6c9;
            color: #2e7d32;
            border: 2px solid #4caf50;
        }
        .hierarchy-arrow {
            color: #9e9e9e;
            margin: 0 15px;
            font-size: 1.2rem;
        }
        .hierarchy-desc {
            color: #424242;
        }
        .status-table {
            width: 100%;
            border-collapse: separate;
            border-spacing: 0;
            margin: 20px 0;
            border-radius: 12px;
            overflow: hidden;
            box-shadow: 0 4px 20px rgba(0,0,0,0.1);
        }
        .status-table th {
            background: #1a1a2e;
            color: white;
            padding: 18px 25px;
            text-align: left;
            font-weight: 600;
            text-transform: uppercase;
            letter-spacing: 1px;
            font-size: 0.9rem;
        }
        .status-table td {
            padding: 18px 25px;
            border-bottom: 1px solid #e0e0e0;
            font-size: 1rem;
        }
        .status-table tr:last-child td {
            border-bottom: none;
        }
        .status-table tr:nth-child(even) {
            background: #f8f9fa;
        }
        .status-table tr:hover {
            background: #e3f2fd;
        }
        .status-icon {
            font-size: 1.5rem;
            margin-right: 10px;
        }
        .disposition-card {
            background: white;
            border: 2px solid #4caf50;
            border-radius: 12px;
            padding: 25px;
            margin: 15px 0;
        }
        .disposition-card.warning {
            border-color: #ff9800;
        }
        .disposition-card.info {
            border-color: #2196f3;
        }
        .disposition-header {
            font-weight: 700;
            font-size: 1.1rem;
            margin-bottom: 10px;
            display: flex;
            align-items: center;
            gap: 10px;
        }
        .disposition-body {
            color: #616161;
            line-height: 1.6;
        }
        .code-example {
            background: #263238;
            color: #80cbc4;
            padding: 20px;
            border-radius: 10px;
            font-family: 'Courier New', monospace;
            margin: 10px 0;
            font-size: 1rem;
        }
        .code-example .parent {
            color: #ffcc80;
        }
        .code-example .child {
            color: #a5d6a7;
        }
        .code-example .arrow {
            color: #90a4ae;
        }
        .freq-table {
            width: 100%;
            border-collapse: collapse;
            margin: 20px 0;
        }
        .freq-table th, .freq-table td {
            padding: 15px 20px;
            text-align: center;
            border: 1px solid #e0e0e0;
        }
        .freq-table th {
            background: #37474f;
            color: white;
            font-weight: 600;
        }
        .freq-table tr:nth-child(even) {
            background: #f5f5f5;
        }
        .validation-box {
            background: linear-gradient(135deg, #e8f5e9 0%, #c8e6c9 100%);
            border: 2px solid #4caf50;
            border-radius: 12px;
            padding: 25px;
            margin: 20px 0;
        }
        .validation-title {
            font-weight: 700;
            color: #2e7d32;
            font-size: 1.2rem;
            margin-bottom: 15px;
        }
        .validation-item {
            display: flex;
            align-items: center;
            padding: 8px 0;
            font-size: 1rem;
        }
        .validation-check {
            color: #4caf50;
            font-size: 1.3rem;
            margin-right: 12px;
        }
        .decision-banner {
            background: linear-gradient(135deg, #1a237e 0%, #283593 100%);
            color: white;
            padding: 30px;
            border-radius: 16px;
            text-align: center;
            margin: 30px 0;
            box-shadow: 0 10px 40px rgba(26, 35, 126, 0.3);
        }
        .decision-title {
            font-size: 1.8rem;
            font-weight: 800;
            margin-bottom: 15px;
        }
        .decision-subtitle {
            font-size: 1.1rem;
            opacity: 0.9;
        }
    </style>
    """
    
    # ==========================================================================
    # BUILD HTML CONTENT
    # ==========================================================================
    
    # Metric cards row
    metrics_html = f"""
    {dashboard_css}
    <div class="dashboard-container">
        <div style="display: flex; justify-content: center; flex-wrap: wrap; gap: 20px; margin: 30px 0;">
            <div class="metric-card">
                <div class="metric-label">Total Records</div>
                <div class="metric-value">{total_records:,}</div>
                <div class="metric-label">Clinical Notes</div>
            </div>
            <div class="metric-card green">
                <div class="metric-label">Unique ICD-10</div>
                <div class="metric-value">{unique_code_count:,}</div>
                <div class="metric-label">Diagnostic Codes</div>
            </div>
            <div class="metric-card orange">
                <div class="metric-label">Billable Rate</div>
                <div class="metric-value">{billable_pct:.1f}%</div>
                <div class="metric-label">Leaf Codes</div>
            </div>
        </div>
    </div>
    """
    
    # Frequency distribution table
    freq_rows = ""
    for row in code_frequencies.iter_rows(named=True):
        occ = row["occurrences"]
        num = row["num_codes"]
        pct = (num / unique_code_count) * 100
        freq_rows += f"<tr><td>{occ}</td><td>{num:,}</td><td>{pct:.1f}%</td></tr>"
    
    frequency_html = f"""
    <div class="section-header">📈 Code Frequency Distribution</div>
    <div class="explanation-box">
        The MedSynth authors used <strong>uniform sampling</strong> — generating exactly 
        <strong>5 dialogue-note pairs per ICD-10 code</strong> — to prevent the dataset 
        from being dominated by common diseases. This table confirms that pattern.
    </div>
    <table class="freq-table">
        <tr>
            <th>Occurrences per Code</th>
            <th># of Codes</th>
            <th>% of Total Codes</th>
        </tr>
        {freq_rows}
    </table>
    """
    
    # ICD-10 hierarchy explanation
    hierarchy_html = """
    <div class="section-header">📚 Understanding ICD-10 Code Resolution</div>
    <div class="explanation-box">
        ICD-10 codes follow a <strong>hierarchical structure</strong>, like a folder system. 
        Only the most specific "leaf" codes can be submitted for insurance billing. 
        Parent codes exist to organize the hierarchy, not for claims submission.
    </div>
    <div class="hierarchy-visual">
        <div class="hierarchy-level">
            <span class="hierarchy-code">M</span>
            <span class="hierarchy-arrow">→</span>
            <span class="hierarchy-desc"><strong>Chapter:</strong> Musculoskeletal diseases (broad category)</span>
        </div>
        <div class="hierarchy-level" style="padding-left: 30px;">
            <span class="hierarchy-code">M25</span>
            <span class="hierarchy-arrow">→</span>
            <span class="hierarchy-desc"><strong>Category:</strong> Other joint disorders</span>
        </div>
        <div class="hierarchy-level" style="padding-left: 60px;">
            <span class="hierarchy-code">M25.5</span>
            <span class="hierarchy-arrow">→</span>
            <span class="hierarchy-desc"><strong>Subcategory:</strong> Pain in joint</span>
        </div>
        <div class="hierarchy-level" style="padding-left: 90px;">
            <span class="hierarchy-code">M25.56</span>
            <span class="hierarchy-arrow">→</span>
            <span class="hierarchy-desc"><strong>Specificity:</strong> Pain in knee</span>
        </div>
        <div class="hierarchy-level" style="padding-left: 120px;">
            <span class="hierarchy-code billable">M25.562</span>
            <span class="hierarchy-arrow">→</span>
            <span class="hierarchy-desc"><strong>Laterality:</strong> Pain in LEFT knee <strong>✅ BILLABLE</strong></span>
        </div>
    </div>
    """
    
    # Status breakdown table
    status_html = f"""
    <div class="section-header">🏥 ICD-10 Code Resolution: Billable vs Non-Billable</div>
    <table class="status-table">
        <tr>
            <th>Status</th>
            <th>Records</th>
            <th>Percentage</th>
            <th>Meaning</th>
        </tr>
        <tr>
            <td><span class="status-icon">✅</span> Billable (Leaf Codes)</td>
            <td><strong>{billable_records:,}</strong></td>
            <td><strong>{billable_pct:.1f}%</strong></td>
            <td>Fully specified codes ready for classification</td>
        </tr>
        <tr>
            <td><span class="status-icon">⚠️</span> Non-Billable (Parent)</td>
            <td><strong>{noisy_111_records:,}</strong></td>
            <td><strong>{noisy_111_pct:.1f}%</strong></td>
            <td>Category-level codes from real clinical practice</td>
        </tr>
        <tr>
            <td><span class="status-icon">❓</span> Absent from CDC Reference</td>
            <td><strong>{invalid_records:,}</strong></td>
            <td><strong>{invalid_pct:.1f}%</strong></td>
            <td>Valid placeholder-X codes (e.g., T814XXA)</td>
        </tr>
    </table>
    """
    
    # Why parent codes appear
    why_parent_html = """
    <div class="section-header">❓ Why Do Parent Codes Appear in This Dataset?</div>
    <div class="explanation-box">
        The MedSynth authors selected the <strong>"top 2,000 most frequent ICD-10 codes"</strong> 
        from real insurance claims (IQVIA PharMetrics Plus database — 800 million claims). 
        <br><br>
        In practice, clinicians sometimes submit category-level codes when:
        <ul style="margin-top: 15px;">
            <li>The specific subtype wasn't clinically distinguished</li>
            <li>Documentation didn't support a more specific code</li>
            <li>The condition genuinely spans multiple subtypes</li>
        </ul>
        <br>
        <strong>These parent codes made the top 2,000 because they're FREQUENTLY USED in real billing</strong> 
        — this is a feature of real-world coding practice, not a dataset error.
    </div>
    """
    
    # Code examples
    billable_examples_html = ""
    for sample in billable_samples[:3]:
        code = sample["code"]
        desc = sample["description"][:60] + "..." if len(sample["description"]) > 60 else sample["description"]
        billable_examples_html += f'<div style="padding: 8px 0;"><span style="color: #a5d6a7; font-weight: bold;">{code}</span> <span class="arrow">→</span> {desc}</div>'
    
    noisy_examples_html = ""
    for item in noisy_with_children[:3]:
        noisy_examples_html += f'''
        <div style="padding: 10px 0; border-bottom: 1px solid #455a64;">
            <div><span class="parent">{item["parent"]}</span> <span class="arrow">→</span> Parent code (not billable)</div>
            <div style="padding-left: 30px; margin-top: 5px;">
                <span class="arrow">└──</span> <span class="child">{item["child_code"]}</span> <span class="arrow">→</span> {item["child_desc"]}
            </div>
        </div>
        '''
    
    examples_html = f"""
    <div class="section-header">🔬 Concrete Examples from This Dataset</div>
    <div style="display: flex; gap: 30px; flex-wrap: wrap;">
        <div style="flex: 1; min-width: 300px;">
            <h3 style="color: #2e7d32;">✅ Billable (Leaf) Codes</h3>
            <p style="color: #616161;">Ready for ICD-10 classification tasks</p>
            <div class="code-example">
                {billable_examples_html}
            </div>
        </div>
        <div style="flex: 1; min-width: 300px;">
            <h3 style="color: #f57c00;">⚠️ Non-Billable (Parent) Codes</h3>
            <p style="color: #616161;">With example billable children</p>
            <div class="code-example">
                {noisy_examples_html}
            </div>
        </div>
    </div>
    """
    
    # Decision banner
    decision_html = f"""
    <div class="decision-banner">
        <div class="decision-title">📋 DISPOSITION DECISION</div>
        <div class="decision-subtitle">
            ALL {total_records:,} RECORDS ARE RETAINED
        </div>
    </div>
    <div style="display: flex; gap: 20px; flex-wrap: wrap;">
        <div class="disposition-card" style="flex: 1; min-width: 280px;">
            <div class="disposition-header">✅ Billable Codes ({billable_records:,})</div>
            <div class="disposition-body">
                Primary targets for ICD-10 classification. These carry full diagnostic specificity.
            </div>
        </div>
        <div class="disposition-card warning" style="flex: 1; min-width: 280px;">
            <div class="disposition-header">⚠️ Parent Codes ({noisy_111_records:,})</div>
            <div class="disposition-body">
                Narratives still contain valid clinical signal. Models can learn category-level patterns.
            </div>
        </div>
        <div class="disposition-card info" style="flex: 1; min-width: 280px;">
            <div class="disposition-header">❓ Placeholder-X Codes ({invalid_records:,})</div>
            <div class="disposition-body">
                Valid ICD-10-CM codes. Absence from tabular file is a known reference gap, not an error.
            </div>
        </div>
    </div>
    """
    
    # Validation against paper
    validation_html = f"""
    <div class="section-header">📄 Validation Against MedSynth Paper</div>
    <div class="validation-box">
        <div class="validation-title">Paper Methodology Confirmed ✓</div>
        <div class="validation-item">
            <span class="validation-check">✓</span>
            Paper states: "5 dialogue-note pairs for each ICD-10 code"
        </div>
        <div class="validation-item">
            <span class="validation-check">✓</span>
            Paper reports: 2,001 unique codes → We found: <strong>{unique_code_count:,}</strong> (+{unique_code_count - 2001})
        </div>
        <div class="validation-item">
            <span class="validation-check">✓</span>
            Paper reports: 10,035 records → We found: <strong>{total_records:,}</strong> (+{total_records - 10035})
        </div>
        <div class="validation-item">
            <span class="validation-check">✓</span>
            Codes with exactly 5 records: <strong>{codes_with_5_count:,}</strong>
        </div>
        <div class="validation-item">
            <span class="validation-check">✓</span>
            Codes with exactly 10 records: <strong>{codes_with_10_count:,}</strong>
        </div>
        <div style="margin-top: 15px; padding-top: 15px; border-top: 1px solid #a5d6a7;">
            <em>The additional records reflect post-publication additions to the HuggingFace release.</em>
        </div>
    </div>
    """
    
    # ==========================================================================
    # ASSEMBLE DASHBOARD
    # ==========================================================================
    
    full_html = f"""
    {dashboard_css}
    <div class="dashboard-container">
        {metrics_html}
        {frequency_html}
        {hierarchy_html}
        {status_html}
        {why_parent_html}
        {examples_html}
        {decision_html}
        {validation_html}
    </div>
    """
    
    # Stop button
    stop_button = pn.widgets.Button(name='🛑 Close Dashboard', button_type='danger')
    
    header = pn.pane.Markdown(
        "# 📊 MedSynth Dataset Composition & ICD-10 Resolution Dashboard\n"
        "**Interactive summary for stakeholders** — Understanding what's in the dataset and why.\n",
        sizing_mode='stretch_width'
    )
    
    content = pn.pane.HTML(full_html, sizing_mode='stretch_width')
    
    app = pn.Column(
        header,
        stop_button,
        content,
        sizing_mode='stretch_width'
    )
    
    # Launch server
    server_thread = pn.serve(
        app,
        show=True,
        threaded=True,
        port=0,
        title="MedSynth Composition Dashboard"
    )
    
    def stop_server(event):
        try:
            server_thread.stop()
            print("✅ Dashboard server stopped successfully.")
        except Exception as e:
            print(f"⚠️  Error stopping server: {e}")
    
    stop_button.on_click(stop_server)
    
    print("✅ Composition Dashboard launched — check your browser.")
    print("   Use the Stop button to shut down the server.")
    
    return server_thread


# ==============================================================================
# LAUNCH DASHBOARD
# ==============================================================================
if 'df_raw' in locals() and 'df_checked' in locals() and 'cdc_df' in locals():
    print("🚀 Launching Dataset Composition Dashboard (Phase 1b.1)...")
    
    try:
        dashboard_server = launch_composition_dashboard(df_raw, df_checked, cdc_df)
        
        # Also log to audit trail
        config.log_event(
            phase="Phase 1b.1: Composition Dashboard",
            action="dashboard_launched",
            details={
                "total_records": len(df_raw),
                "unique_codes": df_raw.select(pl.col("ICD10").list.explode()).filter(pl.col("ICD10").is_not_null()).n_unique(),
                "billable_records": df_checked.filter(pl.col("status") == "billable").height,
                "noisy_111_records": df_checked.filter(pl.col("status") == "non_billable_parent (Noisy 111)").height,
            }
        )
        print("📝 Audit trail updated")
        
    except Exception as e:
        print(f"❌ Failed to launch dashboard: {e}")
        raise
else:
    missing = []
    if 'df_raw' not in locals(): missing.append('df_raw')
    if 'df_checked' not in locals(): missing.append('df_checked')
    if 'cdc_df' not in locals(): missing.append('cdc_df')
    print(f"❌ Missing required DataFrames: {missing}")
    print("   Please run Phase 1a and Phase 1b first.")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 📊 Phase 1c: Token Volume & Truncation Audit

Having confirmed structural integrity and ICD-10 validity in Phases 1a and 1b, we
now quantify the spatial constraints imposed by transformer-based models. This is a
prerequisite step before any architectural decisions are made about text representation.

### ⚖️ The Tokenomics Heuristic

Transformer models process tokens, not raw words. We apply a standard heuristic for
medical narratives to estimate token counts without running a full tokeniser pass:

$$Estimated\ Tokens = Words \times 1.3$$

This estimate is applied to both text fields — **Notes** (structured SOAP narratives)
and **Dialogues** (raw doctor-patient transcripts) — and plotted against the
**ClinicalBERT 512-token context limit**.

**Important caveat:** This heuristic was calibrated against general medical text.
ClinicalBERT uses WordPiece tokenisation, which tends to produce more tokens per
word than the BPE tokenisers used in LLaMA and Mistral — the models used to report
token counts in the MedSynth paper (932 avg for Dialogues, 621 avg for Notes). The
`words × 1.3` multiplier is therefore likely a *conservative* estimate of truncation
risk for ClinicalBERT specifically. The true truncation rates are at least as high
as reported here, and may be higher.

### 🕵️ Audit Objectives

* **Identify the Blind Zone:** Visualise the proportion of Notes and Dialogues that
  exceed the 512-token limit, beyond which the model receives no signal.
* **Quantify Source Density:** Compare token distributions across both text sources
  to understand which carries more truncation risk.
* **Validate Against Source Paper:** Cross-reference empirical token distributions
  against the MedSynth paper's reported averages to confirm the heuristic is
  producing plausible estimates.

### 📊 Observed Results

| Field | Paper Avg (tokens) | Empirical Risk (> 512) |
|---|---|---|
| Note | 621 | 64.7% |
| Dialogue | 932 | 100.0% |

Both fields exceed the 512-token limit on average — consistent with the paper's own
reported statistics. The Note distribution peaks just above the truncation cliff at
approximately 520 tokens, meaning the majority of notes lose their tail content under
a standard 512-token window. The Dialogue distribution sits entirely to the right of
the cliff, peaking around 950 tokens, confirming that dialogue truncation is total
and unavoidable without architectural intervention.

### 🛡️ Implications for Text Architecture (APSO-Flip)

Standard SOAP notes place the **Assessment & Plan** at the end of the document. For
a model with a fixed 512-token context window, content beyond that limit is invisible.
This means the primary diagnostic signal — the coded diagnosis embedded in the
Assessment section — is routinely truncated away before the model ever processes it.

The **APSO-Flip** transformation addresses this by reordering the note structure to
place the **Assessment** first, ensuring the highest-signal content survives truncation
regardless of document length. Given that the MedSynth Note Writer Agent enforced
strict SOAP ordering during generation, and that our Phase 1e SOAP extraction achieved
100% success across all four sections, the Assessment content is reliably isolatable
for this reordering.

> **🛡️ Zero-Trust Verdict:** With 64.7% of Notes and 100% of Dialogues exceeding
> the context window, truncation is not an edge case — it is the structural default
> for this dataset, confirmed by both our empirical measurement and the MedSynth
> paper's own reported token averages. APSO-Flip transformation is required before
> any text field is passed to a token-limited model.

<br/>


"For ICD-10 classification tasks, the Note field is the primary input — it contains the structured SOAP summary with the diagnostic Assessment.
The Dialogue field, while rich in clinical context, is typically used for Note generation tasks (Dial-2-Note) rather than direct classification.
The 64.7% Note truncation rate is therefore the critical metric for this pipeline's objective."

</div>

In [ ]:
# ==============================================================================
# PHASE 1c: TOKEN VOLUME & TRUNCATION AUDIT
# ==============================================================================

import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime

TOKEN_HEURISTIC = 1.3
BERT_LIMIT = 512

# ==============================================================================
# 1. CALCULATE WORD COUNTS (Whitespace-Safe Regex)
# ==============================================================================
print("📊 Calculating token pressure metrics...")

df_lengths = df_raw.select([
    pl.col("Note")
      .str.count_matches(r"\S+")
      .fill_null(0)
      .alias("note_words"),

    pl.col("Dialogue")
      .str.count_matches(r"\S+")
      .fill_null(0)
      .alias("dialogue_words")
])

# ==============================================================================
# 2. CONVERT TO PANDAS FOR VISUALIZATION
# ==============================================================================
pdf = df_lengths.to_pandas()

# ==============================================================================
# 3. APPLY TOKEN HEURISTIC
# ==============================================================================
pdf["note_tokens_est"]     = pdf["note_words"]     * TOKEN_HEURISTIC
pdf["dialogue_tokens_est"] = pdf["dialogue_words"] * TOKEN_HEURISTIC

# ==============================================================================
# 4. COMPUTE TRUNCATION RISK METRICS
# ==============================================================================
note_at_risk     = (pdf["note_tokens_est"]     > BERT_LIMIT).sum()
dialogue_at_risk = (pdf["dialogue_tokens_est"] > BERT_LIMIT).sum()

note_at_risk_pct     = (note_at_risk     / len(pdf) * 100) if len(pdf) > 0 else 0
dialogue_at_risk_pct = (dialogue_at_risk / len(pdf) * 100) if len(pdf) > 0 else 0

print(f"   📈 Note Truncation Risk:     {note_at_risk_pct:.1f}%")
print(f"   📈 Dialogue Truncation Risk: {dialogue_at_risk_pct:.1f}%")

# ==============================================================================
# 5. VISUALIZATION SETUP
# ==============================================================================
plt.figure(figsize=(12, 6))
sns.set_style("whitegrid")

# Sample if >10k rows for performance
display_size = min(len(pdf), 5000)
if len(pdf) > display_size:
    print(f"   ⚠️  Sampling {display_size} rows for visualization (total: {len(pdf)})")
    pdf_sample = pdf.sample(n=display_size, random_state=42)
else:
    pdf_sample = pdf

sns.kdeplot(
    pdf_sample["note_tokens_est"],
    fill=True,
    label="Note (Est. Tokens)",
    color="#2980b9",
    alpha=0.5
)

sns.kdeplot(
    pdf_sample["dialogue_tokens_est"],
    fill=True,
    label="Dialogue (Est. Tokens)",
    color="#27ae60",
    alpha=0.5
)

# ClinicalBERT truncation wall
plt.axvline(
    x=BERT_LIMIT,
    color="#c0392b",
    linestyle="--",
    linewidth=3,
    label="ClinicalBERT Limit (512)"
)

# Chart formatting
plt.title(
    "Forensic Analysis: Token Pressure vs. Context Window",
    fontsize=14,
    fontweight="bold"
)
plt.xlabel("Estimated Tokens (Words × 1.3)", fontsize=12)
plt.ylabel("Density", fontsize=12)
plt.xlim(0, 1500)
plt.legend()

# ==============================================================================
# 5a. SAVE FIGURE BEFORE DISPLAY
# ==============================================================================
from src.plot_utils import save_figure
_fig = plt.gcf()
save_figure(_fig, notebook="01-EDA_SOAP", description="token_pressure_kde")

plt.show()
plt.close()  # Prevent memory leaks from matplotlib figures

# ==============================================================================
# 6. VISUAL AUDIT SUMMARY
# ==============================================================================
print("\n📊 VISUAL AUDIT COMPLETE")
print(f"   ⚠️  {note_at_risk_pct:.1f}% of Notes will be truncated without APSO-Flip optimisation.")
print(f"   ⚠️  {dialogue_at_risk_pct:.1f}% of Dialogues exceed the context window.")

# ==============================================================================
# 7. LOG THE TOKEN AUDIT FOR TRACEABILITY
# ==============================================================================
config.log_event(
    phase="Phase 1c: Token Audit",
    action="token_pressure_analysis_complete",
    details={
        "total_records": len(pdf),
        "note_truncation_risk_pct": round(note_at_risk_pct, 2),
        "dialogue_truncation_risk_pct": round(dialogue_at_risk_pct, 2),
        "bert_context_limit": BERT_LIMIT,
        "token_heuristic_multiplier": TOKEN_HEURISTIC,
        "audit_note": (
            f"{note_at_risk_pct:.1f}% of notes and "
            f"{dialogue_at_risk_pct:.1f}% of dialogues exceed "
            f"ClinicalBERT {BERT_LIMIT}-token limit"
        )
    }
)

print(f"\n📝 Audit trail updated")
print(f"✅ Phase 1c complete: Token pressure quantified and logged")

# ==============================================================================
# 8. ZERO-TRUST VERDICT
# ==============================================================================
print(f"\n🛡️  ZERO-TRUST VERDICT:")

if note_at_risk_pct > 50:
    print(f"   ⚠️  {note_at_risk_pct:.1f}% of Notes exceed context window.")
    print(f"   ⚠️  {dialogue_at_risk_pct:.1f}% of Dialogues exceed context window.")
    print(f"   ✅ APSO-Flip transformation is REQUIRED to protect diagnostic signal.")
else:
    print(f"   ✅ {note_at_risk_pct:.1f}% of Notes within context window.")
    print(f"   ⚠️  {dialogue_at_risk_pct:.1f}% of Dialogues exceed context window.")
    print(f"   ℹ️  Verify dialogue handling strategy before downstream modelling.")

# ==============================================================================
# 9. BUSINESS-FACING SUMMARY
# ==============================================================================
print(f"\n💼 BUSINESS IMPACT SUMMARY")
print(f"   " + "─" * 50)
print(f"   📊 Total Records Analyzed:      {len(pdf):,}")
print(f"   ⚠️  Notes at Risk:              {note_at_risk_pct:.1f}%")
print(f"   ⚠️  Dialogues at Risk:          {dialogue_at_risk_pct:.1f}%")
print(f"   🎯 Recommended Action:          APSO-Flip Transformation")
print(f"   " + "─" * 50)
print(f"   💡 Without optimisation, ~{note_at_risk_pct:.0f}% of Notes and")
print(f"      ~{dialogue_at_risk_pct:.0f}% of Dialogues will lose diagnostic")
print(f"      signal beyond the {BERT_LIMIT}-token truncation cliff.")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🕵️ Phase 1d: Raw MedSynth Discovery Auditor

Having completed automated structural validation (Phase 1a), CDC billability
validation (Phase 1b), and token pressure analysis (Phase 1c), we now perform a
**manual discovery audit** of the raw dataset. This browser-native interface allows
direct observation of the raw text fields before any transformation logic is applied.

This is a human-in-the-loop step. Its findings inform downstream design decisions
but are not captured in the automated audit trail.

### Discovery Objectives

1. **Direct Observation:** A high-fidelity browser view of the three core fields —
   **Note**, **Dialogue**, and **ICD-10 Label with description** — bypassing
   notebook rendering constraints for full-length text inspection.

2. **Narrative Correlation:** Visual audit of the relationship between the clinical
   Note and its corresponding Dialogue. In MedSynth, Notes are the primary generated
   artifact — produced first by the Note Writer and Note Polisher Agents — and
   Dialogues are derived from them by the Dialogue Generator and Dialogue Polisher
   Agents. This auditor confirms that the derivation is coherent and that diagnostic
   signal present in the Note is faithfully reflected in the Dialogue.

3. **Noisy 111 Spot-Check:** Manual inspection of records carrying non-billable
   parent codes (identified in Phase 1b). These codes reflect real clinical coding
   practice in the IQVIA source data. The objective here is to observe the narrative
   depth and clinical specificity of these records — understanding whether the
   associated notes contain sufficient detail for downstream classification tasks
   despite the broad code assignment.

4. **SOAP Structure Baseline:** Direct observation of the raw `Note` format
   establishes the baseline for SOAP-section extraction patterns required in
   subsequent phases. Combined with the token pressure findings from Phase 1c —
   where 64.7% of Notes exceed the 512-token limit — this informs the APSO-Flip
   transformation strategy: reordering note sections to place the Assessment first,
   ensuring the primary diagnostic signal survives truncation.

> **Result:** A ground-truth view of the raw dataset providing the observational
> foundation for extraction pattern design and structured mapping in subsequent
> phases. Any patterns or anomalies observed here should be documented as inline
> notes before proceeding to Phase 1e.

</div>

In [ ]:
# ==============================================================================
# PHASE 1d: RAW DISCOVERY AUDITOR (INK-BLACK HIGH CONTRAST + STATUS BADGE)
# ==============================================================================

# ==============================================================================
# DEPENDENCY CHECK
# ==============================================================================
try:
    import panel as pn
    import warnings
    import time
except ImportError as e:
    missing_pkg = str(e).split("'")[1] if "'" in str(e) else "panel"
    raise ImportError(
        f"❌ Missing dependency: {missing_pkg}\n"
        f"   Install with: pip install {missing_pkg}\n"
        f"   Or: pip install panel matplotlib seaborn (for all Phase 1 deps)"
    ) from e

# Ensure Panel extensions are loaded
pn.extension(design='material')


def launch_browser_auditor(df_pl, df_checked=None, start_index=51):
    """
    Launches a dedicated browser tab to audit raw MedSynth data.
    Enforces Ink-Black rendering for high-resolution displays.
    Displays ICD-10 code, description, and billability status.

    Parameters
    ----------
    df_pl : polars.DataFrame
        The raw DataFrame with 'Note', 'Dialogue', 'ICD10', and 'ICD10_desc' columns.
    df_checked : polars.DataFrame, optional
        The validation DataFrame from Phase 1b with 'ID', 'code', and 'status' columns.
        If provided, enables billability status badges.
    start_index : int, optional
        Initial index to display.

    Returns
    -------
    threading.Thread
        The StoppableThread running the server (can be used to stop).
    """
    if df_pl is None or len(df_pl) == 0:
        raise ValueError("DataFrame is empty or None – cannot launch auditor.")

    # Validate required columns exist
    required_cols = ['Note', 'Dialogue', 'ICD10']
    missing_cols = [col for col in required_cols if col not in df_pl.columns]

    if missing_cols:
        raise ValueError(
            f"❌ Missing required columns for auditor: {missing_cols}\n"
            f"   Available columns: {df_pl.columns}\n"
            f"   Please run Phase 1a first to ensure proper data structure."
        )

    # Memory efficiency warning
    if len(df_pl) > 100000:
        print(f"⚠️  Large dataset detected ({len(df_pl):,} rows). Converting to pandas...")
        print(f"   Memory usage will temporarily increase.")

    # Convert to pandas once for efficient iloc access
    df_pd = df_pl.to_pandas()
    max_idx = len(df_pd) - 1

    # Build status lookup if df_checked is provided
    status_lookup = {}
    if df_checked is not None:
        try:
            # Create a lookup: ID -> status
            status_df = df_checked.select(["ID", "status"]).unique()
            for row in status_df.iter_rows(named=True):
                status_lookup[row["ID"]] = row["status"]
            print(f"   ✅ Status lookup built: {len(status_lookup):,} records")
        except Exception as e:
            print(f"   ⚠️  Could not build status lookup: {e}")
            status_lookup = {}

    # Slider for index navigation
    index_slider = pn.widgets.IntSlider(
        name='Examine Record (Index)',
        start=0, end=max_idx, value=min(start_index, max_idx),
        bar_color='#2196f3', sizing_mode='stretch_width'
    )

    # Stop button
    stop_button = pn.widgets.Button(name='🛑 Stop Auditor', button_type='danger', disabled=True)

    # CSS to enforce Ink-Black, Bold Monospace, and Auto-Wrap
    raw_text_style = """
    <style>
        .ink-black-audit {
            white-space: pre-wrap !important;
            word-wrap: break-word !important;
            font-family: 'Courier New', Courier, monospace !important;
            font-size: 1.0rem !important;
            font-weight: 700 !important;
            color: #000000 !important;
            background-color: #ffffff !important;
            padding: 20px;
            border: 2px solid #000;
            border-radius: 4px;
            line-height: 1.5;
            -webkit-font-smoothing: none;
        }
        .header-container {
            display: flex;
            flex-direction: row;
            gap: 15px;
            margin: 20px 0;
            width: 100%;
            align-items: stretch;
            flex-wrap: wrap;
        }
        .code-box {
            flex: 1;
            min-width: 150px;
            background-color: #e3f2fd;
            border: 3px solid #1976d2;
            border-radius: 10px;
            padding: 20px;
            text-align: center;
            box-shadow: 0 4px 6px rgba(0,0,0,0.1);
        }
        .desc-box {
            flex: 2;
            min-width: 250px;
            background-color: #f3e5f5;
            border: 3px solid #7b1fa2;
            border-radius: 10px;
            padding: 20px;
            text-align: center;
            box-shadow: 0 4px 6px rgba(0,0,0,0.1);
        }
        .status-box {
            flex: 1;
            min-width: 180px;
            border-radius: 10px;
            padding: 20px;
            text-align: center;
            box-shadow: 0 4px 6px rgba(0,0,0,0.1);
        }
        .status-billable {
            background: linear-gradient(135deg, #c8e6c9 0%, #a5d6a7 100%);
            border: 3px solid #4caf50;
        }
        .status-parent {
            background: linear-gradient(135deg, #ffe0b2 0%, #ffcc80 100%);
            border: 3px solid #ff9800;
        }
        .status-invalid {
            background: linear-gradient(135deg, #e3f2fd 0%, #bbdefb 100%);
            border: 3px solid #2196f3;
        }
        .status-unknown {
            background: linear-gradient(135deg, #f5f5f5 0%, #e0e0e0 100%);
            border: 3px solid #9e9e9e;
        }
        .code-label, .desc-label, .status-label {
            font-size: 0.85rem;
            font-weight: bold;
            text-transform: uppercase;
            letter-spacing: 1px;
            display: block;
            margin-bottom: 10px;
        }
        .code-label {
            color: #0d47a1;
        }
        .desc-label {
            color: #4a148c;
        }
        .status-label {
            color: #424242;
        }
        .code-value {
            font-size: 2.5rem;
            color: #0d47a1;
            font-weight: 900;
            font-family: 'Courier New', monospace;
            line-height: 1.2;
        }
        .desc-value {
            font-size: 1.3rem;
            color: #4a148c;
            font-weight: 600;
            line-height: 1.3;
        }
        .status-value {
            font-size: 1.1rem;
            font-weight: 700;
            line-height: 1.3;
        }
        .status-value.billable {
            color: #2e7d32;
        }
        .status-value.parent {
            color: #e65100;
        }
        .status-value.invalid {
            color: #1565c0;
        }
        .status-value.unknown {
            color: #616161;
        }
        .status-icon {
            font-size: 2rem;
            display: block;
            margin-bottom: 8px;
        }
        .status-explanation {
            font-size: 0.8rem;
            color: #757575;
            margin-top: 8px;
            font-style: italic;
        }
    </style>
    """

    def format_raw(text):
        """Wrap raw text in the ink-black styling."""
        if text is None:
            text = "N/A"
        return f'{raw_text_style}<div class="ink-black-audit">{text}</div>'

    def clean_label(label) -> str:
        """
        Safely convert an ICD-10 label to a clean display string.
        Handles: Python list, tuple, numpy array, and stringified list
        representations that result from Polars List→pandas conversion.
        """
        if isinstance(label, (list, tuple)):
            return ', '.join(str(c) for c in label)
        if isinstance(label, str) and label.startswith('['):
            return label.strip('[]').replace("'", '').replace('"', '').strip()
        try:
            import numpy as np
            if isinstance(label, np.ndarray):
                return ', '.join(str(c) for c in label.tolist())
        except ImportError:
            pass
        return str(label) if label is not None else 'N/A'

    def get_status_html(status):
        """Generate HTML for the status badge based on billability classification."""
        if status == "billable":
            return """
                <div class="status-box status-billable">
                    <span class="status-label">Billability Status</span>
                    <span class="status-icon">✅</span>
                    <span class="status-value billable">BILLABLE</span>
                    <span class="status-explanation">Leaf code — valid for classification</span>
                </div>
            """
        elif "non_billable_parent" in status.lower() or "noisy" in status.lower():
            return """
                <div class="status-box status-parent">
                    <span class="status-label">Billability Status</span>
                    <span class="status-icon">⚠️</span>
                    <span class="status-value parent">NOISY 111</span>
                    <span class="status-explanation">Parent code — not billable but retained</span>
                </div>
            """
        elif "invalid" in status.lower() or "malformed" in status.lower():
            return """
                <div class="status-box status-invalid">
                    <span class="status-label">Billability Status</span>
                    <span class="status-icon">❓</span>
                    <span class="status-value invalid">PLACEHOLDER-X</span>
                    <span class="status-explanation">Valid ICD-10-CM code — CDC reference gap</span>
                </div>
            """
        else:
            return """
                <div class="status-box status-unknown">
                    <span class="status-label">Billability Status</span>
                    <span class="status-icon">❔</span>
                    <span class="status-value unknown">UNKNOWN</span>
                    <span class="status-explanation">Run Phase 1b to classify</span>
                </div>
            """

    @pn.depends(index_slider)
    def get_raw_view(index):
        row = df_pd.iloc[index]
        note     = row.get('Note',      'N/A')
        dialogue = row.get('Dialogue',  'N/A')
        label    = row.get('ICD10',     'N/A')
        desc     = row.get('ICD10_desc','No description available')
        record_id = row.get('ID', index)

        label_str = clean_label(label)
        
        # Get status from lookup
        status = status_lookup.get(record_id, "unknown")
        status_html = get_status_html(status)

        # Header HTML with code, description, and status
        header_html = f"""
        {raw_text_style}
        <div class="header-container">
            <div class="code-box">
                <span class="code-label">ICD-10 Code</span>
                <span class="code-value">{label_str}</span>
            </div>
            <div class="desc-box">
                <span class="desc-label">Description</span>
                <span class="desc-value">{desc}</span>
            </div>
            {status_html}
        </div>
        """

        return pn.Column(
            pn.pane.HTML(header_html, sizing_mode='stretch_width'),
            pn.Row(
                pn.Column(
                    "### 📝 Raw Note Content",
                    pn.pane.HTML(format_raw(note), sizing_mode='stretch_width'),
                    height=750, scroll=True, sizing_mode='stretch_width'
                ),
                pn.Column(
                    "### 💬 Raw Dialogue Content",
                    pn.pane.HTML(format_raw(dialogue), sizing_mode='stretch_width'),
                    height=750, scroll=True, sizing_mode='stretch_width'
                ),
                sizing_mode='stretch_width'
            ),
            sizing_mode='stretch_width'
        )

    # Build status summary for header
    if status_lookup:
        billable_count = sum(1 for s in status_lookup.values() if s == "billable")
        parent_count = sum(1 for s in status_lookup.values() if "non_billable_parent" in s.lower())
        invalid_count = sum(1 for s in status_lookup.values() if "invalid" in s.lower())
        status_summary = (
            f"**Status Summary:** "
            f"✅ {billable_count:,} Billable | "
            f"⚠️ {parent_count:,} Noisy 111 | "
            f"❓ {invalid_count:,} Placeholder-X"
        )
    else:
        status_summary = "**Status:** Run Phase 1b to enable billability badges"

    # Build the application layout
    header = pn.pane.Markdown(
        "# 🕵️ MedSynth Raw Discovery Auditor\n"
        "**Browser Mode:** Ink-Black rendering enabled for maximum contrast.\n\n"
        f"{status_summary}\n\n"
        "### 📖 How to Use:\n"
        "- **Slider**: Navigate through records by index\n"
        "- **Left Panel**: Clinical Note (SOAP format)\n"
        "- **Right Panel**: Doctor-Patient Dialogue\n"
        "- **Top Boxes**: ICD-10 Code + Description + Billability Status\n"
        "- **Stop Button**: Shut down the auditor server\n\n"
        "💡 **Tip:** Look for Noisy 111 (⚠️) records to inspect parent-code narratives.",
        sizing_mode='stretch_width'
    )

    app = pn.Column(
        header,
        pn.Row(index_slider, stop_button),
        get_raw_view,
        sizing_mode='stretch_width'
    )

    # Launch the server
    server_thread = pn.serve(
        app,
        show=True,
        threaded=True,
        port=0,
        title="MedSynth Raw Auditor"
    )

    # Enable stop button after server starts
    stop_button.disabled = False

    def stop_server(event):
        try:
            server_thread.stop()
            app[:] = [pn.pane.Markdown("### 🛑 Auditor stopped. You may close this tab.")]
            stop_button.disabled = True
            print("✅ Auditor server stopped successfully.")
        except Exception as e:
            warnings.warn(f"Error stopping server: {e}")
            print(f"⚠️  Error stopping server: {e}")

    stop_button.on_click(stop_server)

    # Wait for server to be ready
    timeout = 5
    start_time = time.time()
    while not hasattr(server_thread, 'server') or server_thread.server is None:
        if time.time() - start_time > timeout:
            print("⚠️  Server did not become ready within timeout.")
            break
        time.sleep(0.1)

    if hasattr(server_thread, 'server') and server_thread.server is not None:
        actual_server = server_thread.server
        app_url = getattr(actual_server, 'app_url', None)
        if app_url:
            print(f"✅ Auditor launched at {app_url}")
            print(f"   Use the Stop button to shut down the server.")
        else:
            print(f"✅ Auditor launched – check your browser for the new tab.")
    else:
        print(f"✅ Auditor launched – check your browser for the new tab.")

    return server_thread


# ==============================================================================
# LAUNCH AUDITOR
# ==============================================================================
if 'df_raw' in locals() and df_raw is not None:
    print("🚀 Launching MedSynth Raw Discovery Auditor (Phase 1d)...")

    # Ensure ICD10_desc exists — add placeholder if missing
    if 'ICD10_desc' not in df_raw.columns:
        print("⚠️  ICD10_desc column not found - adding placeholder")
        df_raw = df_raw.with_columns([
            pl.lit("Description not available").alias("ICD10_desc")
        ])

    # Check if df_checked is available for status badges
    df_checked_param = None
    if 'df_checked' in locals() and df_checked is not None:
        print("   ✅ Phase 1b validation data available — enabling status badges")
        df_checked_param = df_checked
    else:
        print("   ⚠️  df_checked not found — status badges will show 'Unknown'")
        print("      Run Phase 1b first for full billability classification")

    try:
        auditor_server = launch_browser_auditor(
            df_raw, 
            df_checked=df_checked_param, 
            start_index=51
        )
        print("\n💡 TIP: Use the slider to examine different records.")
        print("   Look for: SOAP structure, code-description alignment, and narrative quality.")
        print("   Status badges show: ✅ Billable | ⚠️ Noisy 111 | ❓ Placeholder-X")
    except Exception as e:
        print(f"❌ Failed to launch auditor: {e}")
        raise
else:
    print("❌ df_raw not found. Please run Phase 1a first.")
    print("   The Raw Vault must be loaded before auditing.")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🛡️ Phase 1e: Pydantic Firewall & Silver Layer Registration

Having validated ICD-10 codes against the CDC reference (Phase 1b) and quantified
token pressure (Phase 1c), we now enforce structural schema validation across every
record before promoting the dataset to the **Silver layer**. No record enters the
analytical pipeline without passing this gate.

---

### 🛡️ The Gatekeeper: What is being validated?

The **Pydantic Gatekeeper** (`src/gatekeeper.py`) enforces the following schema
against every record:

* **ID Integrity:** Every record must carry a unique, non-null string anchor for
  downstream audit traceability.

* **Narrative Presence:** The `Note` and `Dialogue` fields must contain text signal.
  Records with null values are imputed with explicit sentinel placeholders
  (`[NO_TRANSCRIPT_AVAILABLE]`, `[EMPTY_NOTE_SIGNAL]`) rather than silently dropped,
  preserving record count while flagging null sections for downstream handlers.
  The 2 empty dialogues identified in Phase 1a are resolved here — retained in the
  Silver layer with their sentinel values intact, with disposition logged to the
  audit trail.

* **SOAP Section Extraction:** The gatekeeper splits each `Note` into its four
  structural components — **Subjective, Objective, Assessment, and Plan** — using
  regex patterns. Successful isolation of the Assessment section is the critical
  prerequisite for the APSO-Flip transformation in Phase 2.

* **Signal Density Check:** The extraction rate of the Assessment section is reported
  explicitly. A rate below 80% would indicate that regex patterns require adjustment
  before proceeding.

### The Surgical Strategy

1. **Deterministic ID Injection:** Where no primary key exists, a row index is cast
   to a string-typed `ID` column, acting as a permanent forensic anchor across all
   subsequent transformations.

2. **Sentinel Imputation:** Null fields are replaced with explicit placeholders rather
   than dropped, ensuring the record count remains stable and null sections are
   identifiable by downstream tokenisers.

3. **Silver Artifact Registration:** All validated records are registered via
   `config.register_dataframe`, creating a persistent recovery point before any
   further transformation is applied.

### On the 100% SOAP Extraction Rate

The SOAP extraction depth report confirms 100% success across all four sections.
This result is expected given the dataset's synthetic origin — the MedSynth Note
Writer Agent was explicitly instructed to produce notes in strict SOAP format, and
the Note Polisher Agent was designed to enforce correct section placement before
the notes were finalised. The structural consistency of the notes is a feature of
controlled synthetic generation, not a property that should be assumed to generalise
to real clinical notes from EHR systems.

The optional sub-sections within Subjective (Chief Complaint, History of Present
Illness, Review of Systems) vary stochastically across records — the Note Writer
Agent prompt instructed the model to "roll a dice" to determine whether to include
them. This explains the regex variation observed in the Phase 1f Surgical Signal
Auditor.

> **Result:** A schema-validated Silver dataset where all 10,240 records have passed
> structural inspection, all four SOAP sections have been successfully isolated, and
> the 2 empty dialogue records have been formally dispositioned via sentinel
> imputation — forming a clean, traceable baseline for all downstream transformation
> phases.

</div>

In [ ]:
# ==============================================================================
# PHASE 1e: PYDANTIC FIREWALL & SILVER LAYER REGISTRATION
# ==============================================================================

import polars as pl
from datetime import datetime

# ==============================================================================
# 1. IMPORT GATEKEEPER
# ==============================================================================
try:
    from src.gatekeeper import validate_dataframe, ClinicalRecord
    print("✅ Successfully imported validate_dataframe and ClinicalRecord from src/gatekeeper.py")
except ImportError as e:
    print(f"❌ CRITICAL: Failed to import from src.gatekeeper.")
    print(f"   Please ensure src/gatekeeper.py exists and Phase 0 setup is correct.")
    print(f"   Error details: {e}")
    raise

# ==============================================================================
# 2. ID CASTING & SENTINEL IMPUTATION
# ==============================================================================
# Note: The 2 empty dialogues identified in Phase 1a are resolved here.
# They are retained via sentinel imputation rather than dropped, preserving
# record count and making null sections explicitly identifiable downstream.
# ==============================================================================
if 'ID' not in df_raw.columns:
    print("⚠️ ID column missing. Generating unique forensic identifiers...")
    df_prepared = df_raw.with_row_index("ID").with_columns([
        pl.col("ID").cast(pl.String),
        pl.col("Dialogue").fill_null("[NO_TRANSCRIPT_AVAILABLE]"),
        pl.col("Note").fill_null("[EMPTY_NOTE_SIGNAL]")
    ])
else:
    print("✅ ID column already exists. Casting to String type...")
    df_prepared = df_raw.with_columns([
        pl.col("ID").cast(pl.String),
        pl.col("Dialogue").fill_null("[NO_TRANSCRIPT_AVAILABLE]"),
        pl.col("Note").fill_null("[EMPTY_NOTE_SIGNAL]")
    ])

# Confirm sentinel imputation resolved the known empty dialogue records
imputed_dialogues = (df_prepared["Dialogue"] == "[NO_TRANSCRIPT_AVAILABLE]").sum()
imputed_notes     = (df_prepared["Note"]     == "[EMPTY_NOTE_SIGNAL]").sum()
if imputed_dialogues > 0 or imputed_notes > 0:
    print(f"   ℹ️  Sentinel imputation applied:")
    print(f"      Dialogues imputed: {imputed_dialogues} (expected: 2 from Phase 1a)")
    print(f"      Notes imputed:     {imputed_notes}")
else:
    print("   ✅ No null fields requiring imputation.")

# ==============================================================================
# 3. EXECUTE VALIDATION WITH SCHEMA DISCLOSURE
# ==============================================================================
print("\n🛡️ Initializing Zero-Trust Validation Firewall...")
print(" ↳ Enforcing ID Unique Anchors")
print(" ↳ Isolating SOAP Clinical Sections")
print(" ↳ Verifying Diagnostic Signal Density")

df_valid_polars, df_errors, df_valid_objects = validate_dataframe(df_prepared)
df_valid = df_valid_polars

# ==============================================================================
# 4. REGISTER THE SILVER DATASET
# ==============================================================================
config.register_dataframe("silver_medsynth", df_valid, phase="Phase 1e")

# ==============================================================================
# 5. FORENSIC REPORT & TELEMETRY
# ==============================================================================
print(f"\n✅ VALIDATION COMPLETE")
print(f"-------------------------------------------")
print(f"✅ Valid Records:        {len(df_valid):,} (Promoted to Silver)")
print(f"⚠️  Quarantined:         {len(df_errors):,} (Isolated for Review)")
print(f"🧬 SOAP Objects Extracted: {len(df_valid_objects):,} (Successfully)")

if len(df_errors) > 0:
    print("\n❌ Error Forensic Sample (first 5):")
    if not df_errors.is_empty():
        for row in df_errors.head(5).iter_rows(return_type="dict"):
            print(f"   ID: {row.get('ID', 'N/A')}, Error: {row.get('error_message', 'N/A')}")
    else:
        print("   No errors in sample.")

# ==============================================================================
# 5b. SOAP EXTRACTION DEPTH REPORT
# ==============================================================================
print(f"\n🧬 SOAP EXTRACTION DEPTH REPORT")
print(f"   " + "─" * 50)

soap_sections = ['subjective', 'objective', 'assessment', 'plan']
for section in soap_sections:
    extracted_count = sum(
        1 for obj in df_valid_objects
        if getattr(obj, section, None) is not None
    )
    extracted_pct = (
        extracted_count / len(df_valid_objects) * 100
    ) if len(df_valid_objects) > 0 else 0
    print(f"   {section.capitalize():12s}: {extracted_count:,}/{len(df_valid_objects):,} ({extracted_pct:.1f}%)")

print(f"   " + "─" * 50)

assessment_count = sum(1 for obj in df_valid_objects if obj.assessment is not None)
assessment_pct = (
    assessment_count / len(df_valid_objects) * 100
) if len(df_valid_objects) > 0 else 0

if assessment_pct < 80:
    print(f"   ⚠️  WARNING: Only {assessment_pct:.1f}% of records have Assessment extracted.")
    print(f"      Regex patterns may need adjustment before Phase 2 (APSO-Flip).")
else:
    print(f"   ✅ Assessment extraction rate: {assessment_pct:.1f}% (Sufficient for APSO-Flip)")

# ==============================================================================
# 5c. BUSINESS-FACING SUMMARY
# ==============================================================================
print(f"\n💼 BUSINESS IMPACT SUMMARY")
print(f"   " + "─" * 50)
print(f"   📊 Total Records Processed:    {len(df_prepared):,}")
print(f"   ✅ Validated for Modelling:    {len(df_valid):,} ({len(df_valid)/len(df_prepared)*100:.1f}%)")
print(f"   ❌ Quarantined for Review:     {len(df_errors):,} ({len(df_errors)/len(df_prepared)*100:.1f}%)")
print(f"   🧬 SOAP Sections Extracted:    {len(df_valid_objects):,}")
print(f"   ℹ️  Sentinel Records (Dialogue):{imputed_dialogues:,} (retained with placeholder)")
print(f"   " + "─" * 50)
print(f"   🎯 Silver layer is ready for APSO-Flip transformation (Phase 2).")

# ==============================================================================
# 6. LOG THE EVENT
# ==============================================================================
config.log_event(
    phase="Phase 1e: Pydantic Firewall",
    action="pydantic_firewall_complete",
    details={
        "valid_count": len(df_valid),
        "error_count": len(df_errors),
        "object_count": len(df_valid_objects),
        "assessment_extraction_pct": round(assessment_pct, 2),
        "imputed_dialogues": int(imputed_dialogues),
        "imputed_notes": int(imputed_notes),
        "empty_dialogue_disposition": "retained_with_sentinel"
    }
)

print(f"\n📝 Audit trail updated")
print(f"✅ PHASE 1e COMPLETE: Silver dataset registered and validated")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🧪 Phase 1f: Surgical Signal Auditor

Following the Pydantic Firewall in Phase 1e, we launch a browser-native auditor
to visually verify that SOAP section extraction was performed correctly across
the Silver dataset. This is a human-in-the-loop confirmation step before any
further transformation is applied.

### Audit Objectives

1. **Visual Verification:** A side-by-side Panel layout compares the raw Clinical
   Note (unparsed, left panel) against the extracted SOAP Signal (structured,
   right panel) produced by the Pydantic Gatekeeper's regex logic. The raw note
   is already in SOAP format — this auditor confirms that the regex extraction
   has correctly identified and isolated each section boundary.

2. **Section Integrity:** Each of the four SOAP sections is rendered in a distinct
   clinical colour — Subjective (slate), Objective (teal), Assessment (medical red),
   Plan (blue) — making extraction gaps or boundary misalignments immediately
   visible. Priority inspection targets the **Assessment** section (medical red),
   which carries the primary diagnostic signal and is the target of the APSO-Flip
   transformation in Phase 2.

3. **Noisy 111 Spot-Check:** Records carrying non-billable parent codes (identified
   in Phase 1b) can be navigated to directly using the slider and their known index
   positions. The objective is to observe the narrative depth and clinical specificity
   of these records — understanding how the MedSynth generation pipeline handled
   broad category-level diagnoses, and whether the SOAP extraction captures the
   relevant clinical content regardless of code granularity.

4. **Traceability Anchor:** The auditor cross-references records via the deterministic
   string `ID` injected in Phase 1e, maintaining a transparent audit chain from raw
   source to structured Silver layer.

> **Result:** A web-based Surgical Auditor confirming extraction quality before
> proceeding to Canonical Label Mapping in Phase 2. Any extraction boundary failures
> or section misalignments observed here should be documented as inline notes before
> continuing.

</div>

In [ ]:
# ==============================================================================
# PHASE 1f: SURGICAL SIGNAL AUDITOR (BROWSER-NATIVE)
# ==============================================================================

import panel as pn

pn.extension(design='material')


def launch_signal_auditor(df_prepared, validated_list):
    """
    Launches a dedicated browser tab to compare Raw Notes vs. Extracted SOAP
    signals produced by the Phase 1e Pydantic Gatekeeper.

    Parameters
    ----------
    df_prepared : polars.DataFrame
        The prepared DataFrame from Phase 1e (with string-cast ID column).
    validated_list : list[ClinicalRecord]
        The list of validated Pydantic objects from Phase 1e.
    """
    # 1. Map validated objects by ID for instant cross-referencing
    val_map = {str(v.id): v for v in validated_list}
    df_pd = df_prepared.to_pandas()

    # 2. Slider — defaults to record 51 as a representative mid-dataset sample
    index_slider = pn.widgets.IntSlider(
        name='Examine Extraction Quality (Index)',
        start=0, end=len(df_pd) - 1, value=51,
        bar_color='#e91e63', sizing_mode='stretch_width'
    )

    @pn.depends(index_slider)
    def update_comparison(index):
        row = df_pd.iloc[index]
        row_id = str(row['ID'])
        validated_obj = val_map.get(row_id)

        if not validated_obj:
            return pn.pane.Alert(
                f"⚠️ No validated signal for ID: {row_id}",
                alert_type="warning"
            )

        # Clinical colour palette — consistent across all SOAP sections
        colors = {
            'subjective': '#607d8b',   # Slate
            'objective':  '#009688',   # Teal
            'assessment': '#d32f2f',   # Medical Red — highest diagnostic signal
            'plan':       '#1976d2'    # Blue
        }

        sections = {
            'subjective': validated_obj.subjective,
            'objective':  validated_obj.objective,
            'assessment': validated_obj.assessment,
            'plan':       validated_obj.plan
        }

        extracted_html = '<div style="font-family: sans-serif; line-height: 1.6;">'
        for title, content in sections.items():
            color = colors.get(title, '#555')
            if content:
                extracted_html += f"""
                <div style="margin-bottom: 15px; padding: 12px; border-radius: 6px;
                            background: #fff; border: 1px solid #eee;
                            border-top: 5px solid {color};">
                    <span style="font-size: 0.7rem; font-weight: 900;
                                 color: {color}; text-transform: uppercase;">{title}</span>
                    <div style="font-size: 0.9rem; color: #333;
                                white-space: pre-wrap;">{content.strip()}</div>
                </div>"""
        extracted_html += "</div>"

        return pn.Row(
            pn.Column(
                "### 🔴 Raw Clinical Note",
                pn.pane.Str(row['Note']),
                height=750, scroll=True, sizing_mode='stretch_width'
            ),
            pn.Column(
                "### 🟢 Extracted SOAP Signal",
                pn.pane.HTML(extracted_html),
                height=750, scroll=True, sizing_mode='stretch_width'
            )
        )

    # Dashboard layout
    layout = pn.Column(
        "# 🧪 Surgical Extraction Audit",
        "**Objective:** Confirm the Phase 1e Pydantic Gatekeeper isolated SOAP "
        "sections correctly before Canonical Label Mapping (Phase 2).",
        index_slider,
        update_comparison,
        sizing_mode='stretch_width'
    )

    return layout.show(title="MedSynth Signal Auditor — Phase 1f", threaded=True)


# ==============================================================================
# LAUNCH THE AUDITOR
# Uses df_prepared and df_valid_objects produced by Phase 1e
# ==============================================================================
if 'df_prepared' not in locals() or 'df_valid_objects' not in locals():
    print("❌ df_prepared or df_valid_objects not found.")
    print("   Please run Phase 1e (Pydantic Firewall) before launching this auditor.")
else:
    print("🚀 Launching Surgical Signal Auditor (Phase 1f)...")
    print("   💡 TIP: Use the slider to navigate records.")
    print("   💡 TIP: Pay particular attention to Assessment sections (Medical Red).")
    print("   💡 TIP: Cross-check Noisy 111 codes from Phase 1b using their known indices.")
    signal_server = launch_signal_auditor(df_prepared, df_valid_objects)

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🦆 Phase 1g: DuckDB Persistence & Silver Vault

With `df_valid` secured by the Phase 1e Pydantic Firewall, we now transition from
volatile in-memory storage to a persistent **Silver Vault** within DuckDB. This
ensures the validated dataset remains queryable via SQL across kernel restarts and
provides the foundation for all downstream SQL-based transformations.

### Persistence Strategy

1. **Table Materialisation:** The validated Polars DataFrame is promoted to a
   permanent `silver_vault` table in DuckDB. Unlike a temporary view, this table
   acts as the Silver layer anchor — a high-performance, persistent source for all
   subsequent clinical transformations.

2. **Schema Verification:** A `DESCRIBE` operation confirms that all column types
   have been correctly mapped from Polars to DuckDB, including the string-cast `id`
   column and the `VARCHAR[]` array type for the `label` column.

3. **Label Cardinality Audit:** A SQL scan using `UNNEST` counts distinct individual
   ICD-10 code strings across all label arrays, establishing the code density
   baseline before any Gold layer transformation is applied.

   Note: `COUNT(DISTINCT label)` on a `VARCHAR[]` column in DuckDB produces
   unreliable results due to array hashing behaviour — `UNNEST` is required to
   count distinct individual code strings correctly. The confirmed count of
   **2,037 distinct ICD-10 codes** is consistent with the MedSynth paper's
   reported 2,001 unique codes, with the difference attributable to post-publication
   additions to the HuggingFace release.

4. **State Transition Logging:** The vault initialisation is captured in the project
   telemetry, recording the table structure and unique code count at the moment of
   persistence.

> **Result:** A persistent, SQL-verifiable Silver Vault — 10,240 validated records
> across 2,037 distinct ICD-10 codes — providing the foundation for decimal
> restoration and Gold layer promotion in Phase 2a.

</div>

In [ ]:
# ==============================================================================
# PHASE 1g: DUCKDB PERSISTENCE (THE SILVER VAULT)
# ==============================================================================

# 'config' object is available globally from Phase 0.
con = config.get_duckdb_conn()

try:
    # Register the validated Polars DataFrame with this DuckDB connection so it
    # can be referenced by name in subsequent SQL statements.
    con.register("df_valid", df_valid)

    # 1. Materialise as a permanent Silver Vault table
    con.execute("CREATE OR REPLACE TABLE silver_vault AS SELECT * FROM df_valid")

    # 2. Verify schema — confirm column types mapped correctly from Polars to DuckDB
    db_audit = con.execute("DESCRIBE silver_vault").pl()
    print("📊 DuckDB Silver Vault Schema (Validated):")
    print(db_audit)

    # 3. Label Cardinality Audit
    #
    # IMPORTANT: COUNT(DISTINCT label) on a VARCHAR[] (array) column in DuckDB
    # does not reliably count distinct array values — it hashes or serialises
    # arrays in a way that produces spurious uniqueness (e.g. 2,037 instead of
    # the true ~73). This is a known DuckDB behaviour with list-type columns.
    #
    # The correct approach is to UNNEST the label arrays into individual code
    # strings first, then count distinct values across those strings.
    # COUNT(DISTINCT id) is used for total_records to remain correct even if
    # multi-code records are ever introduced.
    label_stats = con.execute("""
        SELECT
            COUNT(DISTINCT code) as unique_codes,
            COUNT(DISTINCT id)   as total_records
        FROM (
            SELECT id, UNNEST(label) as code
            FROM silver_vault
        )
    """).pl()

    print("\n📈 Label Cardinality Report:")
    print(label_stats)

    # Sanity check — confirm total_records matches expected row count
    raw_count = con.execute("SELECT COUNT(*) as n FROM silver_vault").pl()["n"].item()
    if label_stats["total_records"].item() != raw_count:
        print(f"\n⚠️  WARNING: Cardinality record count ({label_stats['total_records'].item():,}) "
              f"does not match raw vault count ({raw_count:,}).")
        print(f"   This may indicate multi-code records in the dataset.")
    else:
        print(f"\n   ✅ Record count consistent: {raw_count:,} records")

    # 4. Log state transition
    config.log_event(
        phase="Phase 1g: Persistence",
        action="duckdb_vault_initialized",
        details={
            "table": "silver_vault",
            "unique_codes": label_stats["unique_codes"].item(),
            "total_records": label_stats["total_records"].item(),
            "count_method": "UNNEST — counts distinct individual ICD-10 code strings"
        }
    )

    print("\n✅ PHASE 1g COMPLETE: Silver vault initialized and logged.")

finally:
    con.close()
    print("🦆 DuckDB connection for Phase 1g closed.")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🔬 Phase 2a: Biological Audit & Decimal Restoration

With the Silver Vault persisted in Phase 1g, we now apply the first transformation
to the validated dataset, promoting it to the **Gold layer**. This phase addresses
two things: confirming the label frequency distribution and canonicalising ICD-10
code formatting.

### Surgical Standardisation

1. **Forensic Label Frequency Analysis:** The DuckDB `silver_vault` is queried to
   profile label frequency across all 2,037 distinct ICD-10 codes. This serves as
   a sanity check against the MedSynth paper's stated design: uniform sampling of
   exactly 5 dialogue-note pairs per code. We expect zero rare labels (freq < 2)
   and a minimum frequency of 5 across all codes. Any deviation would indicate
   either a data loading issue or an undocumented change to the HuggingFace release.

2. **Canonical Decimal Restoration:** Raw MedSynth codes omit the ICD-10 decimal
   point (e.g. `M25562`). We restore the canonical format by inserting a decimal
   after the third character (e.g. `M25562 → M25.562`), matching the CDC FY2026
   standard. Note that Phase 1b validation was performed in decimal-free space, so
   this is a canonicalisation step only — it does not alter billability
   classifications.

   3-character parent codes (e.g. `R32`, `J18`) are a special case — they have no
   subclassification component so no decimal is inserted, leaving them as-is. This
   matches their representation in the CDC FY2026 reference.

3. **Gold Layer Promotion:** The standardised records are registered as the
   `gold_medsynth` artifact, creating a recovery point that carries both the raw
   label array and the canonical `standard_icd10` string.

4. **Audit Visibility:** The `standard_icd10` column is displayed immediately after
   restoration to provide visual confirmation that the decimal insertion logic is
   functioning correctly before any downstream transformation consumes it.

> **Result:** A standardised Gold dataset with canonical ICD-10 codes and a confirmed
> uniform label distribution — 2,037 distinct codes, minimum frequency 5, zero rare
> labels — consistent with the MedSynth paper's uniform sampling design.

</div>

In [ ]:
# ==============================================================================
# PHASE 2a: BIOLOGICAL AUDIT & DECIMAL RESTORATION (GOLD LAYER)
# ==============================================================================

import polars as pl
from datetime import datetime

# ==============================================================================
# 1. VERIFY PREREQUISITES
# ==============================================================================
print("🔍 Verifying Phase 2a Prerequisites...")

if 'df_valid' not in locals():
    print("❌ CRITICAL: df_valid not found. Please run Phase 1e first.")
    raise ValueError("df_valid not available in local scope")

print(f"   ✅ df_valid available: {len(df_valid):,} records")

# ==============================================================================
# 2. OPEN FRESH DUCKDB CONNECTION
# ==============================================================================
# Phase 1g closed its connection on completion. We open a fresh one here.
# silver_vault was persisted to disk by Phase 1g so it is available immediately.
# ==============================================================================
con = config.get_duckdb_conn()

try:
    # Verify silver_vault is accessible before proceeding
    vault_check = con.execute("SELECT COUNT(*) as n FROM silver_vault").pl()
    print(f"   ✅ silver_vault accessible: {vault_check['n'].item():,} records")

    
    # ==============================================================================
    # 3. FORENSIC LABEL FREQUENCY ANALYSIS
    # ==============================================================================
    # We query the Silver Vault to build a frequency profile of all ICD-10 labels.
    # This serves as a sanity check against the MedSynth paper's stated design:
    # uniform sampling of exactly 5 dialogue-note pairs per ICD-10 code.
    # Expected result: 0 rare labels (freq < 2), minimum frequency of 5.
    # Any deviation would indicate a data loading issue or an undocumented change
    # to the HuggingFace release.
    # ==============================================================================
    
    
    print("\n🕵️  Running Forensic Label Frequency Analysis...")

    label_stats = con.execute("""
        SELECT
            COUNT(DISTINCT label) as unique_codes,
            COUNT(*)              as total_records
        FROM silver_vault
    """).pl()

    rare_labels = con.execute("""
        SELECT label, COUNT(*) as freq
        FROM silver_vault
        GROUP BY label
        HAVING freq < 2
        ORDER BY freq ASC
    """).pl()

    print(f"   📊 Total unique labels:     {label_stats['unique_codes'].item():,}")
    print(f"   ⚠️  Rare labels (freq < 2): {len(rare_labels):,}")

    config.log_event(
        phase="Phase 2a: Biological Audit",
        action="label_frequency_analysis_complete",
        details={
            "total_unique_codes": label_stats['unique_codes'].item(),
            "rare_label_count": len(rare_labels),
        }
    )

    # ==============================================================================
    # 4. CANONICAL DECIMAL RESTORATION
    # ==============================================================================
    # Raw MedSynth codes omit the ICD-10 decimal point (e.g. M25562, N390).
    # We restore the canonical CDC format by inserting a decimal after the third
    # character (e.g. M25562 → M25.562, N390 → N39.0).
    #
    # IMPORTANT — 3-character codes are a special case:
    #   Codes like R32, J18, L80 are category-level parent codes (Noisy 111).
    #   They have no subclassification component, so inserting a decimal produces
    #   a trailing dot (e.g. R32.) which is invalid. These codes are left as-is,
    #   which matches their representation in the CDC FY2026 reference.
    #
    # This restoration is a canonicalisation step only — it does not alter which
    # codes are considered billable or non-billable (that was determined in Phase 1b).
    #
    # Placeholder-X codes (e.g. T781XXA, identified in Phase 1b) will receive a
    # decimal after position 3 (e.g. T78.1XXA). These are structurally unusual
    # but the transformation is applied consistently.
    # ==============================================================================
    print("\n🔬 Restoring ICD-10 Decimal Points...")

    # Step 1: Extract the first code from the label list.
    # Phase 1g confirmed this dataset is single-code per record, so list.first()
    # is safe and correct here.
    df_gold = df_valid.with_columns([
        pl.col("label").list.first().fill_null("").alias("raw_code")
    ])

    # Step 2: Conditionally insert decimal after 3rd character.
    #
    # Rule:
    #   len > 3  →  insert decimal  (e.g. M25562 → M25.562, N390 → N39.0)
    #   len == 3 →  leave as-is     (e.g. R32 → R32, J18 → J18)
    #   len == 0 →  leave as-is     (empty string — caught by validation below)
    #
    # NOTE: This must be a separate with_columns() call. Polars requires raw_code
    # to exist as a materialised column before it can be referenced in the next
    # expression — it cannot be defined and referenced in the same call.
    df_gold = df_gold.with_columns([
        pl.when(pl.col("raw_code").str.len_chars() > 3)
        .then(
            pl.col("raw_code").str.slice(0, 3) + "." + pl.col("raw_code").str.slice(3)
        )
        .otherwise(pl.col("raw_code"))
        .alias("standard_icd10")
    ])

    # Step 3: Validate restoration results.
    # A valid standard_icd10 must match one of two patterns:
    #   - Canonical format:   3 chars + decimal + 1+ chars  (e.g. M25.562, N39.0)
    #   - Parent code format: exactly 3 alphanumeric chars   (e.g. R32, J18)
    # Anything else (empty string, trailing dot, malformed) is flagged as invalid.
    validation_check = df_gold.with_columns(
        (
            pl.col("standard_icd10").str.contains(r"^\w{3}\.\w+$") |  # canonical
            pl.col("standard_icd10").str.contains(r"^\w{3}$")          # parent code
        ).alias("has_valid_format")
    )

    valid_format_pct = (
        validation_check.select(pl.col("has_valid_format").mean() * 100).item()
    )

    # Count each format type for the detailed audit summary
    canonical_count = validation_check.filter(
        pl.col("standard_icd10").str.contains(r"^\w{3}\.\w+$")
    ).height
    parent_count = validation_check.filter(
        pl.col("standard_icd10").str.contains(r"^\w{3}$")
    ).height
    invalid_count = validation_check.filter(
        ~pl.col("has_valid_format")
    ).height

    print(f"   ✅ Decimal restoration complete")
    print(f"   ✅ Valid format percentage:  {valid_format_pct:.1f}%")
    print(f"      ├─ Canonical (X##.###):  {canonical_count:,}")
    print(f"      ├─ Parent codes (X##):   {parent_count:,}")
    print(f"      └─ Invalid / empty:      {invalid_count:,}")

    # Surface any remaining invalids for inspection
    if invalid_count > 0:
        print(f"\n⚠️  INVALID FORMAT SAMPLE (first 10):")
        invalid_sample = validation_check.filter(
            ~pl.col("has_valid_format")
        ).select(["raw_code", "standard_icd10"]).unique().head(10)
        try:
            from IPython.display import display
            display(invalid_sample)
        except ImportError:
            print(invalid_sample)

    # ==============================================================================
    # 5. REGISTER GOLD LAYER
    # ==============================================================================
    print("\n💾 Registering Gold Layer...")

    config.register_dataframe("gold_medsynth", df_gold, phase="Phase 2a")
    print(f"   ✅ Gold Layer registered: {len(df_gold):,} records")

    # ==============================================================================
    # 6. AUDIT VISIBILITY: GOLD LAYER PREVIEW
    # ==============================================================================
    print("\n👁️  Gold Layer Preview (First 10 Records):")
    try:
        from IPython.display import display
        display(df_gold.select(["id", "label", "raw_code", "standard_icd10"]).head(10))
    except ImportError:
        print(df_gold.select(["id", "label", "raw_code", "standard_icd10"]).head(10))

    # ==============================================================================
    # 7. LOG THE TRANSFORMATION
    # ==============================================================================
    config.log_event(
        phase="Phase 2a: Biological Audit",
        action="decimal_restoration_complete",
        details={
            "total_records": len(df_gold),
            "valid_format_pct": round(valid_format_pct, 2),
            "canonical_format_count": canonical_count,
            "parent_code_count": parent_count,
            "invalid_format_count": invalid_count,
            "rare_label_count": len(rare_labels),
            "gold_artifact": "gold_medsynth"
        }
    )

    print(f"\n📝 Audit trail updated")

    # ==============================================================================
    # 8. ZERO-TRUST VERDICT
    # ==============================================================================
    print(f"\n🛡️  ZERO-TRUST VERDICT:")

    if valid_format_pct == 100.0:
        print(f"   ✅ 100% of codes have valid decimal format")
        print(f"   ✅ Gold layer is ready for downstream transformations")
    elif valid_format_pct >= 95:
        print(f"   ✅ {valid_format_pct:.1f}% of codes have valid decimal format")
        print(f"   ℹ️  {invalid_count} records with invalid format — review sample above")
        print(f"   ✅ Gold layer is ready for downstream transformations")
    elif valid_format_pct >= 80:
        print(f"   ⚠️  {valid_format_pct:.1f}% of codes have valid decimal format")
        print(f"   ⚠️  Review records with invalid formats before proceeding")
    else:
        print(f"   ❌ {valid_format_pct:.1f}% of codes have valid decimal format")
        print(f"   ❌ CRITICAL: Decimal restoration failed. Review Phase 1e validation.")

    # ==============================================================================
    # 9. AUDIT SUMMARY
    # ==============================================================================
    print(f"\n📊 PHASE 2a AUDIT SUMMARY")
    print(f"   " + "─" * 50)
    print(f"   📊 Total Records:              {len(df_gold):,}")
    print(f"   ✅ Valid Decimal Format:       {valid_format_pct:.1f}%")
    print(f"      ├─ Canonical (X##.###):    {canonical_count:,}")
    print(f"      └─ Parent codes (X##):     {parent_count:,}")
    print(f"   ⚠️  Invalid / empty:           {invalid_count:,}")
    print(f"   ⚠️  Rare Labels (freq < 2):    {len(rare_labels):,}")
    print(f"   🏅 Gold Artifact:              gold_medsynth")
    print(f"   " + "─" * 50)
    print(f"\n✅ PHASE 2a COMPLETE: Gold layer registered and validated")

finally:
    con.close()
    print("🦆 DuckDB connection for Phase 2a closed.")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🔬 Phase 2b: Gold Layer Discovery Auditor

Following decimal restoration in Phase 2a, we launch a browser-native auditor
to visually verify the Gold layer output on a record-by-record basis. This is a
human-in-the-loop confirmation step before any downstream modelling work begins.

### Audit Objectives

1. **Decimal Restoration HUD:** Each record displays the raw label (e.g. `M25562`)
   alongside the canonical Gold label (e.g. `M25.562`), allowing direct visual
   confirmation that the restoration logic applied correctly. 3-character parent
   codes (e.g. `R32`) are shown without a decimal, which is their correct canonical
   form.

2. **Billability Status:** Records are flagged based on their Phase 1b
   classification — billable leaf codes, Noisy 111 parent codes, or placeholder-X
   codes. This connects the visual audit to the validated CDC reference findings
   rather than relying on frequency thresholds.

3. **Column-Key Recovery:** The auditor implements case-insensitive column lookup
   to locate clinical text fields regardless of case-shifting during Polars-to-pandas
   conversion, preventing silent failures on column access.

4. **Ink-Black Clinical Focus:** High-contrast monospace rendering ensures SOAP
   section headers remain legible, allowing confirmation that diagnostic signal
   in the Assessment section is intact after Gold layer promotion.

> **Result:** A record-level visual audit of the Gold layer confirming that decimal
> restoration is correct and clinical content is preserved before proceeding to
> downstream transformation phases.

</div>

In [ ]:
# ==============================================================================
# PHASE 2b: GOLD LAYER DISCOVERY AUDITOR (INK-BLACK)
# ==============================================================================

import panel as pn
import numpy as np
import pandas as pd
from datetime import datetime

pn.extension(design='material')


def launch_biological_auditor(df_gold, rare_labels_pl):
    df_pd = df_gold.to_pandas()

    # --- COLUMN NAME RECOVERY LOGIC ---
    # Case-insensitive lookup guards against column case-shifting during
    # Polars → pandas conversion (e.g. 'Note' becoming 'note').
    col_map  = {c.lower(): c for c in df_pd.columns}
    note_key = col_map.get('note',      'Note')
    dial_key = col_map.get('dialogue',  'Dialogue')

    index_slider = pn.widgets.IntSlider(
        name='Examine Gold Layer Record (Index)',
        start=0, end=len(df_pd) - 1, value=51,
        bar_color='#8e44ad', sizing_mode='stretch_width'
    )

    @pn.depends(index_slider)
    def get_audit_view(index):
        row = df_pd.iloc[index]

        raw_label      = row.get('label',        'N/A')
        standard_label = row.get('standard_icd10','N/A')
        note_content   = row.get(note_key,  "⚠️ Note column not found.")
        dial_content   = row.get(dial_key,  "⚠️ Dialogue column not found.")

        # Safely convert label to display string
        if isinstance(raw_label, (list, np.ndarray)):
            raw_label_str = ', '.join(str(c) for c in raw_label) if len(raw_label) > 0 else '[]'
        elif pd.isna(raw_label):
            raw_label_str = 'N/A'
        else:
            raw_label_str = str(raw_label)

        # Determine decimal restoration status for the HUD
        std = str(standard_label)
        import re
        if re.match(r'^\w{3}\.\w+$', std):
            restore_status  = "✅ CANONICAL FORMAT"
            restore_color   = "#27ae60"
        elif re.match(r'^\w{3}$', std):
            restore_status  = "ℹ️ PARENT CODE (NO DECIMAL)"
            restore_color   = "#2980b9"
        else:
            restore_status  = "⚠️ REVIEW REQUIRED"
            restore_color   = "#e74c3c"

        mapping_hud = f"""
        <div style="display: flex; gap: 20px; max-width: 850px; margin-bottom: 20px;">
            <div style="flex: 1; padding: 15px; background: #f8f9fa;
                        border: 2px solid #ddd; border-radius: 8px; text-align: center;">
                <span style="font-size: 0.8rem; color: #666; font-weight: bold;
                             text-transform: uppercase;">Raw Label</span><br>
                <span style="font-size: 1.8rem; color: #333;
                             font-weight: 900;">{raw_label_str}</span>
            </div>
            <div style="flex: 1; padding: 15px; background: #fff;
                        border: 2px solid #2980b9; border-radius: 8px; text-align: center;">
                <span style="font-size: 0.8rem; color: #2980b9; font-weight: bold;
                             text-transform: uppercase;">Canonical (Gold)</span><br>
                <span style="font-size: 1.8rem; color: #2980b9;
                             font-weight: 900;">{standard_label}</span>
            </div>
            <div style="flex: 1; padding: 15px; background: {restore_color};
                        border-radius: 8px; text-align: center; color: white;">
                <span style="font-size: 0.8rem; font-weight: bold;
                             text-transform: uppercase;">Restoration Status</span><br>
                <span style="font-size: 1.1rem; font-weight: 700;">{restore_status}</span>
            </div>
        </div>
        """

        ink_black_style = """
        <style>
            .audit-text-area {
                white-space: pre-wrap !important;
                word-wrap: break-word !important;
                font-family: 'Courier New', monospace !important;
                font-size: 1.0rem !important;
                font-weight: 700 !important;
                color: #000000 !important;
                background-color: #ffffff !important;
                padding: 20px;
                border: 1px solid #000;
                line-height: 1.5;
                -webkit-font-smoothing: none;
            }
        </style>
        """

        content = f'{ink_black_style}<div class="audit-text-area">{note_content}</div>'

        return pn.Column(
            pn.pane.HTML(mapping_hud, sizing_mode='stretch_width'),
            pn.Row(
                pn.Column(
                    "### 📝 Clinical Note (Gold Layer)",
                    pn.pane.HTML(content),
                    height=600, scroll=True, sizing_mode='stretch_width'
                ),
                pn.Column(
                    "### 💬 Source Dialogue",
                    pn.pane.Str(
                        dial_content,
                        styles={'white-space': 'pre-wrap', 'font-weight': '500'}
                    ),
                    height=600, scroll=True, sizing_mode='stretch_width'
                )
            )
        )

    layout = pn.Column(
        "# 🔬 Gold Layer Discovery Auditor — Phase 2b\n"
        "**Objective:** Visually confirm decimal restoration and clinical content "
        "integrity across the Gold layer before downstream transformation.",
        index_slider,
        get_audit_view,
        sizing_mode='stretch_width'
    )

    return layout.show(title="MedSynth Gold Layer Auditor — Phase 2b", threaded=True)


# ==============================================================================
# LAUNCH AUDITOR
# Uses df_gold and rare_labels produced by Phase 2a
# ==============================================================================
if 'df_gold' not in locals():
    print("❌ df_gold not found. Please run Phase 2a first.")
else:
    # rare_labels comes from Phase 2a's DuckDB query
    if 'rare_labels' not in locals():
        print("⚠️  rare_labels not found — creating empty placeholder.")
        import polars as pl
        rare_labels_pl = pl.DataFrame({"label": [], "freq": []})
    else:
        rare_labels_pl = rare_labels

    print("🚀 Launching Gold Layer Discovery Auditor (Phase 2b)...")
    print("   💡 TIP: Use the slider to navigate records.")
    print("   💡 TIP: Verify canonical format (X##.###) for 4+ char codes.")
    print("   💡 TIP: Parent codes (3 chars, no decimal) are correct as-is.")

    bio_server = launch_biological_auditor(df_gold, rare_labels_pl)

    # ==============================================================================
    # AUDIT TRAIL
    # ==============================================================================
    config.log_event(
        phase="Phase 2b: Gold Layer Auditor",
        action="auditor_launched",
        details={
            "total_records": len(df_gold),
            "rare_labels_count": len(rare_labels_pl) if rare_labels_pl is not None else 0,
        }
    )

    print(f"\n📝 Audit trail updated")
    print(f"✅ Phase 2b COMPLETE: Gold layer auditor launched")

    print(f"\n🛡️  ZERO-TRUST VERDICT:")
    if len(rare_labels_pl) > 0:
        print(f"   ⚠️  {len(rare_labels_pl):,} rare labels identified — review before proceeding")
    else:
        print(f"   ✅ No rare labels found — all codes meet minimum frequency threshold")
        print(f"   ✅ Consistent with MedSynth uniform sampling design (5 per code)")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif; border: 2px solid #e74c3c; padding: 20px; border-radius: 8px;">

## 🚨 Forensic Alert: Diagnostic Leakage Detected

During the **Phase 3a.1 Biological Audit**, visual inspection of the clinical narratives (Index 51) revealed a critical threat to model integrity: **Explicit Label Leakage** [cite: 2026-02-12, 2026-02-23].

### The Discovery
In the audited sample for **E66.9 (Obesity, unspecified)**, the raw clinical text explicitly contains the diagnostic answer within the `**3. Assessment:**` block [cite: 2026-02-23]:
> *"...Diagnosis: Obesity, unspecified (ICD-10: E66.9)"*

![Forensic Evidence of ICD-10 Leakage](./resources/images/icd-10_code_leakage.png)

### The Risk: "The Shortcut Trap"
If left unaddressed, ClinicalBERT will ignore the clinical evidence in the **Subjective** and **Objective** sections and simply "cheat" by extracting the code directly from the text [cite: 2026-02-12]. This leads to:
1. **Inflated Accuracy Metrics**: 99%+ accuracy in training that collapses in real-world deployment where codes aren't pre-written [cite: 2026-01-29].
2. **Zero Clinical Reasoning**: The model fails to learn the relationship between symptoms (e.g., BMI, joint pain) and the diagnosis [cite: 2026-02-12].

### Surgical Mitigation Required
This confirms the absolute necessity of the **Phase 5 Greedy Scrub** [cite: 2026-02-12]. We must programmatically redact:
* **Literal ICD-10 Codes**: Any string matching the `[A-Z][0-9][0-9]` pattern.
* **Semantic Labels**: The exact diagnostic string (e.g., "Obesity, unspecified") when it appears in the assessment block.

> **🛡️ Auditor Verdict**: The APSO-flip moves this leaked signal to Token 0, making it even easier for the model to "cheat." **Redaction is no longer optional; it is a prerequisite for a valid training run** [cite: 2026-01-27, 2026-02-23].

</div>

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🏷️ Phase 2c: Gold Layer Status Annotation

With the Gold layer persisted in Phase 2a and visually verified in Phase 2b, we
now formally embed the ICD-10 validation findings from Phase 1b directly into the
Gold layer as a first-class column. This closes the loop on the Noisy 111 analysis
and gives every downstream phase the information needed to make its own informed
decision about how to handle each code category.

No records are removed or modified. The annotation is additive only.

---

### The Three Status Categories

Every record in the Gold layer receives one of three status values, derived
directly from the Phase 1b CDC FY2026 reference join:

| Status | Description | Count | % |
|---|---|---|---|
| `billable` | Confirmed leaf code in CDC FY2026 tabular reference | 9,660 | 94.3% |
| `noisy_111` | Valid ICD-10 parent code, too broad for billing | 60 | 0.6% |
| `placeholder_x` | Structurally valid ICD-10-CM injury code, absent from CDC descriptions file | 25 | 0.24% |
| `invalid_or_malformed` | Code failed all classification tests; ambiguous | 495 | 4.8% |

The `noisy_111` and `placeholder_x` classifications are **not errors**. As
established in Phase 1b and confirmed by the MedSynth paper, the Noisy 111 codes
reflect real clinical coding practice from IQVIA insurance claims, and the
placeholder-X codes are legitimate injury codes not expanded in the CM tabular
reference.

---

### Why Annotate Rather Than Remove?

The appropriate treatment of non-billable codes depends on the downstream
modelling objective — which has not yet been fixed. Annotating preserves optionality:

* A **multi-class classification** task may want to exclude `noisy_111` records
  to avoid label ambiguity.
* A **hierarchical classification** task may want to retain them and use the
  parent-child ICD-10 structure explicitly.
* An **information extraction** task (e.g. extracting clinical entities) does not
  need billable codes at all and should retain everything.

By embedding `code_status` in the Gold layer now, any of these downstream
strategies can be implemented with a single filter — without rerunning the
CDC validation pipeline.

---

### Implementation

The status lookup is built from `df_checked` — the record-level validation
DataFrame produced by Phase 1b — and joined onto `df_gold` by `id`. The Phase 1b
three-bucket vocabulary is mapped to cleaner status labels:

- `"non_billable_parent (Noisy 111)"` → `"noisy_111"`
- `"invalid_or_malformed"` → `"placeholder_x"` *(more precise — these are not
  malformed, they are absent from the tabular reference)*

> **Result:** A fully annotated Gold layer where every record carries a
> validated, CDC-grounded `code_status` — forming the complete, audit-ready
> foundation for all downstream modelling phases.

</div>

In [ ]:
# ==============================================================================
# PHASE 2c: GOLD LAYER STATUS ANNOTATION
# ==============================================================================
# Purpose: Formally annotate every record in the Gold layer with its ICD-10
# validation status, derived from the Phase 1b CDC FY2026 reference join.
#
# Status values (four categories):
#   'billable'            — confirmed leaf code in CDC FY2026 tabular reference
#   'noisy_111'           — valid ICD-10 parent code, too broad for billing
#   'placeholder_x'       — structurally valid ICD-10-CM injury code, absent
#                           from CDC descriptions file (placeholder X convention)
#   'invalid_or_malformed'— code failed all classification tests; ambiguous
# ==============================================================================

import polars as pl

# ==============================================================================
# 1. VERIFY PREREQUISITES
# ==============================================================================
print("🔍 Verifying Phase 2c Prerequisites...")

required = {'df_gold': df_gold, 'df_checked': df_checked, 'cdc_df': cdc_df}
for name, obj in required.items():
    if obj is None or (hasattr(obj, '__len__') and len(obj) == 0):
        raise ValueError(f"❌ CRITICAL: {name} is empty or None. Run Phase 1b and 2a first.")
    print(f"   ✅ {name} available: {len(obj):,} records")

# ==============================================================================
# 2. BUILD STATUS LOOKUP FROM PHASE 1b df_checked
# ==============================================================================
print("\n🔧 Building status lookup from Phase 1b validation results...")

# Four-bucket mapping — scientifically precise, no collapsing of distinct categories:
#   billable            — 9,660 confirmed leaf codes
#   noisy_111           —    60 valid parent codes (real clinical billing practice)
#   placeholder_x       —    25 valid ICD-10-CM injury codes absent from CM tabular
#   invalid_or_malformed—   495 codes that failed all classification tests
STATUS_MAP = {
    "billable":                        "billable",
    "non_billable_parent (Noisy 111)": "noisy_111",
    "placeholder_x":                   "placeholder_x",
    "invalid_or_malformed":            "invalid_or_malformed"
}

status_lookup = (
    df_checked
    .with_columns(
        pl.col("status")
        .replace(list(STATUS_MAP.keys()), list(STATUS_MAP.values()))
        .alias("code_status")
    )
    .select(["ID", "code_status"])
    .unique(subset=["ID"], keep="first")
)

print(f"   ✅ Status lookup built: {len(status_lookup):,} records")

status_dist = status_lookup.group_by("code_status").agg(
    pl.len().alias("count")
).with_columns(
    (pl.col("count") / pl.col("count").sum() * 100).alias("pct")
).sort("count", descending=True)

print(f"\n📊 Status distribution (should match Phase 1b validation summary):")
try:
    from IPython.display import display
    display(status_dist)
except ImportError:
    print(status_dist)

# ==============================================================================
# 3. JOIN STATUS ONTO GOLD LAYER
# ==============================================================================
print("\n🔗 Joining status annotation onto Gold layer...")

# Drop code_status if it already exists from a previous run
if "code_status" in df_gold.columns:
    df_gold = df_gold.drop("code_status")

df_gold_annotated = df_gold.join(
    status_lookup.with_columns(pl.col("ID").cast(pl.String).alias("id")),
    on="id",
    how="left"
)

join_check = df_gold_annotated.filter(pl.col("code_status").is_null()).height
if join_check > 0:
    print(f"   ⚠️  WARNING: {join_check:,} records have null code_status after join.")
else:
    print(f"   ✅ Join complete: {len(df_gold_annotated):,} records, 0 null status values")

# ==============================================================================
# 4. UPDATE df_gold
# ==============================================================================
df_gold = df_gold_annotated.drop("ID")

print(f"\n   ✅ df_gold updated with code_status column")
print(f"   📊 Gold layer schema: {df_gold.schema}")

# ==============================================================================
# 5. SUMMARY REPORT
# ==============================================================================
print(f"\n📊 PHASE 2c ANNOTATION SUMMARY")
print(f"   " + "─" * 50)

final_dist = df_gold.group_by("code_status").agg(
    pl.len().alias("count")
).with_columns(
    (pl.col("count") / pl.col("count").sum() * 100).alias("pct")
).sort("count", descending=True)

icons = {
    "billable":             "✅",
    "noisy_111":            "ℹ️ ",
    "placeholder_x":        "🔍",
    "invalid_or_malformed": "⚠️ "
}

for row in final_dist.iter_rows(named=True):
    status = row["code_status"]
    count  = row["count"]
    pct    = row["pct"]
    icon   = icons.get(status, "📊")
    print(f"   {icon} {status:22s}: {count:>6,} records ({pct:.1f}%)")

print(f"   " + "─" * 50)
print(f"   📊 Total:                    {len(df_gold):,} records")

# ==============================================================================
# 6. RE-REGISTER UPDATED GOLD LAYER
# ==============================================================================
print(f"\n💾 Re-registering annotated Gold layer...")
config.register_dataframe("gold_medsynth", df_gold, phase="Phase 2c")
print(f"   ✅ gold_medsynth updated with code_status annotation")

# ==============================================================================
# 7. AUDIT TRAIL
# ==============================================================================
billable_count    = df_gold.filter(pl.col("code_status") == "billable").height
noisy_count       = df_gold.filter(pl.col("code_status") == "noisy_111").height
placeholder_count = df_gold.filter(pl.col("code_status") == "placeholder_x").height
invalid_count     = df_gold.filter(pl.col("code_status") == "invalid_or_malformed").height

config.log_event(
    phase="Phase 2c: Gold Layer Status Annotation",
    action="status_annotation_complete",
    details={
        "total_records":            len(df_gold),
        "billable_count":           billable_count,
        "billable_pct":             round(billable_count / len(df_gold) * 100, 2),
        "noisy_111_count":          noisy_count,
        "noisy_111_pct":            round(noisy_count / len(df_gold) * 100, 2),
        "placeholder_x_count":      placeholder_count,
        "placeholder_x_pct":        round(placeholder_count / len(df_gold) * 100, 2),
        "invalid_or_malformed_count": invalid_count,
        "invalid_or_malformed_pct": round(invalid_count / len(df_gold) * 100, 2),
        "annotation_source":        "Phase 1b CDC FY2026 validation (df_checked)",
        "null_status_records":      join_check
    }
)

print(f"\n📝 Audit trail updated")

# ==============================================================================
# 8. ZERO-TRUST VERDICT
# ==============================================================================
print(f"\n🛡️  ZERO-TRUST VERDICT:")
print(f"   ✅ All {len(df_gold):,} Gold layer records carry a validated status")
print(f"   ✅ Status derived from CDC FY2026 reference join (Phase 1b)")
print(f"   ✅ Four distinct categories preserved — no scientific collapsing")
print(f"   ℹ️  Downstream phases should filter or weight by code_status")
print(f"      as appropriate for their specific modelling objectives.")
print(f"\n✅ PHASE 2c COMPLETE: Gold layer fully annotated and re-registered")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## ✅ Pipeline Complete: The Annotated Gold Layer

This concludes the forensic ingestion and validation pipeline. Rather than
pruning records based on a rejected semantic recovery experiment, we have taken
a more rigorous approach — annotating every record with its validated ICD-10
status and preserving the full dataset for downstream use.

### What the Pipeline Produced

Starting from 10,240 raw MedSynth records, the pipeline has delivered a fully
validated, annotated Gold layer with the following properties:

| Property | Value |
|---|---|
| Total records | 10,240 |
| Billable (CDC FY2026 leaf codes) | 9,660 (94.3%) |
| Noisy 111 (valid parent codes) | 60 (0.6%) |
| Placeholder-X (injury codes) | 25 (0.24%) |
| SOAP extraction success | 100% |
| Canonical ICD-10 format | 100% |
| Empty dialogue records | 2 (sentinel imputed) |

### On the Noisy 111 Records

The 60 records carrying non-billable parent codes are **not errors**. The
MedSynth authors selected codes from the top 2,001 most frequent ICD-10 codes
in real IQVIA insurance claims data, and clinicians genuinely submit
category-level codes in practice. These records are retained in the Gold layer
with a `code_status` of `noisy_111` — downstream phases can include or exclude
them based on their specific modelling requirements.

### What Each Record Carries

Every record in `df_gold` / `gold_medsynth` now contains:

* Original clinical note and doctor-patient dialogue
* Raw ICD-10 label array (decimal-free, as stored in source data)
* Canonical `standard_icd10` string (decimal-restored, CDC FY2026 format)
* Extracted SOAP sections (Subjective, Objective, Assessment, Plan)
* `code_status` classification grounded in CDC FY2026 reference join

### Next Steps

Downstream phases should select records appropriate to their task:
```python
# Billable records only (strictest label quality)
df_billable = df_gold.filter(pl.col("code_status") == "billable")  # 9,660 records

# All records including parent codes (maximum data)
df_all = df_gold  # 10,240 records

# Exclude only the placeholder-X codes
df_no_placeholders = df_gold.filter(pl.col("code_status") != "placeholder_x")  # 10,215 records
```

> **Result:** A transparent, audit-ready Gold layer where every design decision
> is documented, every record is traceable to its source, and no data has been
> irreversibly discarded.

</div>

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🔄 Phase 3a: APSO-Flip (Diagnostic Signal Prioritisation)

With the Gold layer fully annotated in Phase 2c, we now address the truncation
problem quantified in Phase 1c: 64.7% of clinical notes exceed ClinicalBERT's
512-token context window, and in standard SOAP format the Assessment section —
which contains the primary diagnostic signal — appears last and is routinely
truncated away.

### The APSO-Flip

The transformation reorders the four SOAP sections into **APSO format**:
**Assessment → Plan → Subjective → Objective**. This places the diagnostic
signal at Token 0, guaranteeing its survival through the model's context window
regardless of note length.

The Gold layer already contains pre-extracted SOAP sections as individual
columns (`assessment`, `plan`, `subjective`, `objective`), produced by the
Phase 1e Pydantic Gatekeeper with 100% extraction success. The APSO-Flip
recomposes these directly — no regex re-processing of the raw note is required.
This is more robust and avoids re-introducing section boundary detection
problems the Gatekeeper already solved.

### What Changes, What Doesn't

* **Added:** `apso_note` column — Assessment-first recomposition of the note
* **Added:** `apso_token_estimate` — estimated token count for the APSO note
* **Preserved:** `note` column — original SOAP note retained for full audit
  traceability and for tasks where the original ordering is preferred
* **Unchanged:** All other columns including `code_status`, `standard_icd10`,
  `label`, and all extracted SOAP section columns

The APSO-Flip does not reduce token count — the same content is present, just
reordered. The transformation protects the Assessment from truncation by moving
it to the front; it does not compress the note.

### Scope

The transformation is applied to all 10,240 Gold layer records regardless of
`code_status`. Downstream phases should select records by `code_status` as
appropriate for their modelling objective:
```python

In [ ]:
# ==============================================================================
# PHASE 3a: APSO-FLIP (DIAGNOSTIC SIGNAL PRIORITISATION)
# ==============================================================================
# Purpose: Reorder each clinical note from standard SOAP format into APSO format
# (Assessment, Plan, Subjective, Objective), ensuring the primary diagnostic
# signal appears at Token 0 and survives ClinicalBERT's 512-token context window.
#
# Approach: The Gold layer already contains pre-extracted SOAP sections as
# individual columns (assessment, plan, subjective, objective), produced by the
# Phase 1e Pydantic Gatekeeper. We recompose these directly rather than
# re-running regex on the raw note — this is more robust, faster, and avoids
# re-introducing the section boundary detection problem the Gatekeeper already
# solved for all 10,240 records.
#
# Dataset scope: The full Gold layer (10,240 records) is transformed.
# Downstream phases can filter by code_status as appropriate for their task.
# ==============================================================================

import polars as pl

# ==============================================================================
# 1. VERIFY PREREQUISITES
# ==============================================================================
print("🔍 Verifying Phase 3a Prerequisites...")

if 'df_gold' not in locals() or df_gold is None:
    print("❌ CRITICAL: df_gold not found. Please run Phase 2c first.")
    raise ValueError("df_gold not available in local scope")

required_cols = {'assessment', 'plan', 'subjective', 'objective', 'note', 'code_status'}
missing_cols  = required_cols - set(df_gold.columns)
if missing_cols:
    raise ValueError(
        f"❌ CRITICAL: df_gold is missing required columns: {missing_cols}\n"
        f"   Please run Phase 1e (Pydantic Firewall) and Phase 2c (Status Annotation) first."
    )

print(f"   ✅ df_gold available: {len(df_gold):,} records")
print(f"   ✅ All required SOAP columns present")

# Confirm SOAP extraction coverage before transforming
soap_coverage = {
    section: df_gold.filter(pl.col(section).is_not_null()).height
    for section in ['subjective', 'objective', 'assessment', 'plan']
}
print(f"\n📊 SOAP Column Coverage (pre-transformation):")
for section, count in soap_coverage.items():
    pct = count / len(df_gold) * 100
    print(f"   {section.capitalize():12s}: {count:,}/{len(df_gold):,} ({pct:.1f}%)")

# ==============================================================================
# 2. APSO RECOMPOSITION
# ==============================================================================
# Reorder sections: Assessment → Plan → Subjective → Objective
#
# Rationale (from Phase 1c findings):
#   - 64.7% of Notes exceed 512 tokens
#   - Standard SOAP places Assessment at the end — it is routinely truncated
#   - Placing Assessment at Token 0 guarantees diagnostic signal survival
#     regardless of note length
#
# Null handling: fill_null("") ensures records with missing sections still
# produce a valid apso_note rather than propagating nulls.
#
# Note: This transformation does not modify the original 'note' column.
# Both the original SOAP note and the recomposed APSO note are retained
# in the Gold layer for full audit traceability.
# ==============================================================================
print("\n🔄 Applying APSO-Flip transformation...")

df_gold = df_gold.with_columns([
    (
        pl.col("assessment").fill_null("") + "\n\n" +
        pl.col("plan").fill_null("")       + "\n\n" +
        pl.col("subjective").fill_null("") + "\n\n" +
        pl.col("objective").fill_null("")
    ).str.strip_chars().alias("apso_note")
])

# Verify no null apso_note values
null_apso = df_gold.filter(pl.col("apso_note").is_null()).height
empty_apso = df_gold.filter(pl.col("apso_note") == "").height

if null_apso > 0 or empty_apso > 0:
    print(f"   ⚠️  WARNING: {null_apso} null and {empty_apso} empty apso_note values.")
    print(f"      Review SOAP extraction coverage in Phase 1e.")
else:
    print(f"   ✅ APSO recomposition complete: 0 null, 0 empty values")

# ==============================================================================
# 3. TOKEN PRESSURE AUDIT (POST-FLIP)
# ==============================================================================
# Estimate token counts for the recomposed APSO notes using the same
# words × 1.3 heuristic established in Phase 1c.
#
# The APSO-Flip does not reduce token count — the same content is present,
# just reordered. The purpose is not compression but prioritisation: ensuring
# the Assessment section occupies the first tokens in the model's context window.
# ==============================================================================
print("\n📊 Token Pressure Audit (APSO Notes)...")

TOKEN_HEURISTIC = 1.3
BERT_LIMIT      = 512

df_gold = df_gold.with_columns([
    (
        pl.col("apso_note")
        .str.count_matches(r"\S+")
        .fill_null(0)
        * TOKEN_HEURISTIC
    ).cast(pl.Float64).alias("apso_token_estimate")
])

total_records    = len(df_gold)
at_risk          = df_gold.filter(pl.col("apso_token_estimate") > BERT_LIMIT).height
within_window    = total_records - at_risk
at_risk_pct      = at_risk / total_records * 100
within_window_pct = within_window / total_records * 100

avg_tokens = df_gold.select(pl.col("apso_token_estimate").mean()).item()
max_tokens = df_gold.select(pl.col("apso_token_estimate").max()).item()

print(f"   📈 Average estimated tokens:  {avg_tokens:.0f}")
print(f"   📈 Maximum estimated tokens:  {max_tokens:.0f}")
print(f"   ✅ Within 512-token window:   {within_window:,} ({within_window_pct:.1f}%)")
print(f"   ⚠️  Exceeding 512-token limit: {at_risk:,} ({at_risk_pct:.1f}%)")
print(f"\n   ℹ️  Note: Token count is unchanged by the APSO-Flip.")
print(f"      The flip protects the Assessment section from truncation")
print(f"      by moving it to Token 0 — it does not reduce total length.")

# ==============================================================================
# 4. ASSESSMENT SECTION PREVIEW
# ==============================================================================
# Confirm Assessment content appears at the start of apso_note for the
# first few records — visual spot-check of the recomposition.
# ==============================================================================
print("\n👁️  APSO Note Preview (First 3 Records — Assessment Section Start):")
preview = df_gold.select(["id", "standard_icd10", "apso_note"]).head(3)
for row in preview.iter_rows(named=True):
    apso_preview = row["apso_note"][:200].replace("\n", " ↵ ")
    print(f"\n   ID: {row['id']} | Code: {row['standard_icd10']}")
    print(f"   {apso_preview}...")

# ==============================================================================
# 5. CODE STATUS BREAKDOWN OF TRANSFORMED RECORDS
# ==============================================================================
print(f"\n📊 Code Status Breakdown (post-APSO transformation):")
status_breakdown = df_gold.group_by("code_status").agg(
    pl.len().alias("count")
).with_columns(
    (pl.col("count") / pl.col("count").sum() * 100).alias("pct")
).sort("count", descending=True)

for row in status_breakdown.iter_rows(named=True):
    icon = "✅" if row["code_status"] == "billable" else (
           "ℹ️ " if row["code_status"] == "noisy_111" else "🔍")
    print(f"   {icon} {row['code_status']:20s}: {row['count']:>6,} ({row['pct']:.1f}%)")

# ==============================================================================
# 6. REGISTER UPDATED GOLD LAYER
# ==============================================================================
print(f"\n💾 Registering APSO-flipped Gold layer...")
config.register_dataframe("gold_medsynth", df_gold, phase="Phase 3a")
print(f"   ✅ gold_medsynth updated: {len(df_gold):,} records, "
      f"{len(df_gold.columns)} columns")

# ==============================================================================
# 7. AUDIT TRAIL
# ==============================================================================
config.log_event(
    phase="Phase 3a: APSO-Flip",
    action="apso_transformation_complete",
    details={
        "total_records":          len(df_gold),
        "null_apso_notes":        null_apso,
        "empty_apso_notes":       empty_apso,
        "avg_apso_tokens":        round(avg_tokens, 1),
        "max_apso_tokens":        round(max_tokens, 1),
        "records_within_window":  within_window,
        "records_exceeding_limit": at_risk,
        "records_exceeding_pct":  round(at_risk_pct, 2),
        "token_heuristic":        TOKEN_HEURISTIC,
        "bert_limit":             BERT_LIMIT,
        "transformation_method":  "pre-extracted SOAP columns (Phase 1e Gatekeeper)",
        "original_note_preserved": True
    }
)

print(f"\n📝 Audit trail updated")

# ==============================================================================
# 8. ZERO-TRUST VERDICT
# ==============================================================================
print(f"\n🛡️  ZERO-TRUST VERDICT:")
print(f"   ✅ APSO-Flip applied to all {len(df_gold):,} records")
print(f"   ✅ Assessment section now at Token 0 for every record")
print(f"   ✅ Original SOAP note preserved in 'note' column")
print(f"   ℹ️  {at_risk:,} records ({at_risk_pct:.1f}%) still exceed 512 tokens —")
print(f"      their tail content will be truncated, but diagnostic signal")
print(f"      (Assessment) is now guaranteed to survive.")
print(f"\n✅ PHASE 3a COMPLETE: Gold layer APSO-flipped and re-registered")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

### 📊 Phase 3a Observations

**Token pressure is confirmed as the structural default for this dataset.** The
APSO-Flip token audit is consistent with Phase 1c's empirical measurement
and the MedSynth paper's reported averages (932 tokens for dialogues, 621 for
notes) — 64.8% of APSO notes exceed the 512-token limit (vs 64.7% for raw Notes
in Phase 1c; the minor difference reflects APSO recomposition joining sections
with double newlines, adding a small number of whitespace tokens).

Three findings from the transformation are worth carrying forward:

1. **Assessment survival is now guaranteed.** With Assessment at Token 0, the
   primary diagnostic signal survives truncation for 100% of records regardless
   of note length. The 6,635 records that exceed the window will lose their
   Subjective and Objective tail content — clinically less critical for a
   classification task — but will retain the coded diagnosis.

2. **Markdown formatting is present in the APSO notes.** The preview shows `**`
   bold markers appearing at section boundaries. These are artefacts of the
   MedSynth Note Writer Agent's markdown output style and will be seen as tokens
   by ClinicalBERT's WordPiece tokeniser. A downstream preprocessing step should
   strip markdown formatting before tokenisation to avoid consuming context window
   space with non-clinical tokens.

3. **ICD-10 codes appear inline in the Assessment text.** The preview shows
   strings like "Pain in the left knee (ICD-10 code M25.562)" directly in the
   Assessment section. This is a known property of the MedSynth generation
   pipeline — the Note Writer Agent embedded the ICD-10 code in the narrative.
   Downstream models trained on this data will have access to the code string
   as a feature, which may inflate classification metrics. This should be
   accounted for in evaluation design.

</div>

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🔍 Phase 3b: ICD-10 Leakage Detection

The Phase 3a APSO note previews confirmed that the MedSynth Note Writer Agent
embedded ICD-10 code strings directly in the clinical narratives, typically
within the Assessment section (e.g., *"Diagnosis: Pain in left knee
(ICD-10: M25.562)"*).

This constitutes data leakage. A model trained on notes containing explicit code
strings can learn to predict labels by pattern-matching the code rather than
understanding the clinical narrative. This would inflate evaluation metrics and
produce a model that fails to generalise to real clinical notes, which never
contain ICD-10 codes in the text.

This phase quantifies the leakage precisely before any redaction is applied.
No records are modified here.

### Detection Objectives

1. **Explicit Code Leakage:** Scan the `apso_note` column for ICD-10 code
   strings using a pattern anchored on the ICD-10 structural convention — a
   leading letter followed by exactly two digits — which correctly discriminates
   ICD-10 codes from common medical abbreviations (MCL, CBC, TSH, HPI) that
   share a superficially similar alphanumeric structure. The pattern covers raw
   format (`M25562`), canonical format (`M25.562`), and placeholder-X codes
   (`T78.1XXA`). All 13 pattern validation tests passed before scanning.

2. **Assessment Column Leakage:** The Assessment section is the primary leakage
   source — the Note Writer Agent placed the diagnosis (including the code) in
   the Assessment section by design. Scanning this column separately establishes
   the scope of the primary leakage vector.

3. **Secondary Section Leakage:** Scan the Subjective, Objective, and Plan
   columns to determine whether leakage extends beyond the Assessment section,
   informing the targeted redaction strategy in Phase 3c.

### Why This Matters

The MedSynth paper notes that the dataset is designed for Dial-2-Note and
Note-2-Dial tasks, where the ICD-10 code is used as a generation conditioning
signal, not as a training label. Using this dataset for ICD-10 classification
without removing the embedded codes would create a trivial pattern-matching
task rather than a genuine clinical reasoning challenge.

### Observed Results

| Section | Leaking records | % of dataset |
|---|---|---|
| Assessment | 2,855 | 27.9% |
| Plan | 238 | 2.3% |
| Objective | 89 | 0.9% |
| Subjective | 60 | 0.6% |
| **Total unique records** | **2,923** | **28.5%** |

The Assessment section accounts for 97.7% of all leaking records, confirming
it as the primary and overwhelmingly dominant leakage vector. 71.5% of records
(7,317) contain no ICD-10 code strings and require no redaction.

> **Result:** A forensic leakage report confirming that 28.5% of records contain
> explicit ICD-10 code strings, concentrated almost entirely in the Assessment
> section. The redaction strategy in Phase 3c is targeted accordingly.

</div>

In [ ]:
# ==============================================================================
# PHASE 3b: LEAKAGE DETECTION (IDENTIFICATION ONLY)
# ==============================================================================
# Purpose: Quantify the extent of ICD-10 code leakage in the APSO notes before
# any redaction is applied. This phase adopts a "validate and confirm" mindset —
# we measure the problem precisely before attempting to fix it.
#
# Background: The MedSynth Note Writer Agent embedded ICD-10 codes directly
# in the clinical narratives (e.g., "Diagnosis: Pain in left knee (ICD-10:
# M25.562)"). A model trained on notes containing explicit code strings can
# learn to predict labels by pattern-matching the code rather than understanding
# the clinical narrative, inflating evaluation metrics and producing a model
# that fails to generalise to real clinical notes.
#
# PATTERN DESIGN NOTE:
# ICD-10 codes always begin with one letter followed by EXACTLY TWO DIGITS.
# This distinguishes them from common medical abbreviations (MCL, ACL, CBC,
# TSH, CMP, LFT) which begin with a letter followed by other letters — not digits.
# Using letter + two-digit anchor eliminates the false positives that arise
# from a purely alphanumeric pattern.
#
# Pattern covers:
#   Raw format (no decimal):  M25562, N390, T781XXA, J189
#   Canonical format:         M25.562, N39.0, T78.1XXA, J18.9
#   3-char parent codes:      J18, R32, N10
#
# This phase is identification only. No records are modified here.
# Redaction is applied in Phase 3c.
# ==============================================================================

import polars as pl
import re

# ==============================================================================
# 1. VERIFY PREREQUISITES
# ==============================================================================
print("🔍 Verifying Phase 3b Prerequisites...")

if 'df_gold' not in locals() or df_gold is None:
    print("❌ CRITICAL: df_gold not found. Please run Phase 3a first.")
    raise ValueError("df_gold not available in local scope")

if 'apso_note' not in df_gold.columns:
    print("❌ CRITICAL: apso_note column not found. Please run Phase 3a (APSO-Flip) first.")
    raise ValueError("apso_note column not present in df_gold")

print(f"   ✅ df_gold available: {len(df_gold):,} records")
print(f"   ✅ apso_note column present")

# ==============================================================================
# 2. DEFINE LEAKAGE DETECTION PATTERN
# ==============================================================================
# Anchored on letter + TWO DIGITS to distinguish ICD-10 codes from medical
# abbreviations. The two-digit anchor is the critical discriminator:
#
#   ✅ Matches:  M25.562  M25562  N39.0  N390  J18  T78.1XXA  T781XXA
#   ❌ Rejects:  MCL  ACL  CBC  TSH  CMP  LFT  HPI  ROS  BMI
#
# Two branches:
#   Branch 1: Canonical (with decimal) — letter + 2 digits + decimal + 1-4 alphanum
#   Branch 2: Raw (no decimal)         — letter + 2 digits + 0-5 additional alphanum
#             (covers 3-char codes like J18 and longer codes like M25562)
# ==============================================================================
# Import the canonical pattern from src/preprocessing.py — single source of truth.
# Defined once, used identically by the training pipeline and inference pipeline.
#from preprocessing import ICD10_REDACT_PATTERN as ICD10_LEAK_PATTERN
from src.preprocessing import ICD10_REDACT_PATTERN as ICD10_LEAK_PATTERN

print(f"\n🔍 Leakage detection pattern (imported from src.preprocessing):")
print(f"   ✅ Catches: M25.562, M25562, N39.0, N390, J18, T78.1XXA")
print(f"   ❌ Rejects: MCL, ACL, CBC, TSH, CMP, LFT, HPI, BMI")

# ==============================================================================
# 3. VALIDATE PATTERN ON KNOWN EXAMPLES
# ==============================================================================
# Quick sanity check before running across the full dataset
print(f"\n🧪 Pattern validation on known examples:")

test_cases = [
    ("M25.562",  True,  "canonical code"),
    ("M25562",   True,  "raw code"),
    ("N39.0",    True,  "canonical code"),
    ("N390",     True,  "raw code"),
    ("J18",      True,  "3-char parent code"),
    ("T78.1XXA", True,  "placeholder-X canonical"),
    ("T781XXA",  True,  "placeholder-X raw"),
    ("MCL",      False, "medical abbreviation"),
    ("ACL",      False, "medical abbreviation"),
    ("CBC",      False, "medical abbreviation"),
    ("TSH",      False, "medical abbreviation"),
    ("BMI",      False, "medical abbreviation"),
    ("HPI",      False, "medical abbreviation"),
]

all_pass = True
for term, expected, label in test_cases:
    matched = bool(re.search(ICD10_LEAK_PATTERN, term))
    status  = "✅" if matched == expected else "❌"
    if matched != expected:
        all_pass = False
    print(f"   {status} '{term:12s}' ({label:28s}) — {'matched' if matched else 'rejected'}")

if all_pass:
    print(f"\n   ✅ All pattern validation tests passed")
else:
    print(f"\n   ⚠️  Some pattern validation tests failed — review pattern before proceeding")

# ==============================================================================
# 4. DETECT EXPLICIT CODE LEAKAGE IN APSO NOTES
# ==============================================================================
print("\n🕵️  Scanning APSO notes for explicit ICD-10 code strings...")

df_explicit_leaks = df_gold.filter(
    pl.col("apso_note").str.contains(ICD10_LEAK_PATTERN)
)

explicit_leak_count = df_explicit_leaks.height
explicit_leak_pct   = explicit_leak_count / len(df_gold) * 100

print(f"   📊 Records with explicit ICD-10 codes in apso_note: "
      f"{explicit_leak_count:,} ({explicit_leak_pct:.1f}%)")

# ==============================================================================
# 5. DETECT LEAKAGE BY SECTION
# ==============================================================================
# Scan each SOAP section independently to understand the distribution of
# leakage and inform the targeted redaction strategy in Phase 3c.
# ==============================================================================
print("\n🕵️  Scanning individual SOAP sections...")

section_leakage = {}
for section in ['assessment', 'subjective', 'objective', 'plan']:
    leaks = df_gold.filter(
        pl.col(section).is_not_null() &
        pl.col(section).str.contains(ICD10_LEAK_PATTERN)
    ).height
    section_leakage[section] = leaks
    pct = leaks / len(df_gold) * 100
    print(f"   📊 {section.capitalize():12s}: {leaks:,} records ({pct:.1f}%)")

# ==============================================================================
# 6. SAMPLE LEAKING RECORDS FOR INSPECTION
# ==============================================================================
print(f"\n👁️  SAMPLE LEAKING RECORDS (First 5 — Assessment column):")

df_assessment_leaks = df_gold.filter(
    pl.col("assessment").is_not_null() &
    pl.col("assessment").str.contains(ICD10_LEAK_PATTERN)
)

if df_assessment_leaks.height > 0:
    sample_leaks = df_assessment_leaks.select([
        "id", "standard_icd10", "assessment"
    ]).head(5)

    for row in sample_leaks.iter_rows(named=True):
        assessment_preview = (row["assessment"] or "")[:200].replace("\n", " ↵ ")
        print(f"\n   ID: {row['id']} | Code: {row['standard_icd10']}")
        print(f"   Assessment: {assessment_preview}...")

# ==============================================================================
# 7. LEAKAGE SUMMARY REPORT
# ==============================================================================
assessment_leak_count = section_leakage['assessment']
assessment_leak_pct   = assessment_leak_count / len(df_gold) * 100

print(f"\n📊 PHASE 3b LEAKAGE DETECTION SUMMARY")
print(f"   " + "─" * 50)
print(f"   📊 Total records scanned:         {len(df_gold):,}")
print(f"   ⚠️  Explicit leaks (apso_note):   {explicit_leak_count:,} ({explicit_leak_pct:.1f}%)")
print(f"   " + "─" * 30)
print(f"   📊 Section breakdown:")
for section, count in section_leakage.items():
    pct  = count / len(df_gold) * 100
    icon = "⚠️ " if count > 0 else "✅"
    print(f"   {icon} {section.capitalize():12s}: {count:>6,} ({pct:.1f}%)")
print(f"   " + "─" * 50)
print(f"   ℹ️  No records modified in this phase.")
print(f"   ℹ️  Redaction will be applied in Phase 3c.")

# ==============================================================================
# 8. AUDIT TRAIL
# ==============================================================================
config.log_event(
    phase="Phase 3b: Leakage Detection",
    action="leakage_detection_complete",
    details={
        "total_records":           len(df_gold),
        "explicit_leak_count":     explicit_leak_count,
        "explicit_leak_pct":       round(explicit_leak_pct, 2),
        "assessment_leak_count":   section_leakage['assessment'],
        "subjective_leak_count":   section_leakage['subjective'],
        "objective_leak_count":    section_leakage['objective'],
        "plan_leak_count":         section_leakage['plan'],
        "detection_pattern":       ICD10_LEAK_PATTERN,
        "pattern_validation":      "passed" if all_pass else "failed",
        "records_modified":        False
    }
)

print(f"\n📝 Audit trail updated")
print(f"✅ PHASE 3b COMPLETE: Leakage quantified — redaction pending (Phase 3c)")

# ==============================================================================
# 9. ZERO-TRUST VERDICT
# ==============================================================================
print(f"\n🛡️  ZERO-TRUST VERDICT:")

if explicit_leak_count == 0:
    print(f"   ✅ No explicit ICD-10 leakage detected")
    print(f"   ✅ Dataset may proceed to training without redaction")
else:
    print(f"   ⚠️  {explicit_leak_count:,} records ({explicit_leak_pct:.1f}%) contain "
          f"explicit ICD-10 code strings")
    print(f"   ⚠️  Training on these notes would allow the model to predict labels")
    print(f"      by code pattern-matching rather than clinical reasoning")
    print(f"   ✅ Redaction strategy to be applied in Phase 3c")
    dominant = max(section_leakage, key=section_leakage.get)
    print(f"   ℹ️  Primary leakage source: {dominant.capitalize()} column "
          f"({section_leakage[dominant]:,} records, "
          f"{section_leakage[dominant]/len(df_gold)*100:.1f}%)")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

### 🔬 Phase 3b.1: Leakage Forensic Observations

The leakage detection results reveal several patterns worth documenting before
proceeding to redaction.

---

#### 1. Assessment Section Dominance

The Assessment section accounts for **97.7%** of all leaking records (2,855 of
2,923). This is expected — the MedSynth Note Writer Agent was designed to place
the diagnosis in the Assessment section, and the prompt explicitly instructed
it to include the ICD-10 code description. The leakage is not a bug; it's a
feature of the generation pipeline that happens to be problematic for
classification tasks.

The remaining leakage in Plan (238), Objective (89), and Subjective (60)
sections likely reflects:
- **Plan:** References to diagnosis codes in treatment rationale or referral notes
- **Objective:** Lab orders or imaging requests that include diagnostic codes
- **Subjective:** Prior diagnosis history mentioned by the patient

---

#### 2. Mismatched Codes in Assessment Text

The sample leaking records reveal an important subtlety: **some Assessment
sections contain ICD-10 codes that differ from the record's assigned label.**

| ID | Assigned Label | Code in Assessment Text |
|----|----------------|-------------------------|
| 2  | M25.562 (Pain in left knee) | M25.551 (Pain in right hip) |
| 3  | M25.562 (Pain in left knee) | M17.12 (Primary osteoarthritis, left knee) |

This occurs because the Note Writer Agent sometimes included:
- **Differential diagnoses** — alternative codes being ruled out
- **Related conditions** — comorbidities or contributing factors
- **More specific codes** — a refinement of the assigned category

This finding has two implications:

1. **Redaction must be pattern-based, not label-matched.** A redaction strategy
   that only removes the assigned label would miss these secondary codes. The
   Phase 3c regex-based approach correctly targets *all* ICD-10 code patterns,
   regardless of whether they match the record's label.

2. **Some "leakage" may actually be clinically relevant.** A differential
   diagnosis code appearing in the Assessment is clinically meaningful — it
   reflects the physician's reasoning process. However, for a classification
   benchmark, any explicit code string (matched or not) provides a shortcut
   that bypasses genuine clinical reasoning, so redaction remains appropriate.

---

#### 3. Clean Majority

**71.5% of records (7,317)** contain no detectable ICD-10 code strings. These
records are already suitable for classification tasks without modification.
The redaction in Phase 3c will bring the remaining 28.5% into alignment,
producing a fully clean dataset.

---

> **Takeaway:** The leakage is concentrated, predictable, and addressable. The
> pattern-based redaction strategy in Phase 3c will neutralise all detected
> leakage — including mismatched codes — without requiring label-specific logic.

</div>

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🕵️ Phase 3b.1: Leakage Forensic Auditor

Before applying redaction in Phase 3c, we deploy an interactive browser-native
auditor to manually verify that the 2,923 records flagged by Phase 3b genuinely
contain explicit ICD-10 code strings — confirming that the detection pattern is
identifying real leakage rather than false positives.

This is a human-in-the-loop validation gate. No records are modified here.

### What to Look For

Navigate the slider across a representative sample of records and confirm that
the APSO note (left panel) visibly contains ICD-10 code strings in one of the
formats the Phase 3b pattern was designed to catch:

- `(ICD-10: M25.562)` — parenthetical with colon
- `(ICD-10 code N39.0)` — parenthetical with label
- `(M25.562)` — bare parenthetical
- `M25.562` — inline without parentheses

Sample across different `code_status` values — `billable`, `noisy_111`, and
`placeholder_x` — to confirm the leakage pattern is consistent across all
three categories. The `code_status` field is shown in the HUD for each record.

The source dialogue (right panel) provides the original doctor-patient
transcript for cross-reference, confirming that the clinical content is rich
and the leakage is confined to the administrative code string rather than
being a structural property of the narrative itself.

> **Validation Gate:** Once satisfied that the detected leaks are genuine,
> proceed to Phase 3c to apply redaction. The auditor should be stopped using
> the browser tab before continuing.

</div>

In [ ]:
# ==============================================================================
# PHASE 3b.1: LEAKAGE FORENSIC AUDITOR (VISUAL INSPECTION)
# ==============================================================================
# Purpose: Interactive browser-native audit of records identified as leaking
# in Phase 3b. Allows manual confirmation that detected leaks are genuine
# ICD-10 code strings before redaction is applied in Phase 3c.
#
# This is a human-in-the-loop validation gate between detection and redaction.
# It operates on df_explicit_leaks produced by Phase 3b.
# ==============================================================================

import panel as pn
pn.extension(design='material')


def launch_leakage_auditor(df_leaking_records):
    """
    Launches a browser-native auditor showing only the records identified
    as leaking in Phase 3b, for manual verification before redaction.
    """
    df_pd = df_leaking_records.to_pandas()

    if len(df_pd) == 0:
        print("✅ No leaks detected — auditor not required.")
        return None

    print(f"🔎 Launching auditor for {len(df_pd):,} leaking records...")

    index_slider = pn.widgets.IntSlider(
        name='Inspect Leaky Record (Index)',
        start=0, end=len(df_pd) - 1, value=0,
        bar_color='#f39c12', sizing_mode='stretch_width'
    )

    @pn.depends(index_slider)
    def get_leakage_view(index):
        row        = df_pd.iloc[index]
        std_code   = row.get('standard_icd10', 'N/A')
        record_id  = row.get('id',             'N/A')
        code_status = row.get('code_status',   'N/A')
        apso_note  = row.get('apso_note',      row.get('note', 'N/A'))
        dialogue   = row.get('dialogue',       'N/A')

        mapping_hud = f"""
        <div style="display: flex; gap: 15px; max-width: 850px;
                    margin-bottom: 20px; font-family: sans-serif;">
            <div style="flex: 1; padding: 12px; background: #fff;
                        border: 3px solid #f39c12; border-radius: 8px;
                        text-align: center;">
                <span style="font-size: 0.75rem; color: #666; font-weight: bold;
                             text-transform: uppercase;">Record ID</span><br>
                <span style="font-size: 1.6rem; color: #333;
                             font-weight: 900;">{record_id}</span>
            </div>
            <div style="flex: 1; padding: 12px; background: #fff;
                        border: 3px solid #2980b9; border-radius: 8px;
                        text-align: center;">
                <span style="font-size: 0.75rem; color: #2980b9; font-weight: bold;
                             text-transform: uppercase;">ICD-10 Label</span><br>
                <span style="font-size: 1.6rem; color: #2980b9;
                             font-weight: 900;">{std_code}</span>
            </div>
            <div style="flex: 1; padding: 12px; background: #f39c12;
                        border-radius: 8px; text-align: center; color: white;">
                <span style="font-size: 0.75rem; font-weight: bold;
                             text-transform: uppercase;">Status</span><br>
                <span style="font-size: 1.1rem; font-weight: 800;">
                    ⚠️ LEAKAGE DETECTED</span>
                <div style="font-size: 0.8rem; margin-top: 5px;">
                    {code_status} &nbsp;|&nbsp;
                    Record {index + 1} of {len(df_pd):,}
                </div>
            </div>
        </div>
        """

        ink_style = """
        <style>
            .forensic-text {
                white-space: pre-wrap;
                font-family: 'Courier New', monospace;
                font-weight: 700;
                font-size: 0.95rem;
                padding: 20px;
                border: 2px solid #333;
                background-color: #fdfdfd;
                line-height: 1.5;
            }
        </style>
        """

        return pn.Column(
            pn.pane.HTML(mapping_hud, sizing_mode='stretch_width'),
            pn.Row(
                pn.Column(
                    "### 🔎 APSO Note (Pre-Redaction — Leakage Visible)",
                    pn.pane.HTML(
                        f'{ink_style}<div class="forensic-text">{apso_note}</div>'
                    ),
                    height=650, scroll=True, sizing_mode='stretch_width'
                ),
                pn.Column(
                    "### 💬 Source Dialogue",
                    pn.pane.Str(
                        dialogue,
                        styles={'white-space': 'pre-wrap', 'font-weight': '500'}
                    ),
                    height=650, scroll=True, sizing_mode='stretch_width'
                )
            )
        )

    layout = pn.Column(
        "# 🕵️ Leakage Forensic Auditor — Phase 3b.1",
        f"**Objective:** Visually confirm that the {len(df_pd):,} records "
        "identified by Phase 3b genuinely contain ICD-10 code strings "
        "before redaction is applied in Phase 3c. "
        "Look for patterns like `(ICD-10: M25.562)`, `(ICD-10 code N39.0)`, "
        "or bare codes like `M25.562` in the Assessment section.",
        index_slider,
        get_leakage_view,
        sizing_mode='stretch_width'
    )

    return layout.show(title="Leakage Forensic Auditor — Phase 3b.1", threaded=True)


# ==============================================================================
# LAUNCH AUDITOR
# Uses df_explicit_leaks produced by Phase 3b
# ==============================================================================
if 'df_explicit_leaks' not in locals() or df_explicit_leaks is None:
    print("❌ df_explicit_leaks not found. Please run Phase 3b first.")
else:
    leak_server = launch_leakage_auditor(df_explicit_leaks)
    print("\n💡 TIP: Navigate the slider to sample different leaking records.")
    print("   Confirm that code strings are genuinely present in the APSO note.")
    print("   When satisfied, proceed to Phase 3c for redaction.")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🔬 Phase 3b.2: ICD-10 String Format Corpus Audit

Before applying redaction in Phase 3c, this cell catalogues every distinct
`(ICD-10...)` parenthetical format present across all 10,240 raw notes and
tests our three-pass regex against each one.

This audit was introduced after discovering that the original narrow pattern
missed 301 distinct formats (454 occurrences) including `(ICD-10 Code: ...)`,
`(ICD-10)`, `(ICD-10 description)`, `(ICD-10-CM ...)`, and reversed formats
like `(CODE ICD-10)`.

The broad pattern `(?i)\s*\([^)]*\bICD[^)]*\)` was validated against all
1,739 distinct formats found in the corpus. The only expected miss is `(AICD)`
— Automated Implantable Cardioverter Defibrillator, a legitimate medical device
abbreviation correctly excluded by the `\b` word boundary.

**Expected result:** 1,738 caught / 1 missed (`(AICD)` — correct behaviour).

</div>

In [ ]:
# ==============================================================================
# PHASE 3b.2: ICD-10 STRING FORMAT AUDIT
# ==============================================================================
# Before applying redaction, catalogue EVERY distinct (ICD-10...) string
# format present in the corpus. Tests our regex against all of them.
# No modifications made — purely diagnostic.
# ==============================================================================
import re
from collections import Counter
import importlib
import src.preprocessing
importlib.reload(src.preprocessing)

# Import patterns
from src.preprocessing import (
    ICD10_REDACT_PATTERN,
    PARENTHETICAL_ICD10_PATTERN,
    REDACTED_ARTIFACT_PATTERN,
    REDACTION_MARKER,
)

print(f"PARENTHETICAL_ICD10_PATTERN = {PARENTHETICAL_ICD10_PATTERN!r}")
print("\U0001f52c ICD-10 STRING FORMAT AUDIT")
print("=" * 70)
print("Cataloguing every distinct (ICD-10...) format across all 10,240 notes")
print()

# Step 1: find ALL strings matching a broad "anything containing ICD"
# Use df_raw Note column — guaranteed pre-redaction regardless of run order
BROAD_ICD_PATTERN = re.compile(r'\([^)]*ICD[^)]*\)', re.IGNORECASE)

all_icd_strings = Counter()
notes = df_raw.select("Note").to_series().to_list()

for note in notes:
    if note:
        for match in BROAD_ICD_PATTERN.findall(note):
            normalised = re.sub(r'\s+', ' ', match.strip())
            all_icd_strings[normalised] += 1

print(f"\U0001f4ca Total distinct (ICD...) patterns found: {len(all_icd_strings):,}")
print(f"\U0001f4ca Total occurrences across corpus:        {sum(all_icd_strings.values()):,}")
print()

# Step 2: test each pattern against our 3-pass regex
print("=" * 70)
print("TESTING EACH FORMAT AGAINST CURRENT REGEX (3-pass redaction)")
print("=" * 70)
print()

caught = Counter()
missed = Counter()

for pattern_str, count in sorted(all_icd_strings.items(), key=lambda x: -x[1]):
    result = re.sub(PARENTHETICAL_ICD10_PATTERN, '', pattern_str)
    result = re.sub(ICD10_REDACT_PATTERN, REDACTION_MARKER, result)
    result = re.sub(REDACTED_ARTIFACT_PATTERN, '', result)
    result = result.strip()

    still_has_code     = bool(re.search(r'[A-Z][0-9]{2}', result))
    still_has_icd      = 'ICD' in result.upper()
    still_has_artifact = '[REDACTED]' in result
    is_clean = not still_has_code and not still_has_icd and not still_has_artifact

    if is_clean:
        caught[pattern_str] = count
    else:
        missed[pattern_str] = count
        print(f"\u274c MISSED ({count:,}x): {pattern_str!r}")
        print(f"   After redaction: {result!r}")
        print()

print("=" * 70)
print(f"\u2705 Caught (fully redacted): {len(caught):,} distinct formats ({sum(caught.values()):,} occurrences)")
print(f"\u274c Missed (need fix):       {len(missed):,} distinct formats ({sum(missed.values()):,} occurrences)")
print()

# Step 3: top 20 most common formats
print("=" * 70)
print("TOP 20 MOST COMMON (ICD-10...) FORMATS IN CORPUS")
print("=" * 70)
for pattern_str, count in all_icd_strings.most_common(20):
    status = "\u2705" if pattern_str in caught else "\u274c"
    print(f"  {status} {count:>5,}x  {pattern_str!r}")

print()
print("\u2705 Audit complete — expected: 1,738 caught / 1 missed (AICD only)")


<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🔒 Phase 3c: ICD-10 Code Redaction

Having quantified the leakage in Phase 3b — 2,923 records (28.5%) containing
explicit ICD-10 code strings, concentrated in the Assessment section — we now
apply targeted redaction to eliminate the leakage before any downstream model
training occurs.

### Redaction Strategy

A three-pass approach removes ICD-10 code strings without leaving artefacts,
validated against a full corpus audit of 1,739 distinct `(ICD-10...)` formats
found in the 10,240 notes (Phase 3b.2):

**Pass 1 — Parenthetical removal:** Any parenthetical containing the word `ICD`
is removed entirely using a broad pattern (`(?i)\s*\([^)]*\bICD[^)]*\)`).
This covers all observed MedSynth variants — `(ICD-10: M25.562)`,
`(ICD-10 Code: M25.562)`, `(ICD-10 code M25.562)`, `(ICD-10)`,
`(ICD-10 description)`, `(ICD-10-CM N39.0)`, reversed formats, and more.
A word boundary (`\b`) prevents matching `(AICD)` — a legitimate medical
device abbreviation.

**Pass 2 — Standalone code redaction:** Any remaining bare ICD-10 code strings
(e.g. `M25.562` or `M25562` inline without parentheses) are replaced with
`[REDACTED]`.

**Pass 3 — Artefact cleanup:** Any stray `([REDACTED])` left when Pass 2 fired
inside a parenthetical that Pass 1 didn't match (e.g. bare `(M25.551)`) is
removed entirely.

> *"Diagnosis: Pain in left knee (ICD-10: M25.562)"*
> becomes:
> *"Diagnosis: Pain in left knee"*

> *"Primary Diagnosis: Pain in the right hip (M25.551)"*
> becomes:
> *"Primary Diagnosis: Pain in the right hip"*

### Scope

All four SOAP section columns are redacted (`assessment`, `plan`, `objective`,
`subjective`), then `apso_note` is rebuilt from the cleaned sections. This
ensures `apso_note` is the single source of truth for downstream model input —
fully cleaned, Assessment-first, zero code strings.

The original `note` column is **not modified**. It is preserved in the Gold
layer for audit traceability and for tasks where the original unredacted text
is needed.

### What Is and Isn't Changed

| Column | Action |
|---|---|
| `assessment`, `plan`, `objective`, `subjective` | ICD-10 strings replaced with `[REDACTED]` |
| `apso_note` | Rebuilt from redacted sections |
| `note` | **Preserved unchanged** — original SOAP text |
| `standard_icd10`, `label`, `code_status` | **Preserved unchanged** — labels intact |
| All other columns | **Preserved unchanged** |

No records are removed. The labels (`standard_icd10`, `label`) are not touched —
redaction targets only the text fields where code strings appear as leakage.

### Post-Redaction Verification

The same detection pattern from Phase 3b is re-applied to the cleaned
`apso_note` and all section columns after redaction. The expected result is
0 remaining leaks. Any residual leaks would indicate code formats not covered
by the pattern and would block progression to downstream training.

> **Result:** A fully redacted Gold layer where `apso_note` contains zero
> explicit ICD-10 code strings — forcing any downstream model to learn from
> clinical language rather than label pattern-matching.

</div>

In [ ]:
# ==============================================================================
# PHASE 3c: ICD-10 CODE REDACTION
# ==============================================================================
import polars as pl

# 1. VERIFY PREREQUISITES
print("🔍 Verifying Phase 3c Prerequisites...")

if 'df_gold' not in locals() or df_gold is None:
    raise ValueError("df_gold not available — run Phase 3b first.")

required_cols = {'apso_note', 'assessment', 'plan', 'subjective', 'objective'}
missing_cols = required_cols - set(df_gold.columns)
if missing_cols:
    raise ValueError(f"❌ df_gold missing columns: {missing_cols}")

if 'explicit_leak_count' not in locals():
    explicit_leak_count = None

print(f"   ✅ df_gold available: {len(df_gold):,} records")
print(f"   ✅ All required SOAP columns present")

# 2. IMPORT PATTERNS — three-pass strategy
from src.preprocessing import (
    ICD10_REDACT_PATTERN,
    PARENTHETICAL_ICD10_PATTERN,
    REDACTED_ARTIFACT_PATTERN,
    REDACTION_MARKER,
)

print(f"\n🔧 Redaction configuration (imported from src.preprocessing):")
print(f"   Pass 1 — Parenthetical pattern: {PARENTHETICAL_ICD10_PATTERN}")
print(f"   Pass 2 — Code pattern:          {ICD10_REDACT_PATTERN}")
print(f"   Pass 3 — Artifact pattern:      {REDACTED_ARTIFACT_PATTERN}")
print(f"   Replacement: {REDACTION_MARKER}")

# 3. REDACT SOAP SECTION COLUMNS
print("\n🔄 Applying redaction to SOAP section columns...")

SECTIONS_TO_REDACT = ['assessment', 'plan', 'objective', 'subjective']

redaction_exprs = [
    pl.col(section)
    .str.replace_all(PARENTHETICAL_ICD10_PATTERN, '')      # Pass 1: remove (ICD-10: CODE)
    .str.replace_all(ICD10_REDACT_PATTERN, REDACTION_MARKER)  # Pass 2: redact stray codes
    .str.replace_all(REDACTED_ARTIFACT_PATTERN, '')         # Pass 3: clean ([REDACTED])
    .alias(f"{section}_clean")
    for section in SECTIONS_TO_REDACT
]

df_gold = df_gold.with_columns(redaction_exprs)

print(f"\n   📊 Redaction counts per section:")
for section in SECTIONS_TO_REDACT:
    changed = df_gold.filter(
        pl.col(section).is_not_null() &
        (pl.col(f"{section}_clean") != pl.col(section))
    ).height
    pct = changed / len(df_gold) * 100
    print(f"   ✅ {section.capitalize():12s}: {changed:,} records redacted ({pct:.1f}%)")

# 4. REPLACE ORIGINALS
print("\n🔄 Replacing original section columns with redacted versions...")
df_gold = df_gold.with_columns([
    pl.col(f"{section}_clean").alias(section) for section in SECTIONS_TO_REDACT
]).drop([f"{section}_clean" for section in SECTIONS_TO_REDACT])
print(f"   ✅ Original section columns replaced with redacted versions")

# 5. REBUILD APSO NOTE
print("\n🔄 Rebuilding apso_note from cleaned sections...")
df_gold = df_gold.with_columns([
    (
        pl.col("assessment").fill_null("") + "\n\n" +
        pl.col("plan").fill_null("")       + "\n\n" +
        pl.col("subjective").fill_null("") + "\n\n" +
        pl.col("objective").fill_null("")
    ).str.strip_chars().alias("apso_note")
])
print(f"   ✅ apso_note rebuilt from redacted sections")

# 6. POST-REDACTION VERIFICATION
print("\n🔍 Post-redaction verification...")

remaining_leaks = df_gold.filter(
    pl.col("apso_note").str.contains(ICD10_REDACT_PATTERN)
).height

remaining_by_section = {}
for section in SECTIONS_TO_REDACT:
    remaining_by_section[section] = df_gold.filter(
        pl.col(section).is_not_null() &
        pl.col(section).str.contains(ICD10_REDACT_PATTERN)
    ).height

print(f"\n📊 POST-REDACTION LEAK CHECK:")
print(f"   " + "─" * 50)
if explicit_leak_count is not None:
    print(f"   📊 Pre-redaction leaks (apso_note):  {explicit_leak_count:,}")
icon = "✅" if remaining_leaks == 0 else "❌"
print(f"   {icon} Post-redaction leaks (apso_note): {remaining_leaks:,}")
all_clean = True
for section, remaining in remaining_by_section.items():
    icon = "✅" if remaining == 0 else "❌"
    if remaining > 0:
        all_clean = False
    print(f"   {icon} {section.capitalize():12s}: {remaining:,} remaining leaks")
print(f"   " + "─" * 50)
if all_clean and remaining_leaks == 0:
    print(f"   ✅ REDACTION COMPLETE: 0 ICD-10 code strings remain")
else:
    print(f"   ❌ WARNING: {remaining_leaks:,} leaks remain after redaction")

# 7. PREVIEW
print(f"\n👁️  APSO Note Preview (First 3 Records — post-redaction):")
for record_id in ["0", "1", "2"]:
    row = df_gold.filter(pl.col("id") == record_id)
    if row.height > 0:
        apso_preview = (row["apso_note"][0] or "")[:300].replace("\n", " ↵ ")
        code = row["standard_icd10"][0]
        print(f"\n   ID: {record_id} | Code: {code}")
        print(f"   {apso_preview}...")

# 8. REGISTER
print(f"\n💾 Re-registering redacted Gold layer...")
config.register_dataframe("gold_medsynth", df_gold, phase="Phase 3c")
print(f"   ✅ gold_medsynth updated: {len(df_gold):,} records, {len(df_gold.columns)} columns")
print(f"   ✅ Original 'note' column preserved (unredacted) for audit traceability")

# 9. AUDIT TRAIL
config.log_event(
    phase="Phase 3c: ICD-10 Redaction",
    action="redaction_complete",
    details={
        "total_records":           len(df_gold),
        "pre_redaction_leaks":     explicit_leak_count if explicit_leak_count else "unknown",
        "post_redaction_leaks":    remaining_leaks,
        "redaction_successful":    remaining_leaks == 0,
        "redaction_marker":        REDACTION_MARKER,
        "sections_redacted":       SECTIONS_TO_REDACT,
        "apso_note_rebuilt":       True,
        "original_note_preserved": True,
        "section_remaining_leaks": remaining_by_section,
        "three_pass_redaction":    True,
    }
)
print(f"\n📝 Audit trail updated")

# 10. ZERO-TRUST VERDICT
print(f"\n🛡️  ZERO-TRUST VERDICT:")
if remaining_leaks == 0:
    print(f"   ✅ 0 ICD-10 code strings remain in apso_note")
    print(f"   ✅ 0 ICD-10 code strings remain in any SOAP section column")
    print(f"   ✅ apso_note rebuilt from redacted sections")
    print(f"   ✅ Original SOAP note preserved in 'note' column")
    print(f"   ✅ Gold layer ready for downstream model training")
else:
    print(f"   ❌ {remaining_leaks:,} ICD-10 code strings remain")
    print(f"   ❌ Review redaction pattern and re-run before proceeding")

print(f"\n✅ PHASE 3c COMPLETE: ICD-10 codes redacted from all SOAP sections")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 📖 Interpretation of Results

This section synthesises the findings from the full ingestion, validation, and
preparation pipeline. It is intended as a standing reference before any
downstream modelling phase begins.

---

### 1. The Dataset

MedSynth is a fully synthetic dataset of 10,240 doctor-patient dialogue and
clinical note pairs, generated by a four-agent GPT-4o pipeline informed by
real-world disease distributions from 800 million IQVIA insurance claims.
Notes follow strict SOAP format; dialogues are derived from notes by a
separate generation pipeline.

The dataset was engineered with **uniform sampling** — exactly 5 records per
ICD-10 code — deliberately preventing domination by common diseases. This
produces a perfectly balanced label distribution across 2,037 distinct codes
(the MedSynth paper reports 2,001 codes; the additional 36 codes reflect
post-publication additions to the HuggingFace release), with a minimum
frequency of 5 per code and zero rare labels. This is an unusual property
that should be explicitly acknowledged in any evaluation design: models
trained on this data will not learn to handle real-world class imbalance.

---

### 2. ICD-10 Label Quality

Validation against the CDC FY2026 reference confirmed the following
classification across all 10,240 records:

| Status | Records | % | Interpretation |
|---|---|---|---|
| `billable` | 9,660 | 94.3% | Leaf-level codes confirmed in CDC tabular reference |
| `noisy_111` | 60 | 0.6% | Valid parent codes reflecting real billing practice |
| `placeholder_x` | 25 | 0.24% | Legitimate injury codes absent from CM descriptions file |

The 0.6% Noisy 111 rate is not a data quality problem. The MedSynth authors
selected from the top 2,001 most frequent ICD-10 codes in real insurance
claims, and clinicians genuinely submit category-level codes in practice.
These records are retained in the Gold layer with a `code_status` annotation.

The 25 placeholder-X codes (e.g. `T78.1XXA`) are structurally valid ICD-10-CM
injury codes using the mandatory placeholder X convention. Their absence from
the CDC descriptions file is a known gap in the tabular reference, not a
dataset error.

---

### 3. Token Pressure

Both text fields exceed ClinicalBERT's 512-token context window on average,
consistent with the MedSynth paper's own reported statistics:

| Field | Paper avg (tokens) | Records exceeding 512 |
|---|---|---|
| Note | 621 | 64.7% |
| Dialogue | 932 | 100.0% |

The APSO-Flip transformation (Phase 3a) addresses this by reordering the note
to place the Assessment section — which contains the primary diagnostic signal
— at Token 0. After the flip, 64.8% of notes still exceed the window, but the
content that survives truncation is now the diagnostically critical content
rather than the Subjective section preamble.

The `words × 1.3` token heuristic used in Phase 1c is conservative for
ClinicalBERT specifically, which uses WordPiece tokenisation and tends to
produce more tokens per word than the BPE tokenisers used to report counts in
the MedSynth paper. True truncation rates for ClinicalBERT are likely higher
than reported here.

---

### 4. SOAP Extraction

The Phase 1e Pydantic Gatekeeper achieved 100% extraction success across all
four SOAP sections. This result is expected for a synthetically generated
dataset — the MedSynth Note Writer Agent was explicitly instructed to produce
SOAP-formatted notes, and the Note Polisher Agent enforced correct section
placement. This extraction rate should not be assumed to generalise to real
EHR notes.

The Subjective section varies stochastically across records: some include
CC/HPI/ROS sub-sections, others do not. This is by design — the Note Writer
Agent prompt instructed the model to "roll a dice" to determine sub-section
inclusion.

---

### 5. ICD-10 Code Leakage

The MedSynth Note Writer Agent embedded ICD-10 code strings directly in the
Assessment section of 27.9% of records (2,855 records). This constitutes
data leakage for any ICD-10 classification task — a model trained on unredacted
notes can predict labels by pattern-matching the code string rather than
reasoning from clinical language.

Phase 3b quantified the leakage precisely:

| Section | Leaking records | % |
|---|---|---|
| Assessment | 2,855 | 27.9% |
| Plan | 238 | 2.3% |
| Objective | 89 | 0.9% |
| Subjective | 60 | 0.6% |
| **Total unique** | **2,923** | **28.5%** |

Note: Section percentages sum to more than 28.5% because some records contain
leakage in multiple sections.

Phase 3c redacted all explicit code strings using a pattern anchored on the
ICD-10 structural convention (letter + two digits), replacing them with
`[REDACTED]` to preserve sentence structure. Post-redaction verification
confirmed 0 remaining leaks across all sections.

The original `note` column is preserved unredacted for tasks where the
original text is required.

---

### 6. What the Gold Layer Provides

The final Gold layer (`df_gold` / `gold_medsynth`) contains 10,240 records
and 13 columns, providing multiple representations of each clinical encounter:

| Column | Description |
|---|---|
| `id` | Unique string identifier for audit traceability |
| `note` | Original SOAP note (unmodified, unredacted) |
| `dialogue` | Doctor-patient conversation transcript |
| `label` | Raw ICD-10 code array (decimal-free, as stored in source) |
| `raw_code` | First code from label array (decimal-free) |
| `standard_icd10` | Canonical decimal-restored ICD-10 code (e.g. `M25.562`) |
| `code_status` | CDC-grounded classification: `billable`, `noisy_111`, or `placeholder_x` |
| `subjective` | Extracted Subjective section (redacted) |
| `objective` | Extracted Objective section (redacted) |
| `assessment` | Extracted Assessment section (redacted) |
| `plan` | Extracted Plan section (redacted) |
| `apso_note` | APSO-recomposed note (Assessment-first, redacted) |
| `apso_token_estimate` | Estimated token count for apso_note |

---

### 7. Recommended Downstream Configurations

Downstream phases should select records and text fields based on their specific
modelling objective:

```python
# ==============================================================================
# CONFIGURATION 1: Strict Classification (Billable Codes Only)
# ==============================================================================
# Use when: Training an ICD-10 classifier for production billing systems
# Rationale: Only leaf-level codes are accepted for reimbursement
df_strict = df_gold.filter(pl.col("code_status") == "billable")  # 9,660 records

# ==============================================================================
# CONFIGURATION 2: Maximum Data (All Records)
# ==============================================================================
# Use when: Pre-training, representation learning, or research experiments
# Rationale: Maximise training signal; handle parent codes via label hierarchy
df_all = df_gold  # 10,240 records

# ==============================================================================
# CONFIGURATION 3: Exclude Placeholder-X Only
# ==============================================================================
# Use when: Parent codes are acceptable but placeholder-X codes are not
# Rationale: Placeholder-X codes have no CDC description for label embedding
df_no_placeholder = df_gold.filter(
    pl.col("code_status") != "placeholder_x"
)  # 10,215 records

# ==============================================================================
# TEXT FIELD SELECTION
# ==============================================================================
# For ICD-10 classification:
#   Use 'apso_note' — Assessment-first, redacted, truncation-safe
#
# For Dial-2-Note or Note-2-Dial generation:
#   Use 'note' (original SOAP) and 'dialogue' — leakage is acceptable
#   when the ICD-10 code is a conditioning signal, not a prediction target
#
# For section-level analysis:
#   Use individual columns: 'assessment', 'plan', 'subjective', 'objective'
```

---

### 8. Limitations and Caveats

The following limitations should be acknowledged in any downstream evaluation:

1. **Synthetic origin**: All clinical content was generated by GPT-4o, not
   extracted from real EHR systems. Linguistic patterns, abbreviation usage,
   and clinical reasoning may not generalise to real clinical notes.

2. **Uniform label distribution**: The deliberate 5-per-code sampling produces
   a perfectly balanced dataset. Models trained on this data will not learn
   to handle the extreme class imbalance present in real ICD-10 distributions.

3. **100% SOAP extraction**: The perfect extraction rate reflects controlled
   synthetic generation, not real-world formatting variability. Pipelines
   built on this data may fail on real clinical notes with non-standard
   section headers or missing sections.

4. **Redaction marker tokens**: The `[REDACTED]` markers will be tokenised
   by downstream models. Consider whether to strip these markers entirely
   or leave them as explicit signals that content was removed.

5. **Token estimates are heuristic**: The `words × 1.3` multiplier is
   approximate. For precise truncation analysis, compute actual token counts
   using the target model's tokeniser.

---

> **Summary**: The Gold layer provides a rigorously validated, fully annotated,
> leakage-neutralised dataset ready for downstream ICD-10 classification
> experiments. Every design decision is documented, every record is traceable
> to its source, and no data has been irreversibly discarded.



</div>

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🛡️ Phase 3c.1: Post-Redaction Integrity Auditor

Following Phase 3c's automated verification of 0 remaining leaks, this
browser-native auditor provides human confirmation that the redaction is
visually correct before the Gold layer is committed for downstream use.

### What to Confirm

Navigate across a representative sample of records and verify:

1. **Redaction markers are absent** — ICD-10 code strings and the parentheticals that
   contained them should be gone entirely. The HUD shows how many redactions
   fired for each record. Records with 0 redactions had no code strings to begin with.

2. **Clinical language is intact** — the text surrounding each removed code
   should read naturally (e.g. *"Pain in the left knee"* not
   *"Pain in the left knee (ICD-10: [REDACTED])"*).
   The clinical description of the diagnosis should still be present.

3. **APSO ordering is preserved** — Assessment content should appear at the
   top of the note (green panel), followed by Plan, then Subjective, then
   Objective.

4. **No visible code strings remain** — scan the full note for any `X##.###`
   or `X##X##` patterns that may have been missed.

Sample across `billable`, `noisy_111`, and `placeholder_x` records using the
`code_status` shown in the HUD. The soft green background of the note panel
distinguishes this post-redaction view from the pre-redaction amber view in
Phase 3b.1.

> **Final Gate:** Once satisfied, the Gold layer is ready for downstream model
> training. Close the auditor before proceeding.

</div>

In [ ]:
# ==============================================================================
# PHASE 3c.1: POST-REDACTION INTEGRITY AUDITOR
# ==============================================================================
# Purpose: Visual confirmation that Phase 3c redaction is correct on a
# record-by-record basis. Complements the automated 0-leak verification
# by allowing human inspection of the redacted apso_note alongside the
# original dialogue.
#
# What to confirm:
#   1. [REDACTED] markers appear where ICD-10 codes were
#   2. Surrounding clinical language is intact and readable
#   3. Assessment section appears at the top (APSO ordering preserved)
#   4. No code strings are visually detectable in the note
# ==============================================================================

import panel as pn
pn.extension(design='material')


def launch_sanitization_auditor(df_redacted):
    df_pd = df_redacted.to_pandas()

    index_slider = pn.widgets.IntSlider(
        name='Verify Redacted Record (Index)',
        start=0, end=len(df_pd) - 1, value=0,
        bar_color='#27ae60', sizing_mode='stretch_width'
    )

    @pn.depends(index_slider)
    def get_audit_view(index):
        row           = df_pd.iloc[index]
        standard_label = row.get('standard_icd10',      'N/A')
        code_status    = row.get('code_status',          'N/A')
        apso_note      = row.get('apso_note',            '⚠️ apso_note not found.')
        dialogue       = row.get('dialogue',             '⚠️ dialogue not found.')
        token_est      = row.get('apso_token_estimate',  0)

        # Check for any remaining [REDACTED] markers — confirms redaction fired
        redacted_count = apso_note.count('[REDACTED]') if isinstance(apso_note, str) else 0
        redact_label   = (f"✅ {redacted_count} code(s) redacted"
                          if redacted_count > 0
                          else "ℹ️  No codes present in this record")

        mapping_hud = f"""
        <div style="display: flex; gap: 20px; max-width: 850px;
                    margin-bottom: 20px; font-family: sans-serif;">
            <div style="flex: 1; padding: 15px; background: #fff;
                        border: 2px solid #27ae60; border-radius: 8px;
                        text-align: center;">
                <span style="font-size: 0.8rem; color: #27ae60; font-weight: bold;
                             text-transform: uppercase;">Gold Label</span><br>
                <span style="font-size: 1.8rem; color: #27ae60;
                             font-weight: 900;">{standard_label}</span><br>
                <span style="font-size: 0.8rem; color: #666;">{code_status}</span>
            </div>
            <div style="flex: 1; padding: 15px; background: #2c3e50;
                        border-radius: 8px; text-align: center; color: white;">
                <span style="font-size: 0.8rem; font-weight: bold;
                             text-transform: uppercase;">Redaction Status</span><br>
                <span style="font-size: 1.0rem; font-weight: 700;">
                    🛡️ ZERO-LEAK VERIFIED</span><br>
                <span style="font-size: 0.85rem; margin-top: 4px;
                             display: block;">{redact_label}</span>
            </div>
            <div style="flex: 1; padding: 15px; background: #f8f9fa;
                        border: 2px solid #ddd; border-radius: 8px;
                        text-align: center;">
                <span style="font-size: 0.8rem; color: #666; font-weight: bold;
                             text-transform: uppercase;">Est. Tokens</span><br>
                <span style="font-size: 1.8rem; color: #333;
                             font-weight: 900;">{int(token_est):,}</span>
            </div>
        </div>
        """

        ink_style = """
        <style>
            .sanitized-text {
                white-space: pre-wrap !important;
                font-family: 'Courier New', monospace !important;
                font-size: 1.0rem !important;
                font-weight: 700 !important;
                color: #000 !important;
                background-color: #e8f8f5 !important;
                padding: 20px;
                border: 2px solid #27ae60;
                line-height: 1.5;
            }
        </style>
        """

        return pn.Column(
            pn.pane.HTML(mapping_hud, sizing_mode='stretch_width'),
            pn.Row(
                pn.Column(
                    "### 🛡️ Redacted APSO Note (Training View)",
                    pn.pane.HTML(
                        f'{ink_style}<div class="sanitized-text">{apso_note}</div>'
                    ),
                    height=650, scroll=True, sizing_mode='stretch_width'
                ),
                pn.Column(
                    "### 💬 Original Dialogue (Ground Truth)",
                    pn.pane.Str(
                        dialogue,
                        styles={'white-space': 'pre-wrap', 'font-weight': '500'}
                    ),
                    height=650, scroll=True, sizing_mode='stretch_width'
                )
            )
        )

    layout = pn.Column(
        "# 🛡️ Post-Redaction Integrity Auditor — Phase 3c.1",
        f"**Objective:** Visually confirm that [REDACTED] markers appear correctly "
        f"and no ICD-10 code strings remain visible. Check that Assessment content "
        f"appears first (APSO ordering) and surrounding clinical language is intact. "
        f"Showing {len(df_pd):,} records.",
        index_slider,
        get_audit_view,
        sizing_mode='stretch_width'
    )

    return layout.show(title="Post-Redaction Integrity Auditor — Phase 3c.1",
                       threaded=True)


# ==============================================================================
# LAUNCH AUDITOR
# Uses df_gold produced by Phase 3c
# ==============================================================================
if 'df_gold' not in locals() or df_gold is None:
    print("❌ df_gold not found. Please run Phase 3c first.")
else:
    print("🚀 Launching Post-Redaction Integrity Auditor (Phase 3c.1)...")
    print("   💡 TIP: Navigate the slider to sample redacted records.")
    print("   💡 TIP: Look for [REDACTED] markers in the Assessment section.")
    print("   💡 TIP: Confirm surrounding clinical language is intact.")
    print("   💡 TIP: Sample across billable, noisy_111, and placeholder_x records.")
    final_audit_server = launch_sanitization_auditor(df_gold)

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 💾 Phase 4: Gold Layer Parquet Export

This is the final checkpoint of the preprocessing pipeline. We persist the
fully prepared Gold layer to a Parquet file, transitioning from an active
transformation pipeline to a fixed, versioned artifact ready for downstream
model training or analysis.

### Export Strategy

1. **Parquet Format:** Parquet preserves the strict Polars schema — string
   `id`, `List(String)` labels, `Float64` token estimates — ensuring type
   consistency when the artifact is loaded in a downstream notebook without
   needing to re-run the full preprocessing pipeline.

2. **Snappy Compression:** Balances disk footprint with fast read/write
   performance, appropriate for iterative training workflows.

3. **Immutable Artifact:** This export freezes all transformations applied
   throughout the pipeline — APSO reordering, decimal restoration, status
   annotation, and ICD-10 redaction. The `apso_note` and `standard_icd10`
   columns in this file are the canonical training inputs.

4. **Audit Closure:** The export event is logged to the audit trail with
   full metadata — path, row count, column set, and pipeline completion
   timestamp — closing the preprocessing audit record.

> **Result:** A versioned, schema-consistent Parquet artifact containing all
> 10,240 annotated Gold layer records, ready for downstream use without
> re-running any preprocessing step.

</div>

In [ ]:
# ==============================================================================
# PHASE 4: GOLD LAYER PARQUET EXPORT
# ==============================================================================

import polars as pl
from datetime import datetime

# ==============================================================================
# 1. VERIFY PREREQUISITES
# ==============================================================================
print("🔍 Verifying Phase 4 Prerequisites...")

if 'df_gold' not in locals() or df_gold is None:
    print("❌ CRITICAL: df_gold not found. Please run Phase 3c first.")
    raise ValueError("df_gold not available in local scope")

print(f"   ✅ df_gold available: {len(df_gold):,} records, {len(df_gold.columns)} columns")

# Confirm redaction is complete before exporting
icd10_check = df_gold.filter(
    pl.col("apso_note").str.contains(
        r'\b[A-Z][0-9]{2}\.[0-9A-Z]{1,4}\b|\b[A-Z][0-9]{2}[0-9A-Z]{0,5}\b'
    )
).height

if icd10_check > 0:
    print(f"   ❌ WARNING: {icd10_check:,} records still contain ICD-10 code strings.")
    print(f"      Please run Phase 3c before exporting.")
    raise ValueError("Redaction incomplete — do not export until Phase 3c is complete.")
else:
    print(f"   ✅ Redaction verified: 0 ICD-10 code strings in apso_note")

# ==============================================================================
# 2. DEFINE EXPORT PATH
# ==============================================================================
gold_dir    = config.resolve_path("data", "gold")
timestamp   = datetime.now().strftime("%Y%m%d_%H%M%S")
export_path = gold_dir / f"medsynth_gold_apso_{timestamp}.parquet"

print(f"\n💾 Export path: {export_path}")

# ==============================================================================
# 3. WRITE PARQUET
# ==============================================================================
df_gold.write_parquet(export_path, compression="snappy")

file_size_mb = export_path.stat().st_size / (1024 * 1024)
print(f"   ✅ Parquet written: {file_size_mb:.1f} MB")

# ==============================================================================
# 4. VERIFY THE WRITTEN FILE
# ==============================================================================
# Load back and confirm row count and schema match
df_verify = pl.read_parquet(export_path)

if df_verify.height != len(df_gold) or df_verify.columns != df_gold.columns:
    print(f"   ❌ WARNING: Written file does not match source DataFrame.")
    print(f"      Source: {len(df_gold):,} rows, {df_gold.columns}")
    print(f"      Written: {df_verify.height:,} rows, {df_verify.columns}")
    raise ValueError("Parquet export verification failed.")
else:
    print(f"   ✅ Verification passed: {df_verify.height:,} rows, "
          f"{len(df_verify.columns)} columns")

# ==============================================================================
# 5. AUDIT TRAIL
# ==============================================================================
config.log_event(
    phase="Phase 4: Parquet Export",
    action="gold_parquet_exported",
    details={
        "export_path":        str(export_path),
        "file_size_mb":       round(file_size_mb, 2),
        "total_rows":         len(df_gold),
        "total_columns":      len(df_gold.columns),
        "columns":            df_gold.columns,
        "billable_count":     df_gold.filter(pl.col("code_status") == "billable").height,
        "noisy_111_count":    df_gold.filter(pl.col("code_status") == "noisy_111").height,
        "placeholder_x_count": df_gold.filter(pl.col("code_status") == "placeholder_x").height,
        "icd10_leaks_verified": 0,
        "compression":        "snappy",
        "primary_text_col":   "apso_note",
        "label_col":          "standard_icd10",
        "status_col":         "code_status"
    }
)

print(f"\n📝 Audit trail updated — preprocessing pipeline closed")

# ==============================================================================
# 6. COMPLETION SUMMARY
# ==============================================================================
print(f"\n🚀 PIPELINE COMPLETE")
print(f"   " + "─" * 50)
print(f"   📊 Records exported:      {len(df_gold):,}")
print(f"   💾 File size:             {file_size_mb:.1f} MB")
print(f"   📁 Location:              {export_path.name}")
print(f"   ✅ Billable:              "
      f"{df_gold.filter(pl.col('code_status') == 'billable').height:,}")
print(f"   ℹ️  Noisy 111:            "
      f"{df_gold.filter(pl.col('code_status') == 'noisy_111').height:,}")
print(f"   🔍 Placeholder-X:         "
      f"{df_gold.filter(pl.col('code_status') == 'placeholder_x').height:,}")
print(f"   ⚠️  Invalid/Malformed:     "
      f"{df_gold.filter(pl.col('code_status') == 'invalid_or_malformed').height:,}")
print(f"   🔒 ICD-10 leaks:          0")
print(f"   🔄 APSO-flipped:          Yes")
print(f"   " + "─" * 50)
print(f"   📌 Primary input column:  apso_note")
print(f"   📌 Label column:          standard_icd10")
print(f"   📌 Status column:         code_status")
print(f"   📌 Original note:         note (unredacted)")

<div style="max-width: 850px; line-height: 1.6; font-family: sans-serif;">

## 🏁 Pipeline Conclusion

This notebook has taken the MedSynth dataset from raw ingestion through to a
fully validated, annotated, and redacted Gold layer — ready for downstream
model training or analysis.

### Final Statistics

| Metric | Value | Notes |
|---|---|---|
| Total records | 10,240 | Full dataset retained — no records removed |
| Billable codes | 9,660 (94.3%) | Confirmed leaf codes, CDC FY2026 reference |
| Noisy 111 codes | 60 (0.6%) | Valid parent codes — real billing practice |
| Placeholder-X codes | 25 (0.24%) | Legitimate injury codes, retained with status |
| APSO coverage | 100% | Assessment at Token 0 for every record |
| ICD-10 leaks remaining | 0 | Verified post-redaction across all columns |
| Export size | 58.9 MB | Snappy-compressed Parquet |

### What Was Built

Starting from 10,240 raw synthetic clinical records, the pipeline produced a
Gold layer (`medsynth_gold_apso_<timestamp>.parquet`) with 13 columns
providing multiple representations of each encounter:

* **`apso_note`** — APSO-recomposed, ICD-10-redacted note for classification training
* **`note`** — original SOAP note, unmodified, for reference tasks
* **`standard_icd10`** — canonical decimal-restored ICD-10 code
* **`code_status`** — CDC-grounded classification (`billable`, `noisy_111`, `placeholder_x`)
* **`assessment`, `plan`, `subjective`, `objective`** — extracted, redacted SOAP sections
* **`label`** — raw ICD-10 label array, preserved from source
* **`dialogue`** — original doctor-patient transcript, unmodified

### Important Caveats for Downstream Use

1. **Class balance is artificial.** MedSynth was engineered with uniform
   sampling (5 records per code). Models trained on this data will not learn
   real-world class frequency distributions. Evaluation on held-out MedSynth
   splits will not reflect deployment performance on real clinical data.

2. **`[REDACTED]` is a learnable token.** The redaction marker replaces code
   strings but is itself present in 28.5% of records. Models with sufficient
   capacity may learn to associate `[REDACTED]` with specific codes via
   surrounding context. Final evaluation should ideally use a real clinical
   dataset such as MIMIC-III.

3. **Dialogue leakage has not been assessed.** The `dialogue` column has not
   been scanned or redacted. The Dialogue Polisher Agent was designed to
   ensure all note content — including ICD-10 codes — appears in the dialogue.
   If dialogue is used as model input, a separate leakage detection pass is
   required before training.

4. **Markdown formatting is present.** The `apso_note` column contains `**`
   bold markers from the Note Writer Agent's output. These consume context
   window tokens without adding clinical signal. Strip markdown formatting
   before tokenisation.

---

**Next Step:** Proceed to Notebook 02 for model training. Use `apso_note` as
the primary text input and `standard_icd10` as the label. Filter by
`code_status` as appropriate for your modelling objective.

</div>